In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# NEW

In [1]:
pip install -qU chromadb groq langchain-groq langgraph==1.0.8 sentence-transformers langchain-huggingface langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 83.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [2]:
# Only %run the summarizer — it's clean and gives us process_uploaded_document
%run /kaggle/input/datasets/rafsharahim/summaryy/multi-agent-legal-summarizer-upload.ipynb
print("✅ Summarizer loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.4 MB/s eta 0:00:00a 0:00:01
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
✅ All imports successful!
✅ Configuration complete
📁 Upload folder: /kaggle/working/up
📁 Output folder: /kaggle/working/outputs
🔍 OCR enabled: True
✅ LLM initialized
✅ Fixed functions ready to use
✅ Workflow compiled successfully!

🏛️  LEGAL DOCUMENT SUMMARIZER - FILE UPLOAD VERSION


📄 Processing: C.A.775_2015.pdf
   File type: .pdf

❌ Extraction failed: Failed to extract


📤 Upload your legal document above and click 'Process Document'
   Supported formats: PDF, DOCX, TXT, PNG, JPG
   OCR will be applied automatically for scanned documents
✅ Summarizer loaded


In [ ]:
import os
import shutil
import chromadb
from chromadb.utils import embedding_functions

GROQ_API_KEY = "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"

SOURCE_DB   = "/kaggle/input/datasets/rafsharahim/chromadb1"
WORKING_DB  = "/kaggle/working/legal_rag_db"


if os.path.exists(WORKING_DB):
    shutil.rmtree(WORKING_DB)

shutil.copytree(SOURCE_DB, WORKING_DB)
print("✅ DB copied")

client = chromadb.PersistentClient(path=WORKING_DB)

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-large-en-v1.5"
)

collection = client.get_collection(
    name="legal_judgments_collection",
    embedding_function=embed_fn
)

print(f"🎯 Collection ready: {collection.count()} judgments")

✅ DB copied


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

🎯 Collection ready: 9226 judgments


In [4]:
import re
import json
from groq import Groq
from typing import TypedDict, List
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  
    temperature=0,
    api_key=GROQ_API_KEY
)

print("✅ All imports done, LLM ready")

✅ All imports done, LLM ready


In [16]:
def get_agentic_rag_data(user_story: str) -> dict:
    """Queries the 9226-judgment collection and extracts legal grounds."""
    results = collection.query(query_texts=[user_story], n_results=3)

    if not results['documents'] or not results['documents'][0]:
        return {"status": "fail", "message": "No precedents matched."}

    best_context = results['documents'][0][0]
    case_id = results['metadatas'][0][0].get('case_id', 'Unknown Citation')

    groq_client = Groq(api_key=GROQ_API_KEY)  # FIXED: use variable, not os.environ.get(key)

    extraction_prompt = f"""
SYSTEM: You are a Senior Constitutional Lawyer in Pakistan.
Analyze the USER STORY against the SUPREME COURT PRECEDENT provided.

USER STORY: {user_story}
PRECEDENT: {best_context}

TASK:
1. Identify the Ratio Decidendi (the legal rule).
2. Draft 3 formal Grounds (starting with 'That...') for a Writ Petition.
3. Identify the Prayer (what we want the court to do).

FORMAT:
CITATION: {case_id}
RATIO: [Principle]
GROUNDS: [The drafted grounds]
PRAYER: [The relief requested]
"""
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": extraction_prompt}],
        temperature=0
    )

    return {
        "status": "success",
        "analysis": response.choices[0].message.content,
        "citation": case_id
    }


def get_easy_draft_data(user_story: str) -> dict:
    """Returns structured JSON with petition fields."""
    results = collection.query(query_texts=[user_story], n_results=1)
    context = results['documents'][0][0]
    case_id = results['metadatas'][0][0].get('case_id', 'Unknown')

    groq_client = Groq(api_key=GROQ_API_KEY)

    json_prompt = f"""
You are a Legal API. Analyze the Case and Story.
Return ONLY a JSON object with these exact keys:
{{
    "case_citation": "{case_id}",
    "petition_title": "Writ Petition under Article 199...",
    "petitioner_name": "Family member name from the story",
    "detenu_accused_name": "Person in custody from the story",
    "grounds": ["Ground A starting with That...", "Ground B", "Ground C"],
    "prayer": "The specific relief requested"
}}

STORY: {user_story}
CASE: {context}
"""
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": json_prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )

    return json.loads(response.choices[0].message.content)


print("✅ RAG functions ready")

✅ RAG functions ready


In [17]:
def safe_parse_json(text: str) -> dict:
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print(f"⚠️ JSON parse error: {e}")
    return {}


class LegalGenState(TypedDict):
    user_story:       str
    is_complete:      bool
    missing_info:     List[str]
    petition_type:    str
    target_court:     str
    next_step:        str
    research_context: str
    extracted_ratio:  str
    case_id:          str
    is_relevant:      bool
    final_petition:   str

print("✅ State defined")

✅ State defined


In [18]:
def check_missing_info_node(state: LegalGenState):
    print("📋 [InfoCheck] Scanning for missing details...")
    groq_client = Groq(api_key=GROQ_API_KEY)
    prompt = f"""
Analyze: "{state['user_story']}"
Check if ALL are present: petitioner name, detained person name, location/police station, date.
Return ONLY JSON:
{{"is_complete": true/false, "missing_fields": ["field1"], "guidance": "polite message"}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(resp.choices[0].message.content)
    is_complete = result.get("is_complete", False)
    return {
        "missing_info": result.get("missing_fields", []),
        "is_complete":  is_complete,
        "next_step":    "proceed" if is_complete else "ask_user"
    }


def orchestrator_node(state: LegalGenState):
    print("🧠 [Orchestrator] Classifying case...")
    groq_client = Groq(api_key=GROQ_API_KEY)
    prompt = f"""
Analyze this Pakistani legal situation: "{state['user_story']}"
Return ONLY JSON:
{{"petition_type": "Habeas Corpus|Pre-Arrest Bail|Post-Arrest Bail|Quashment|Property",
  "target_court": "High Court|Sessions Court",
  "next_step": "summarize"}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(resp.choices[0].message.content)
    return {
        "petition_type": result.get("petition_type", "Habeas Corpus"),
        "target_court":  result.get("target_court",  "High Court"),
        "next_step":     "summarize"
    }


def rag_research_node(state: LegalGenState):
    print(f"🔎 [RAG] Searching {collection.count()} judgments...")
    rag_output = get_agentic_rag_data(state["user_story"])
    if rag_output["status"] == "success":
        print(f"✅ Found: {rag_output['citation']}")
        return {
            "research_context": rag_output["analysis"],
            "extracted_ratio":  rag_output["citation"],
            "case_id":          rag_output["citation"],
            "is_relevant":      True
        }
    print("⚠️ No precedent — using constitutional defaults")
    return {
        "research_context": "Articles 9 and 10A of the Constitution of Pakistan — right to life, liberty and fair trial.",
        "extracted_ratio":  "Articles 9 & 10A — Constitution of Pakistan 1973",
        "case_id":          "Constitutional Principles",
        "is_relevant":      True
    }


def drafter_node(state: LegalGenState):
    print("✍️ [Drafter] Writing petition...")
    try:
        structured      = get_easy_draft_data(state["user_story"])
        petitioner      = structured.get("petitioner_name",    "The Petitioner")
        detenu          = structured.get("detenu_accused_name","the detained person")
        grounds_list    = structured.get("grounds",            [])
        prayer          = structured.get("prayer",             "Release the detained person forthwith")
        citation        = structured.get("case_citation",      state.get("case_id", ""))
        grounds_text    = "\n".join([f"{i+1}. {g}" for i, g in enumerate(grounds_list)])
    except Exception as e:
        print(f"⚠️ Structured extraction failed ({e}) — using defaults")
        petitioner   = "The Petitioner"
        detenu       = "the detained person"
        grounds_text = "1. That the detention is illegal and without lawful authority."
        prayer       = "Release the detained person forthwith."
        citation     = state.get("case_id", "Articles 9 & 10A")

    prompt = ChatPromptTemplate.from_template("""
You are a Senior Pakistani Advocate. Draft a complete formal court petition.

PETITION TYPE : {petition_type}
TARGET COURT  : {target_court}
PETITIONER    : {petitioner}
SUBJECT       : {detenu}
CITATION      : {citation}
RAG ANALYSIS  : {research_context}
GROUNDS       : {grounds_text}
PRAYER        : {prayer}
CLIENT FACTS  : {user_story}

Write the FULL petition:
1. Court heading and title block
2. Parties (Petitioner vs Respondents)
3. "Most Respectfully Sheweth:" — facts paragraph
4. Numbered GROUNDS (minimum 4, reference the citation explicitly)
5. PRAYER paragraph
6. Signature block — "Respectfully Submitted, Advocate for the Petitioner"

Use formal Pakistani court language throughout.
""")
    chain  = prompt | llm | StrOutputParser()
    draft  = chain.invoke({
        "petition_type":    state.get("petition_type",    "Habeas Corpus Petition"),
        "target_court":     state.get("target_court",     "Honourable High Court"),
        "petitioner":       petitioner,
        "detenu":           detenu,
        "citation":         citation,
        "research_context": state.get("research_context", ""),
        "grounds_text":     grounds_text,
        "prayer":           prayer,
        "user_story":       state["user_story"]
    })
    return {"final_petition": draft}

print("✅ All nodes defined")

✅ All nodes defined


In [19]:
builder = StateGraph(LegalGenState)

builder.add_node("info_check",   check_missing_info_node)
builder.add_node("orchestrator", orchestrator_node)
builder.add_node("rag_research", rag_research_node)
builder.add_node("drafter",      drafter_node)

builder.add_edge(START, "info_check")

builder.add_conditional_edges(
    "info_check",
    lambda s: s.get("next_step", "ask_user"),
    {"ask_user": END, "proceed": "orchestrator"}
)

builder.add_conditional_edges(
    "orchestrator",
    lambda s: s.get("next_step", "summarize"),
    {"summarize": "rag_research", "draft": "drafter"}
)

builder.add_edge("rag_research", "drafter")
builder.add_edge("drafter",      END)

legal_gen_app = builder.compile()
print("✅ Graph compiled — ready")

✅ Graph compiled — ready


In [23]:
def run_legal_assistant(story: str):
    state = {
        "user_story":       story,
        "is_complete":      False,
        "missing_info":     [],
        "petition_type":    "",
        "target_court":     "",
        "next_step":        "",
        "research_context": "",
        "extracted_ratio":  "",
        "case_id":          "",
        "is_relevant":      False,
        "final_petition":   ""
    }

    while True:
        output = legal_gen_app.invoke(state)
        state.update(output)

        # Missing info → ask user, append reply, reset flag, re-invoke
        if not state.get("is_complete") and state.get("missing_info"):
            print(f"\n📋 Missing: {', '.join(state['missing_info'])}")
            reply = input("👤 Please provide these details: ")

            # FIXED: update state directly before next loop iteration
            state["user_story"]  = state["user_story"] + f"\nAdditional details: {reply}"
            state["is_complete"] = False
            state["missing_info"] = []  # clear so we don't re-trigger if LLM is slow
            continue

        # Success
        if state.get("final_petition"):
            print("\n" + "="*65)
            print(f"⚖️  {state.get('petition_type', 'LEGAL').upper()}")
            print(f"🏛️  {state.get('target_court', '')}")
            print(f"📌 Citation: {state.get('case_id', 'N/A')}")
            print("="*65)
            print(state["final_petition"])
            break

        print("⚠️ Ended without petition. Keys:", list(state.keys()))
        break

In [24]:
run_legal_assistant("My brother was arrested from my house last night without a warrant.")

📋 [InfoCheck] Scanning for missing details...

📋 Missing: petitioner name, detained person name, location/police station, date


👤 Please provide these details:  My name is Rafsha Rahim. My brother Ali Khan was arrested from our home in Gulshan-e-Iqbal, Karachi. The incident happened on March 24, 2026. He is being held at Gulshan-e-Iqbal Police Station.


📋 [InfoCheck] Scanning for missing details...
🧠 [Orchestrator] Classifying case...
🔎 [RAG] Searching 9226 judgments...
✅ Found: Crl.P.L.A.660_2024
✍️ [Drafter] Writing petition...

⚖️  HABEAS CORPUS
🏛️  High Court
📌 Citation: Crl.P.L.A.660_2024
**IN THE HIGH COURT OF SINDH AT KARACHI**

**Crl.P.L.A. No. 660 of 2024**

**RAFSHA RAHIM**
**Petitioner**
**Versus**
**THE STATE**
**Respondent No. 1**
**STATION HOUSE OFFICER, GULSHAN-E-IQBAL POLICE STATION**
**Respondent No. 2**

**Most Respectfully Sheweth:**

That the petitioner is the sister of Ali Khan, who was arrested from the petitioner's house on the night of March 24, 2026, by the respondents without a warrant. The petitioner states that the arrest was made without any lawful authority and in clear violation of the principles enunciated in Abdul Qudoos v. Hafiz Israr Ahmed (Crl.P.L.A.660_2024), which holds that an arrest without a warrant is not permissible under the law. The petitioner further states that her brother is being detain

### More examples

In [25]:
# ── Petition templates per type ──────────────────────────────────────────────

PETITION_TEMPLATES = {

"Habeas Corpus": """
You are a Senior Pakistani Advocate at the High Court.
Draft a complete HABEAS CORPUS PETITION under Article 199 of the Constitution.

COURT          : {target_court}
PETITIONER     : {petitioner}
DETENU         : {detenu}
CITATION       : {citation}
RAG ANALYSIS   : {research_context}
GROUNDS        : {grounds_text}
PRAYER         : {prayer}
CLIENT FACTS   : {user_story}

STRUCTURE:
1. IN THE {target_court} — Writ Petition No. ___/2026 (Habeas Corpus)
2. Parties: [Petitioner] vs [SHO Police Station + Home Secretary + AG as Respondents]
3. "Most Respectfully Sheweth:" — facts of illegal detention
4. GROUNDS (min 5):
   - Arrest without warrant violates Article 9
   - Not produced before magistrate within 24 hours (Section 61 CrPC)
   - Plain clothes officers = unlawful (Police Rules)
   - Reference {citation} ratio explicitly
   - Article 10 right to be informed of grounds of arrest
5. PRAYER: produce detenu + release + action against officers
6. Signature block
""",

"Pre-Arrest Bail": """
You are a Senior Pakistani Advocate.
Draft a complete PRE-ARREST BAIL APPLICATION under Section 498 CrPC.

COURT          : {target_court}
APPLICANT      : {petitioner}
FIR DETAILS    : {user_story}
CITATION       : {citation}
RAG ANALYSIS   : {research_context}
GROUNDS        : {grounds_text}
PRAYER         : {prayer}

STRUCTURE:
1. IN THE {target_court} — Crl. Misc. Application No. ___/2026
   (Application for Pre-Arrest Bail under Section 498 CrPC)
2. Parties: [Applicant/Accused] vs [The State]
3. "Most Respectfully Sheweth:" — FIR details, applicant's apprehension of arrest
4. GROUNDS (min 5):
   - Offence is bailable / applicant is not a flight risk
   - Applicant has deep roots in community
   - Arrest would cause irreparable harm
   - Malafide FIR / no independent evidence
   - Reference {citation} ratio on bail jurisprudence
   - Applicant ready to cooperate with investigation
5. PRAYER: grant pre-arrest bail on surety + conditions
6. Signature block
""",

"Post-Arrest Bail": """
You are a Senior Pakistani Advocate.
Draft a complete POST-ARREST BAIL APPLICATION under Section 497 CrPC.

COURT          : {target_court}
APPLICANT      : {petitioner}
ACCUSED        : {detenu}
CITATION       : {citation}
RAG ANALYSIS   : {research_context}
GROUNDS        : {grounds_text}
PRAYER         : {prayer}
CLIENT FACTS   : {user_story}

STRUCTURE:
1. IN THE {target_court} — Crl. Misc. Application No. ___/2026
   (Application for Post-Arrest Bail under Section 497 CrPC)
2. Parties: [Accused/Applicant] vs [The State]
3. "Most Respectfully Sheweth:" — arrest details, time in custody, case status
4. GROUNDS (min 5):
   - Offence punishable with less than 10 years (if applicable) — bailable as of right
   - Challan not submitted / investigation incomplete
   - Applicant not a previous convict
   - Continued detention is punitive not preventive
   - Reference {citation} bail ratio
   - Surety offered, no flight risk
5. PRAYER: release on bail with surety amount
6. Signature block
""",

"Quashment": """
You are a Senior Pakistani Advocate.
Draft a complete WRIT PETITION FOR QUASHMENT OF FIR under Article 199.

COURT          : {target_court}
PETITIONER     : {petitioner}
CITATION       : {citation}
RAG ANALYSIS   : {research_context}
GROUNDS        : {grounds_text}
PRAYER         : {prayer}
CLIENT FACTS   : {user_story}

STRUCTURE:
1. IN THE {target_court} — Writ Petition No. ___/2026
   (Petition for Quashment of FIR under Article 199 of the Constitution)
2. Parties: [Petitioner] vs [SHO + State + Complainant]
3. "Most Respectfully Sheweth:" — FIR details, why FIR is mala fide/illegal
4. GROUNDS (min 5):
   - FIR lodged with mala fide intention / personal vendetta
   - No cognizable offence disclosed in FIR
   - FIR is abuse of process of court
   - Continuing prosecution causes irreparable harm
   - Reference {citation} on quashment jurisdiction
   - High Court's inherent powers under Article 199
5. PRAYER: quash FIR + restrain police from arrest
6. Signature block
""",

"Property/Encroachment": """
You are a Senior Pakistani Advocate.
Draft a complete CONSTITUTIONAL WRIT PETITION under Article 199
challenging illegal sealing/demolition/encroachment action.

COURT          : {target_court}
PETITIONER     : {petitioner}
CITATION       : {citation}
RAG ANALYSIS   : {research_context}
GROUNDS        : {grounds_text}
PRAYER         : {prayer}
CLIENT FACTS   : {user_story}

STRUCTURE:
1. IN THE {target_court} — Writ Petition No. ___/2026
   (Constitutional Petition under Article 199 — Illegal Sealing/Demolition)
2. Parties: [Petitioner] vs [KBCA/KMC/Relevant Authority + Federation]
3. "Most Respectfully Sheweth:" — property details, authority's illegal action, no notice
4. GROUNDS (min 5):
   - Action taken without notice violates principles of natural justice
   - Article 10A — right to fair hearing before adverse action
   - Article 23 — right to acquire and hold property
   - Authority exceeded its statutory powers
   - Reference {citation}
   - Petitioner has valid title/lease documents
5. PRAYER: restore property + declare action void + damages
6. Signature block
"""
}

# ── Updated drafter node ─────────────────────────────────────────────────────

def drafter_node(state: LegalGenState):
    petition_type = state.get("petition_type", "Habeas Corpus")
    print(f"✍️ [Drafter] Writing {petition_type} petition...")

    # Pick the right template — fallback to Habeas Corpus if unknown type
    template_str = PETITION_TEMPLATES.get(petition_type, PETITION_TEMPLATES["Habeas Corpus"])

    # Get structured fields from RAG notebook's function
    try:
        structured   = get_easy_draft_data(state["user_story"])
        petitioner   = structured.get("petitioner_name",    "The Petitioner")
        detenu       = structured.get("detenu_accused_name","the detained/accused person")
        grounds_list = structured.get("grounds",            [])
        prayer       = structured.get("prayer",             "Grant the relief prayed for")
        citation     = structured.get("case_citation",      state.get("case_id", ""))
        grounds_text = "\n".join([f"{i+1}. {g}" for i, g in enumerate(grounds_list)])
    except Exception as e:
        print(f"⚠️ Structured extraction failed ({e}) — using defaults")
        petitioner   = "The Petitioner"
        detenu       = "the detained/accused person"
        grounds_text = "1. That the action complained of is illegal and without lawful authority."
        prayer       = "Grant the relief prayed for."
        citation     = state.get("case_id", "Articles 9, 10A & 199 — Constitution of Pakistan 1973")

    prompt   = ChatPromptTemplate.from_template(template_str)
    chain    = prompt | llm | StrOutputParser()

    draft = chain.invoke({
        "petition_type":    petition_type,
        "target_court":     state.get("target_court", "Honourable High Court"),
        "petitioner":       petitioner,
        "detenu":           detenu,
        "citation":         citation,
        "research_context": state.get("research_context", ""),
        "grounds_text":     grounds_text,
        "prayer":           prayer,
        "user_story":       state["user_story"]
    })

    return {"final_petition": draft}

print("✅ Petition-type-aware drafter ready")

✅ Petition-type-aware drafter ready


In [27]:
def check_missing_info_node(state: LegalGenState):
    print("📋 [InfoCheck] Scanning for missing details...")
    groq_client = Groq(api_key=GROQ_API_KEY)
    prompt = f"""
You are a Pakistani legal assistant. Analyze this situation: "{state['user_story']}"

Check ONLY for these 3 absolutely essential fields:
1. Petitioner name OR applicant name (the person filing)
2. Name of detained/accused/affected person
3. Location OR police station OR authority involved

DO NOT ask for dates, FIR numbers, duration, or any other details.
If any of the 3 fields above can be reasonably inferred from context, mark as present.

Return ONLY valid JSON:
{{
    "is_complete": true or false,
    "missing_fields": ["only list fields from the 3 above that are truly missing"],
    "guidance": "polite one-line request for only the missing fields"
}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(resp.choices[0].message.content)
    is_complete = result.get("is_complete", False)
    return {
        "missing_info": result.get("missing_fields", []),
        "is_complete":  is_complete,
        "next_step":    "proceed" if is_complete else "ask_user"
    }

print("✅ Info check updated — only 3 essential fields required")

✅ Info check updated — only 3 essential fields required


In [29]:
def run_legal_assistant(story: str):
    state = {
        "user_story":       story,
        "is_complete":      False,
        "missing_info":     [],
        "petition_type":    "",
        "target_court":     "",
        "next_step":        "",
        "research_context": "",
        "extracted_ratio":  "",
        "case_id":          "",
        "is_relevant":      False,
        "final_petition":   ""
    }

    # ── PASS 1: info check only ──────────────────────────────────────
    print("📋 [InfoCheck] Scanning for missing details...")
    info_result = check_missing_info_node(state)
    state.update(info_result)

    if not state.get("is_complete"):
        print(f"\n📋 Missing: {', '.join(state['missing_info'])}")
        reply = input("👤 Please provide these details: ")
        # Append and mark complete — skip re-checking
        state["user_story"]   = state["user_story"] + f"\nAdditional details: {reply}"
        state["is_complete"]  = True   # FORCE proceed after one round
        state["next_step"]    = "proceed"

    # ── PASS 2: run rest of graph (orchestrator → RAG → drafter) ─────
    # Build a mini graph without info_check
    builder2 = StateGraph(LegalGenState)
    builder2.add_node("orchestrator", orchestrator_node)
    builder2.add_node("rag_research", rag_research_node)
    builder2.add_node("drafter",      drafter_node)

    builder2.add_edge(START, "orchestrator")
    builder2.add_conditional_edges(
        "orchestrator",
        lambda s: s.get("next_step", "summarize"),
        {"summarize": "rag_research", "draft": "drafter"}
    )
    builder2.add_edge("rag_research", "drafter")
    builder2.add_edge("drafter",      END)

    pipeline = builder2.compile()
    output   = pipeline.invoke(state)
    state.update(output)

    if state.get("final_petition"):
        print("\n" + "="*65)
        print(f"⚖️  {state.get('petition_type', 'LEGAL').upper()}")
        print(f"🏛️  {state.get('target_court', '')}")
        print(f"📌 Citation: {state.get('case_id', 'N/A')}")
        print("="*65)
        print(state["final_petition"])
    else:
        print("⚠️ No petition generated. State:", list(state.keys()))

print("✅ Runner fixed — no more infinite loop")

✅ Runner fixed — no more infinite loop


In [30]:
test_cases = [
    "My brother Ali Khan was arrested without warrant from our home in Gulshan-e-Iqbal by plain clothes officers on March 24 2026. My name is Rafsha Rahim. He is at Gulshan Police Station.",
    
    "An FIR has been registered against me at Clifton Police Station under Section 324 PPC. I have not been arrested yet. My name is Ahmed Raza. FIR date March 28 2026.",
    
    "My client Bilal Sheikh was arrested on March 20 2026 and has been in custody for 12 days at Defence Police Station. FIR under Section 409 PPC. I am his lawyer Sara Khan.",
    
    "My shop in Saddar Karachi was sealed by KBCA on March 25 2026 without any prior notice. I am Tariq Mehmood and have valid lease documents."
]

for i, case in enumerate(test_cases, 1):
    print(f"\n{'='*65}")
    print(f"🧪 TEST CASE {i}")
    print(f"{'='*65}")
    run_legal_assistant(case)
    print("\n")


🧪 TEST CASE 1
📋 [InfoCheck] Scanning for missing details...
📋 [InfoCheck] Scanning for missing details...

📋 Missing: Location OR police station OR authority involved


👤 Please provide these details:  Gulshan-e-Iqbal Police Station, Karachi


🧠 [Orchestrator] Classifying case...
🔎 [RAG] Searching 9226 judgments...
✅ Found: Crl.P.L.A.660_2024
✍️ [Drafter] Writing Habeas Corpus petition...

⚖️  HABEAS CORPUS
🏛️  High Court
📌 Citation: Crl.P.L.A.660_2024
**IN THE HIGH COURT OF SINDH AT KARACHI**

**Writ Petition No. 1234/2026 (Habeas Corpus)**

**Rafsha Rahim**
**Petitioner**

**Versus**

**1. Station House Officer, Gulshan-e-Iqbal Police Station, Karachi**
**2. Home Secretary, Government of Sindh**
**3. Advocate General, Sindh**

**Most Respectfully Sheweth:**

That the petitioner is the sister of Ali Khan, who was arrested without a warrant by plain clothes officers from their home in Gulshan-e-Iqbal on March 24, 2026. The petitioner has been unable to ascertain the reasons for her brother's arrest, and he has not been produced before a magistrate within 24 hours of his arrest, as required by law.

**GROUNDS:**

1. That the arrest of Ali Khan without a warrant is in contravention of Article 9 of the Constitution of Pakistan,

👤 Please provide these details:  Rafsha Rahim


🧠 [Orchestrator] Classifying case...
🔎 [RAG] Searching 9226 judgments...
✅ Found: Crl.P.L.A.1117_2024
✍️ [Drafter] Writing Pre-Arrest Bail petition...

⚖️  PRE-ARREST BAIL
🏛️  Sessions Court
📌 Citation: Crl.P.L.A.1117_2024
IN THE Sessions Court — Crl. Misc. Application No. ___/2026
(Application for Pre-Arrest Bail under Section 498 CrPC)

Parties: 
Ahmed Raza, son of _______________________, resident of _______________________
(Applicant/Accused) 
vs 
The State

Most Respectfully Sheweth:

That an FIR has been registered against the applicant, Ahmed Raza, under Section 324 PPC on March 28, 2026, at Clifton Police Station. The applicant has not been arrested yet, but he has a reasonable apprehension of arrest, as the police may arrest him at any moment. The applicant is a law-abiding citizen and has cooperated with the investigation as and when required.

GROUNDS:

1. That the offence under Section 324 PPC is bailable, and the applicant is not a flight risk, as he has deep roots in the 

👤 Please provide these details:  Rafsha Rahim


🧠 [Orchestrator] Classifying case...
🔎 [RAG] Searching 9226 judgments...
✅ Found: Crl.P.L.A.1117_2024
✍️ [Drafter] Writing Habeas Corpus petition...

⚖️  HABEAS CORPUS
🏛️  High Court
📌 Citation: Crl.P.L.A.1117_2024
IN THE HIGH COURT OF [PROVINCE] AT [CITY]
Writ Petition No. Crl.P.L.A.1117/2024 (Habeas Corpus)

BILAL SHEIKH
Petitioner
Versus
1. SHO, DEFENCE POLICE STATION
2. HOME SECRETARY, GOVERNMENT OF [PROVINCE]
3. ADVOCATE GENERAL, [PROVINCE]

Most Respectfully Sheweth:

That the petitioner, Bilal Sheikh, is a citizen of Pakistan and has been illegally detained by the respondents since March 20, 2026, at Defence Police Station. The petitioner was arrested without a warrant and has been kept in custody for 12 days without being produced before a magistrate within 24 hours of his arrest, as mandated by Section 61 of the Code of Criminal Procedure, 1898.

The petitioner was implicated in a supplementary statement of the complainant with a noticeable delay, and there is no incriminating

👤 Please provide these details:  Rafsha Rahim


🧠 [Orchestrator] Classifying case...
🔎 [RAG] Searching 9226 judgments...
✅ Found: C.P.L.A.1331-L_2017
✍️ [Drafter] Writing Quashment petition...

⚖️  QUASHMENT
🏛️  High Court
📌 Citation: C.P.L.A.1331-L_2017
IN THE HIGH COURT OF SINDH AT KARACHI

Writ Petition No. ___/2026
(Petition for Quashment of Order of Sealing under Article 199 of the Constitution)

Between:

Tariq Mehmood
Petitioner

Versus

1. SHO, Karachi Building Control Authority (KBCA)
2. State
3. KBCA

Most Respectfully Sheweth:

That the petitioner is the owner of a shop located in Saddar, Karachi, which was sealed by the respondent, KBCA, on March 25, 2026, without any prior notice. The petitioner has valid lease documents and has been running the business lawfully. The sealing of the premises was done without any prior notice, and the petitioner was not afforded an opportunity of hearing, which is a clear violation of the principles of natural justice and the right to due process under Article 10-A of the Constitution of

## VERSION 2 (complete)

In [37]:
from groq import Groq
from typing import TypedDict, List, Optional
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ Imports & LLM ready")

✅ Imports & LLM ready


In [38]:
# Simple in-session memory per user
# Stores: petition type preferences, jurisdiction, past citations used
USER_MEMORY = {}

def save_to_memory(user_id: str, state: dict):
    if user_id not in USER_MEMORY:
        USER_MEMORY[user_id] = {
            "past_petitions":   [],
            "preferred_court":  None,
            "citations_used":   [],
            "petition_types":   []
        }
    mem = USER_MEMORY[user_id]
    mem["past_petitions"].append({
        "story":        state.get("user_story", "")[:200],
        "type":         state.get("petition_type", ""),
        "court":        state.get("jurisdiction", ""),
        "citation":     state.get("primary_citation", ""),
        "draft_style":  state.get("final_petition", "")[:300]
    })
    if state.get("jurisdiction"):
        mem["preferred_court"] = state["jurisdiction"]
    if state.get("primary_citation"):
        mem["citations_used"].append(state["primary_citation"])
    if state.get("petition_type"):
        mem["petition_types"].append(state["petition_type"])
    print(f"💾 Memory saved for user: {user_id}")

def get_memory_context(user_id: str) -> str:
    if user_id not in USER_MEMORY or not USER_MEMORY[user_id]["past_petitions"]:
        return "No previous petitions on record."
    mem = USER_MEMORY[user_id]
    return f"""
User has filed {len(mem['past_petitions'])} previous petition(s).
Preferred court: {mem['preferred_court'] or 'Not set'}
Most used petition type: {max(set(mem['petition_types']), key=mem['petition_types'].count) if mem['petition_types'] else 'None'}
Citations previously used: {', '.join(mem['citations_used'][-3:]) or 'None'}
Last petition style sample: {mem['past_petitions'][-1]['draft_style'] if mem['past_petitions'] else 'None'}
"""

print("✅ Memory system ready")

✅ Memory system ready


In [39]:
class LegalGenState(TypedDict):
    # Input
    user_story:          str
    user_id:             str
    jurisdiction:        str      # Sindh HC | Lahore HC | Islamabad HC | Sessions Court

    # Info check
    is_complete:         bool
    missing_info:        List[str]

    # Orchestrator
    petition_type:       str
    target_court:        str
    next_step:           str

    # RAG — multi-hop
    primary_context:     str
    primary_citation:    str
    supporting_context:  str
    supporting_citation: str
    rag_attempts:        int      # tracks self-correction retries

    # Structured fields
    petitioner:          str
    detenu:              str
    grounds_text:        str
    prayer:              str

    # Validator
    is_valid:            bool
    validation_notes:    str
    revision_count:      int

    # Output
    final_petition:      str
    memory_context:      str

print("✅ State defined")


✅ State defined


In [40]:
def check_missing_info_node(state: LegalGenState):
    print("📋 [InfoCheck] Scanning for missing details...")
    text  = state["user_story"].lower()
    orig  = state["user_story"]
    missing = []

    # 1. Filer identity
    has_filer = bool(
        re.search(r'my name is', text) or
        re.search(r'\bi am\b', text) or
        re.search(r"\bi'm\b", text) or
        re.search(r'\b(lawyer|advocate|counsel|sister|brother|wife|'
                  r'husband|father|mother|son|daughter|client)\b', text)
    )
    if not has_filer:
        missing.append("your name or relationship to the affected person")

    # 2. Affected person
    has_person = bool(
        re.search(r'\b(brother|sister|son|daughter|husband|wife|'
                  r'father|mother|client|accused|detenu|applicant)\b', text) or
        re.search(r'\b(mr|mrs|ms|dr)\.?\s+\w+', text) or
        len(re.findall(r'[A-Z][a-z]+\s+[A-Z][a-z]+', orig)) >= 1
    )
    if not has_person:
        missing.append("name of detained or accused person")

    # 3. Location
    has_location = bool(
        re.search(r'police station', text) or
        re.search(r'\b(karachi|lahore|islamabad|peshawar|quetta|'
                  r'multan|faisalabad|rawalpindi|hyderabad|sukkur)\b', text) or
        re.search(r'\b(gulshan|clifton|defence|dha|saddar|korangi|'
                  r'malir|orangi|landhi|johar|nazimabad|pechs|north\s*nazimabad)\b', text) or
        re.search(r'\b(kbca|kmc|nha|wasa|court|authority|agency|tribunal)\b', text) or
        re.search(r'\b(sector|block|town|district|area|road|colony|phase)\b', text) or
        re.search(r'station\b', text)
    )
    if not has_location:
        missing.append("location or police station")

    is_complete = len(missing) == 0
    print(f"   → Complete: {is_complete} | Missing: {missing}")
    return {
        "missing_info": missing,
        "is_complete":  is_complete,
        "next_step":    "proceed" if is_complete else "ask_user"
    }

print("✅ Info check ready")

✅ Info check ready


In [41]:
COURT_FORMATS = {
    "Sindh HC": {
        "full_name":  "IN THE HIGH COURT OF SINDH AT KARACHI",
        "city":       "Karachi",
        "ag":         "Advocate General, Sindh",
        "home_sec":   "Home Secretary, Government of Sindh",
        "short":      "Sindh High Court"
    },
    "Lahore HC": {
        "full_name":  "IN THE HIGH COURT OF LAHORE AT LAHORE",
        "city":       "Lahore",
        "ag":         "Advocate General, Punjab",
        "home_sec":   "Home Secretary, Government of Punjab",
        "short":      "Lahore High Court"
    },
    "Islamabad HC": {
        "full_name":  "IN THE HIGH COURT OF ISLAMABAD",
        "city":       "Islamabad",
        "ag":         "Advocate General, Islamabad",
        "home_sec":   "Home Secretary, ICT Administration",
        "short":      "Islamabad High Court"
    },
    "Sessions Court Karachi": {
        "full_name":  "IN THE COURT OF SESSIONS JUDGE, KARACHI",
        "city":       "Karachi",
        "ag":         "N/A",
        "home_sec":   "N/A",
        "short":      "Sessions Court Karachi"
    },
    "Sessions Court Lahore": {
        "full_name":  "IN THE COURT OF SESSIONS JUDGE, LAHORE",
        "city":       "Lahore",
        "ag":         "N/A",
        "home_sec":   "N/A",
        "short":      "Sessions Court Lahore"
    }
}

def select_jurisdiction(petition_type: str, user_story: str, user_id: str) -> str:
    """Auto-detect jurisdiction from story, memory, or petition type."""
    text = user_story.lower()
    mem  = USER_MEMORY.get(user_id, {})

    # 1. Detect from story text
    if any(w in text for w in ["karachi", "sindh", "gulshan", "clifton",
                                "defence", "dha karachi", "saddar"]):
        if petition_type in ["Habeas Corpus", "Quashment", "Property/Encroachment"]:
            return "Sindh HC"
        return "Sessions Court Karachi"

    if any(w in text for w in ["lahore", "punjab", "dha lahore",
                                "gulberg", "faisalabad", "multan"]):
        if petition_type in ["Habeas Corpus", "Quashment", "Property/Encroachment"]:
            return "Lahore HC"
        return "Sessions Court Lahore"

    if any(w in text for w in ["islamabad", "ict", "rawalpindi", "pindi"]):
        return "Islamabad HC"

    # 2. Fall back to user memory preference
    if mem.get("preferred_court"):
        return mem["preferred_court"]

    # 3. Default
    if petition_type in ["Pre-Arrest Bail", "Post-Arrest Bail"]:
        return "Sessions Court Karachi"
    return "Sindh HC"

print("✅ Jurisdiction selector ready")

✅ Jurisdiction selector ready


In [42]:
def orchestrator_node(state: LegalGenState):
    print("🧠 [Orchestrator] Classifying case...")

    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
You are a Pakistani legal classifier. Read carefully and classify.

SITUATION: "{state['user_story']}"

STRICT DECISION TREE — follow in order:

STEP 1: Is there any mention of property, shop, land, sealing, demolition, KBCA, KMC, encroachment?
-> YES = "Property/Encroachment" (even if person was also detained)

STEP 2: Is there an FIR AND the person is already in custody/arrested?
-> YES = "Post-Arrest Bail"

STEP 3: Is there an FIR but person NOT yet arrested?
-> YES = "Pre-Arrest Bail"

STEP 4: Person detained/arrested with NO FIR mentioned?
-> YES = "Habeas Corpus"

STEP 5: Want to cancel/quash an existing FIR itself?
-> YES = "Quashment"

Return ONLY JSON with these exact keys:
petition_type: one of the 5 types above
reasoning: which step matched and why
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    result       = json.loads(resp.choices[0].message.content)
    p_type       = result.get("petition_type", "Habeas Corpus")
    jurisdiction = select_jurisdiction(p_type, state["user_story"], state.get("user_id", "default"))

    print(f"   → Type: {p_type} | Jurisdiction: {jurisdiction} | Reason: {result.get('reasoning','')}")
    return {
        "petition_type": p_type,
        "jurisdiction":  jurisdiction,
        "target_court":  COURT_FORMATS[jurisdiction]["full_name"],
        "next_step":     "rag"
    }

print("✅ Orchestrator fixed")

✅ Orchestrator fixed


In [43]:
def rag_primary_node(state: LegalGenState):
    """First RAG hop — find primary precedent directly matching the case."""
    print(f"🔎 [RAG Primary] Searching {collection.count()} judgments...")

    attempts   = state.get("rag_attempts", 0)
    user_story = state["user_story"]
    p_type     = state.get("petition_type", "")

    # Build query — refine if this is a retry
    if attempts == 0:
        query = user_story
    elif attempts == 1:
        query = f"{p_type} illegal arrest without warrant Pakistan Supreme Court"
    else:
        query = f"fundamental rights Article 9 10A Constitution Pakistan detention"

    results  = collection.query(query_texts=[query], n_results=3)

    if not results['documents'] or not results['documents'][0]:
        return {
            "primary_context":  "No precedent found.",
            "primary_citation": "Articles 9 & 10A Constitution of Pakistan 1973",
            "rag_attempts":     attempts + 1
        }

    # Pick best result
    best_doc  = results['documents'][0][0]
    best_meta = results['metadatas'][0][0]
    citation  = best_meta.get('case_id', 'Unknown')

    print(f"   → Primary found: {citation} (attempt {attempts + 1})")

    # Quick relevance check
    relevance_resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
Is this case relevant to the user's situation?
USER: {user_story[:300]}
CASE SUMMARY: {best_doc[:400]}
PETITION TYPE NEEDED: {p_type}

Return ONLY JSON: {{"is_relevant": true/false, "reason": "one line"}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    rel = json.loads(relevance_resp.choices[0].message.content)

    if not rel.get("is_relevant") and attempts < 2:
        # SELF-CORRECT — retry with refined query
        print(f"   ⚠️ Not relevant ({rel.get('reason')}) — retrying...")
        return {
            "primary_context":  "",
            "primary_citation": "",
            "rag_attempts":     attempts + 1,
            "next_step":        "rag_retry"
        }

    return {
        "primary_context":  best_doc,
        "primary_citation": citation,
        "rag_attempts":     attempts + 1,
        "next_step":        "rag_secondary"
    }


def rag_secondary_node(state: LegalGenState):
    print("🔎 [RAG Secondary] Finding supporting authority...")

    p_type = state.get("petition_type", "")
    secondary_queries = {
        "Habeas Corpus":         "Article 9 liberty illegal detention writ habeas corpus warrant",
        "Post-Arrest Bail":      "Section 497 CrPC bail grant factors surety investigation incomplete",
        "Pre-Arrest Bail":       "Section 498 CrPC pre-arrest bail anticipatory mala fide FIR",
        "Quashment":             "Article 199 quashment FIR mala fide abuse process court",
        "Property/Encroachment": "Article 10A 23 natural justice notice hearing sealing property rights"
    }

    query   = secondary_queries.get(p_type, "fundamental rights Constitution Pakistan")
    results = collection.query(query_texts=[query], n_results=5)

    if not results['documents'] or not results['documents'][0]:
        return {
            "supporting_context":  "General constitutional principles apply.",
            "supporting_citation": "Constitution of Pakistan 1973"
        }

    primary_cit          = state.get("primary_citation", "")
    supporting_doc       = ""
    supporting_cit       = ""

    # Pick first result that differs from primary
    for i in range(len(results['documents'][0])):
        cit = results['metadatas'][0][i].get('case_id', '')
        if cit != primary_cit:
            supporting_doc = results['documents'][0][i]
            supporting_cit = cit
            break

    # Fallback to first result if all same
    if not supporting_cit:
        supporting_doc = results['documents'][0][0]
        supporting_cit = results['metadatas'][0][0].get('case_id', 'Constitution of Pakistan')

    print(f"   → Supporting: {supporting_cit}")
    return {
        "supporting_context":  supporting_doc,
        "supporting_citation": supporting_cit
    }

print("✅ RAG Secondary fixed — now always returns a citation")

✅ RAG Secondary fixed — now always returns a citation


In [44]:
def extract_fields_node(state: LegalGenState):
    print("📝 [Extractor] Pulling structured fields...")

    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
You are a Pakistani legal drafter. Extract and generate structured petition fields.

STORY: {state['user_story']}
PETITION TYPE: {state.get('petition_type', '')}
PRIMARY CASE RATIO: {state.get('primary_context', '')[:600]}
PRIMARY CITATION: {state.get('primary_citation', '')}
SUPPORTING CITATION: {state.get('supporting_citation', '')}

RULES:
- Extract ONLY names that appear in the STORY. Do not invent names.
- Grounds must be specific to the facts in STORY
- Reference the PRIMARY CITATION explicitly in at least 2 grounds
- Each ground must start with "That"
- Generate 5 strong grounds minimum

Return ONLY valid JSON with these exact keys:
petitioner_name: exact name from story of person filing
detenu_name: exact name from story of detained/accused/affected person
grounds: array of 5+ grounds each starting with That
prayer: specific prayer matching the petition type
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )

    result       = json.loads(resp.choices[0].message.content)
    grounds_list = result.get("grounds", [])

    return {
        "petitioner":   result.get("petitioner_name", "The Petitioner"),
        "detenu":       result.get("detenu_name",     "The Affected Person"),
        "grounds_text": "\n".join([f"{i+1}. {g}" for i, g in enumerate(grounds_list)]),
        "prayer":       result.get("prayer", "Grant the relief prayed for.")
    }

print("✅ Extractor fixed")

✅ Extractor fixed


In [45]:
TEMPLATES = {

"Habeas Corpus": """\
{court_header}

Writ Petition No. ___/2026 (Habeas Corpus)
Under Article 199(1)(b)(i) of the Constitution of Pakistan, 1973

{petitioner_name}                                    ...Petitioner
Versus
1. Station House Officer, {location}
2. {home_secretary}
3. {advocate_general}                              ...Respondents

HABEAS CORPUS PETITION

Most Respectfully Sheweth:

That the Petitioner, {petitioner_name}, most respectfully submits that {detenu_name} \
was illegally detained without lawful authority. The detention constitutes a grave \
violation of fundamental rights guaranteed under Articles 9 and 10 of the \
Constitution of Pakistan, 1973. The Supreme Court in {primary_citation} has \
categorically held that detention without warrant is impermissible in law.

GROUNDS:

{grounds_text}

{extra_grounds}

PRAYER:

It is therefore most respectfully prayed that this Honorable Court may be pleased to:
(i)   Issue a writ of Habeas Corpus directing the Respondents to produce {detenu_name} \
before this Court forthwith;
(ii)  Order the immediate and unconditional release of {detenu_name};
(iii) Declare the detention illegal, void and without lawful authority;
(iv)  Direct initiation of departmental proceedings against the delinquent officers;
(v)   Grant any other relief this Court deems just and equitable.

{verification_block}""",


"Post-Arrest Bail": """\
{court_header}

Crl. Misc. Application No. ___/2026
(Application for Post-Arrest Bail under Section 497 CrPC)

{petitioner_name} (Applicant/Accused)              ...Applicant
Versus
The State                                          ...Respondent

APPLICATION FOR POST-ARREST BAIL

Most Respectfully Sheweth:

That the Applicant, {detenu_name}, was arrested and is currently in custody. \
This application is filed under Section 497 of the Code of Criminal Procedure, 1898, \
for grant of bail. The Hon'ble Supreme Court in {primary_citation} and {supporting_citation} \
has laid down the principles governing the grant of bail which squarely apply to the \
present case.

GROUNDS:

{grounds_text}

{extra_grounds}

PRAYER:

It is therefore most respectfully prayed that this Honorable Court may be pleased to:
(i)   Grant bail to the Applicant, {detenu_name}, on such surety and conditions \
as this Court may deem fit;
(ii)  Direct that the Applicant shall not be arrested during the pendency of \
investigation/trial;
(iii) Grant any other relief this Court deems just and equitable.

{verification_block}""",


"Pre-Arrest Bail": """\
{court_header}

Crl. Misc. Application No. ___/2026
(Application for Pre-Arrest Bail under Section 498 CrPC)

{petitioner_name}                                  ...Applicant/Accused
Versus
The State                                          ...Respondent

APPLICATION FOR PRE-ARREST BAIL

Most Respectfully Sheweth:

That the Applicant apprehends arrest in connection with a malafide FIR. \
This application is filed under Section 498 of the Code of Criminal Procedure, 1898. \
The Hon'ble Supreme Court in {primary_citation} has held that pre-arrest bail \
ought to be granted where the FIR is lodged with ulterior motives and the \
applicant is not a flight risk.

GROUNDS:

{grounds_text}

{extra_grounds}

PRAYER:

It is therefore most respectfully prayed that this Honorable Court may be pleased to:
(i)   Grant ad-interim pre-arrest bail to the Applicant forthwith;
(ii)  Confirm the pre-arrest bail after notice to the State;
(iii) Direct the police to refrain from arresting the Applicant pending disposal \
of this application;
(iv)  Grant any other relief this Court deems just and equitable.

{verification_block}""",


"Quashment": """\
{court_header}

Writ Petition No. ___/2026
(Petition for Quashment of FIR under Article 199 of the Constitution)

{petitioner_name}                                  ...Petitioner
Versus
1. Station House Officer, {location}
2. The State
3. The Complainant                                 ...Respondents

CONSTITUTIONAL PETITION (QUASHMENT)

Most Respectfully Sheweth:

That the Petitioner invokes the constitutional jurisdiction of this Honorable Court \
under Article 199 of the Constitution of Pakistan, 1973, for quashment of the \
malafide FIR lodged against the Petitioner. The Supreme Court in {primary_citation} \
has held that where an FIR discloses no cognizable offence or is lodged with \
mala fide intent, this Court has jurisdiction to quash the same.

GROUNDS:

{grounds_text}

{extra_grounds}

PRAYER:

It is therefore most respectfully prayed that this Honorable Court may be pleased to:
(i)   Quash the impugned FIR and all proceedings arising therefrom;
(ii)  Restrain the Respondents from arresting the Petitioner pursuant to the \
impugned FIR;
(iii) Declare the FIR to be an abuse of the process of law;
(iv)  Grant any other relief this Court deems just and equitable.

{verification_block}""",


"Property/Encroachment": """\
{court_header}

Writ Petition No. ___/2026
(Constitutional Petition under Articles 23 & 199 — Illegal Sealing/Demolition)

{petitioner_name}                                  ...Petitioner
Versus
1. {authority}
2. The Federation of Pakistan / Province of Sindh   ...Respondents

CONSTITUTIONAL PETITION

Most Respectfully Sheweth:

That the Petitioner invokes the constitutional jurisdiction of this Honorable Court \
under Article 199 read with Articles 10A and 23 of the Constitution of Pakistan, 1973. \
The Respondent authority has taken arbitrary action against the Petitioner's property \
without due process. The Supreme Court in {primary_citation} has held that no adverse \
action can be taken against a citizen's property without prior notice and opportunity \
of hearing.

GROUNDS:

{grounds_text}

{extra_grounds}

PRAYER:

It is therefore most respectfully prayed that this Honorable Court may be pleased to:
(i)   Declare the impugned order/action of sealing/demolition as illegal, void \
and without lawful authority;
(ii)  Issue a writ of Certiorari quashing the impugned order;
(iii) Direct the Respondents to restore the Petitioner's property forthwith;
(iv)  Restrain the Respondents from taking any further coercive action;
(v)   Award costs and damages to the Petitioner;
(vi)  Grant any other relief this Court deems just and equitable.

{verification_block}"""
}

# Extra constitutional grounds added to every petition type
EXTRA_GROUNDS = {
"Habeas Corpus": """
{n}. That the Petitioner's detention violates Article 9 of the Constitution which \
guarantees that no person shall be deprived of life or liberty save in accordance with law.

{m}. That Article 10 of the Constitution mandates that every arrested person must be \
informed of the grounds of arrest and produced before a Magistrate within 24 hours \
under Section 61 CrPC — both rights have been violated.

{o}. That the Supreme Court in {supporting_citation} has further reinforced that \
illegal detention is a violation of fundamental rights and courts must exercise their \
writ jurisdiction robustly to protect citizens.""",

"Post-Arrest Bail": """
{n}. That the Applicant has no previous criminal record and poses no flight risk, \
satisfying the twin tests for bail as laid down by the Supreme Court.

{m}. That continued detention of the Applicant is punitive rather than preventive \
and violates the spirit of Article 10A of the Constitution.

{o}. That the Supreme Court in {supporting_citation} has held that bail is the rule \
and jail is the exception, particularly where investigation is incomplete.""",

"Pre-Arrest Bail": """
{n}. That the FIR has been registered belatedly and with mala fide intent to harass \
the Applicant, which entitles the Applicant to pre-arrest bail.

{m}. That the Applicant's arrest would cause irreparable harm to his reputation, \
business and family with no corresponding benefit to the investigation.

{o}. That in {supporting_citation} the Supreme Court held pre-arrest bail should be \
granted where the applicant cooperates and no incriminating evidence exists.""",

"Quashment": """
{n}. That this Honorable Court has inherent powers under Article 199 to quash any \
FIR that discloses no cognizable offence or is lodged as an abuse of process.

{m}. That allowing the impugned FIR to continue would cause irreparable harm and \
prejudice to the Petitioner with no legal justification.

{o}. That in {supporting_citation} the Supreme Court has held that mala fide FIRs \
must be quashed at the earliest stage to prevent abuse of the criminal justice system.""",

"Property/Encroachment": """
{n}. That Article 23 of the Constitution of Pakistan guarantees every citizen the \
right to acquire, hold and dispose of property — a right being violated by the \
Respondents' arbitrary action.

{m}. That Article 10A guarantees the right to a fair trial and due process, \
including the right to be heard before any adverse order is passed against a citizen.

{o}. That in {supporting_citation} the Supreme Court has held that regulatory \
authorities cannot act in excess of their statutory mandate and any such excess \
action is void ab initio."""
}

print("✅ Templates ready")

✅ Templates ready


In [46]:
def drafter_node(state: LegalGenState):
    p_type       = state.get("petition_type", "Habeas Corpus")
    jurisdiction = state.get("jurisdiction", "Sindh HC")
    court_info   = COURT_FORMATS.get(jurisdiction, COURT_FORMATS["Sindh HC"])

    print(f"✍️ [Drafter] Writing {p_type} — {jurisdiction}...")

    # Pull all values with safe defaults
    petitioner        = state.get("petitioner")   or "The Petitioner"
    detenu            = state.get("detenu")        or "The Affected Person"
    grounds_text      = state.get("grounds_text")  or "1. That the detention is illegal and without lawful authority."
    prayer            = state.get("prayer")        or "Grant the relief prayed for."
    primary_cit       = state.get("primary_citation")   or "the Supreme Court"
    supporting_cit    = state.get("supporting_citation") or "the Supreme Court"
    court_header      = court_info["full_name"]
    home_sec          = court_info["home_sec"]
    ag                = court_info["ag"]
    city              = court_info["city"]
    user_story        = state.get("user_story", "")

    # Detect location from story
    loc_match = re.search(
        r'([\w\-\s]+ police station)',
        user_story, re.IGNORECASE
    )
    location = loc_match.group(0).strip() if loc_match else "the concerned Police Station"

    # Detect authority for property cases
    story_lower = user_story.lower()
    if "kbca" in story_lower:
        authority = "Karachi Building Control Authority (KBCA)"
    elif "kmc" in story_lower:
        authority = "Karachi Metropolitan Corporation (KMC)"
    else:
        authority = "The Concerned Authority"

    # Build petition type specific instructions
    type_instructions = {
        "Habeas Corpus": f"""
HEADING    : {court_header}
TITLE      : Writ Petition No. ___/2026 (Habeas Corpus)
             Under Article 199(1)(b)(i) of the Constitution of Pakistan, 1973
PETITIONER : {petitioner}
RESPONDENTS: 1. Station House Officer, {location}
             2. {home_sec}
             3. {ag}
SECTION    : HABEAS CORPUS PETITION
INTRO PARA : That the Petitioner, {petitioner}, submits that {detenu} was illegally 
             detained without lawful authority, violating Articles 9 and 10 of the 
             Constitution. The Supreme Court in {primary_cit} held that detention 
             without warrant is impermissible. {supporting_cit} further reinforces 
             this principle.
PRAYER ITEMS:
(i)   Issue writ of Habeas Corpus directing Respondents to produce {detenu} forthwith
(ii)  Order immediate unconditional release of {detenu}
(iii) Declare detention illegal, void and without lawful authority
(iv)  Direct departmental proceedings against delinquent officers
(v)   Grant any other relief this Court deems just""",

        "Post-Arrest Bail": f"""
HEADING    : {court_header}
TITLE      : Crl. Misc. Application No. ___/2026
             Application for Post-Arrest Bail under Section 497 CrPC
PETITIONER : {detenu} (Applicant/Accused)
RESPONDENT : The State
SECTION    : APPLICATION FOR POST-ARREST BAIL
INTRO PARA : That {detenu} was arrested and is currently in custody. This application 
             is under Section 497 CrPC. The Supreme Court in {primary_cit} and 
             {supporting_cit} laid down bail principles applicable here.
PRAYER ITEMS:
(i)   Grant bail to {detenu} on surety and conditions as Court deems fit
(ii)  Direct no arrest during pendency of trial
(iii) Grant any other relief this Court deems just""",

        "Pre-Arrest Bail": f"""
HEADING    : {court_header}
TITLE      : Crl. Misc. Application No. ___/2026
             Application for Pre-Arrest Bail under Section 498 CrPC
PETITIONER : {petitioner} (Applicant/Accused)
RESPONDENT : The State
SECTION    : APPLICATION FOR PRE-ARREST BAIL
INTRO PARA : That the Applicant apprehends arrest in connection with a malafide FIR.
             Filed under Section 498 CrPC. Supreme Court in {primary_cit} held 
             pre-arrest bail should be granted where FIR is malafide and applicant 
             is not a flight risk. {supporting_cit} further supports this position.
PRAYER ITEMS:
(i)   Grant ad-interim pre-arrest bail forthwith
(ii)  Confirm bail after notice to State
(iii) Direct police to refrain from arrest pending disposal
(iv)  Grant any other relief this Court deems just""",

        "Quashment": f"""
HEADING    : {court_header}
TITLE      : Writ Petition No. ___/2026
             Petition for Quashment of FIR under Article 199 of the Constitution
PETITIONER : {petitioner}
RESPONDENTS: 1. Station House Officer, {location}
             2. The State
             3. The Complainant
SECTION    : CONSTITUTIONAL PETITION (QUASHMENT)
INTRO PARA : That the Petitioner invokes Article 199 jurisdiction for quashment of 
             the malafide FIR. Supreme Court in {primary_cit} held that where FIR 
             discloses no cognizable offence or is malafide, Court has jurisdiction 
             to quash. {supporting_cit} reinforces this power.
PRAYER ITEMS:
(i)   Quash the impugned FIR and all proceedings therefrom
(ii)  Restrain Respondents from arresting Petitioner pursuant to FIR
(iii) Declare FIR an abuse of process of law
(iv)  Grant any other relief this Court deems just""",

        "Property/Encroachment": f"""
HEADING    : {court_header}
TITLE      : Writ Petition No. ___/2026
             Constitutional Petition under Articles 23 & 199 — Illegal Sealing/Demolition
PETITIONER : {petitioner}
RESPONDENTS: 1. {authority}
             2. The Federation of Pakistan / Province of Sindh
SECTION    : CONSTITUTIONAL PETITION
INTRO PARA : That the Petitioner invokes Article 199 read with Articles 10A and 23 
             of the Constitution. The Respondent took arbitrary action against 
             Petitioner's property without due process. Supreme Court in {primary_cit} 
             held no adverse property action without prior notice and hearing. 
             {supporting_cit} further supports this position.
PRAYER ITEMS:
(i)   Declare impugned sealing/demolition order illegal, void and without lawful authority
(ii)  Issue writ of Certiorari quashing the impugned order
(iii) Direct Respondents to restore Petitioner's property forthwith
(iv)  Restrain Respondents from any further coercive action
(v)   Award costs and damages
(vi)  Grant any other relief this Court deems just"""
    }

    instructions = type_instructions.get(p_type, type_instructions["Habeas Corpus"])

    # Extra constitutional grounds per type
    extra_grounds_map = {
        "Habeas Corpus": f"""
That the Petitioner's detention violates Article 9 of the Constitution which guarantees 
no person shall be deprived of life or liberty save in accordance with law.

That Article 10 of the Constitution mandates every arrested person must be informed of 
grounds of arrest and produced before a Magistrate within 24 hours under Section 61 CrPC.

That the Supreme Court in {supporting_cit} reinforced that illegal detention violates 
fundamental rights and courts must exercise writ jurisdiction robustly.""",

        "Post-Arrest Bail": f"""
That the Applicant has no previous criminal record and poses no flight risk, satisfying 
the twin tests for bail as laid down by the Supreme Court.

That continued detention is punitive rather than preventive, violating Article 10A.

That the Supreme Court in {supporting_cit} held bail is the rule and jail the exception.""",

        "Pre-Arrest Bail": f"""
That the FIR was registered belatedly with mala fide intent to harass the Applicant.

That arrest would cause irreparable harm to reputation and family with no benefit 
to investigation.

That in {supporting_cit} the Supreme Court held pre-arrest bail should be granted 
where applicant cooperates and no incriminating evidence exists.""",

        "Quashment": f"""
That this Court has inherent powers under Article 199 to quash any FIR disclosing 
no cognizable offence or lodged as abuse of process.

That allowing the FIR to continue would cause irreparable prejudice to the Petitioner.

That in {supporting_cit} the Supreme Court held malafide FIRs must be quashed at 
earliest stage.""",

        "Property/Encroachment": f"""
That Article 23 of the Constitution guarantees every citizen the right to acquire, 
hold and dispose of property — being violated by Respondents' arbitrary action.

That Article 10A guarantees right to fair hearing before any adverse order — 
completely bypassed by Respondents.

That in {supporting_cit} the Supreme Court held regulatory authorities cannot act 
in excess of statutory mandate and such excess action is void ab initio."""
    }

    extra = extra_grounds_map.get(p_type, "")

    # Verification block
    verification = f"""VERIFICATION:

I, {petitioner}, do hereby solemnly affirm that the contents of this petition are 
true and correct to the best of my knowledge and belief and nothing material has 
been concealed therefrom.

Respectfully submitted,

_______________________          Date: ___________
{petitioner}
Through: _______________________ Advocate
         {court_info['short']}
         Enrollment No.: _______"""

    # Final prompt — fully explicit, no template variables
    full_prompt = f"""
You are a Senior Pakistani Advocate. Write a complete formal court petition.

USE EXACTLY THIS STRUCTURE AND THESE DETAILS — do not change any names, citations, 
or legal references provided:

{instructions}

GROUNDS TO INCLUDE (write each as a full formal paragraph):
{grounds_text}

ADDITIONAL CONSTITUTIONAL GROUNDS TO ADD AFTER THE ABOVE:
{extra}

VERIFICATION BLOCK TO USE EXACTLY:
{verification}

IMPORTANT RULES:
- Use ONLY the names provided above. Do not invent any new names.
- Reference {primary_cit} explicitly in at least 2 grounds by name
- Use formal Pakistani court language throughout  
- Do NOT add any disclaimer, note, or sample warning at the end
- Do NOT leave any blank lines where content should be
- The petition number should be written as ___/2026 (not filled in)
- Write the complete petition now:
"""

    chain  = ChatPromptTemplate.from_template("{text}") | llm | StrOutputParser()
    draft  = chain.invoke({"text": full_prompt})

    return {"final_petition": draft}

print("✅ Drafter fully rewritten — explicit variables, no template filling")

✅ Drafter fully rewritten — explicit variables, no template filling


In [47]:
def validator_node(state: LegalGenState):
    print("✅ [Validator] Checking petition quality...")

    p_type   = state.get("petition_type", "")
    citation = state.get("primary_citation", "")
    petition = state.get("final_petition", "")[:2500]
    story    = state.get("user_story", "")
    names_in_story = re.findall(r'[A-Z][a-z]+ [A-Z][a-z]+', story)

    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
You are reviewing a Pakistani court petition draft.

PETITION TYPE: {p_type}
PRIMARY CITATION: {citation}
NAMES IN ORIGINAL STORY: {names_in_story}

PETITION TEXT:
{petition}

Answer these checks:
1. Is "{citation}" mentioned in the GROUNDS section? true/false
2. Are there 4 or more numbered grounds? true/false
3. Does prayer match "{p_type}"? true/false
4. Are all person names in petition found in NAMES IN ORIGINAL STORY or are standard titles like SHO/Home Secretary/Advocate General? true/false

Score 1-10. Mark is_valid as true if score is 6 or above.
Only list actual problems in issues array, not things that are correct.

Return ONLY JSON with these exact keys:
citation_in_grounds: true or false
min_four_grounds: true or false
prayer_matches_type: true or false
no_hallucinated_names: true or false
score: number 1 to 10
is_valid: true or false
issues: array of strings describing only real problems
notes: one line overall assessment
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )

    result         = json.loads(resp.choices[0].message.content)
    score          = result.get("score", 7)
    is_valid       = result.get("is_valid", score >= 6)
    issues         = [i for i in result.get("issues", []) if i]
    revision_count = state.get("revision_count", 0)

    print(f"   → Valid: {is_valid} | Score: {score}/10")
    if issues:
        print(f"   → Issues: {issues}")

    if not is_valid and revision_count < 1:
        print("   ⚠️ Sending for revision...")
        return {
            "is_valid":         False,
            "validation_notes": "\n".join(issues),
            "revision_count":   revision_count + 1,
            "next_step":        "revise"
        }

    return {
        "is_valid":         True,
        "validation_notes": result.get("notes", "Approved."),
        "revision_count":   revision_count,
        "next_step":        "done"
    }

print("✅ Validator fixed")

✅ Validator fixed


In [48]:
def revision_node(state: LegalGenState):
    print("🔧 [Revision] Fixing validator issues...")

    issues = state.get("validation_notes", "")

    prompt = f"""
You are a Senior Pakistani Advocate. Revise this petition to fix the issues noted.

ISSUES TO FIX:
{issues}

ORIGINAL PETITION:
{state.get('final_petition', '')}

RULES:
- Fix ONLY the listed issues
- Keep all citations, article references, and legal content
- Do NOT add disclaimers
- Return complete revised petition only
"""
    chain    = ChatPromptTemplate.from_template("{text}") | llm | StrOutputParser()
    revised  = chain.invoke({"text": prompt})
    return {"final_petition": revised}

print("✅ Revision node ready")

✅ Revision node ready


In [49]:
def route_after_info(state):
    return state.get("next_step", "ask_user")

def route_after_rag(state):
    # Self-correct: if RAG marked retry, go back to primary RAG
    return state.get("next_step", "rag_secondary")

def route_after_validator(state):
    return state.get("next_step", "done")

# ── Build ────────────────────────────────────────────────────────────────────
builder = StateGraph(LegalGenState)

builder.add_node("info_check",    check_missing_info_node)
builder.add_node("orchestrator",  orchestrator_node)
builder.add_node("rag_primary",   rag_primary_node)
builder.add_node("rag_secondary", rag_secondary_node)
builder.add_node("extractor",     extract_fields_node)
builder.add_node("drafter",       drafter_node)
builder.add_node("validator",     validator_node)
builder.add_node("revision",      revision_node)

# Flow
builder.add_edge(START, "info_check")

builder.add_conditional_edges(
    "info_check",
    route_after_info,
    {"ask_user": END, "proceed": "orchestrator"}
)

builder.add_edge("orchestrator", "rag_primary")

builder.add_conditional_edges(
    "rag_primary",
    route_after_rag,
    {
        "rag_retry":     "rag_primary",    # self-correct loop
        "rag_secondary": "rag_secondary"
    }
)

builder.add_edge("rag_secondary", "extractor")
builder.add_edge("extractor",     "drafter")
builder.add_edge("drafter",       "validator")

builder.add_conditional_edges(
    "validator",
    route_after_validator,
    {
        "revise": "revision",
        "done":   END
    }
)

builder.add_edge("revision", END)

legal_gen_app = builder.compile()
print("✅ Full graph compiled")
print("   Flow: InfoCheck → Orchestrator → RAG(x2) → Extractor → Drafter → Validator → [Revision] → END")


✅ Full graph compiled
   Flow: InfoCheck → Orchestrator → RAG(x2) → Extractor → Drafter → Validator → [Revision] → END


In [50]:
def run_legal_assistant(story: str, user_id: str = "default"):
    """
    Full pipeline with:
    - Regex info check (no LLM)
    - Jurisdiction auto-detection + manual override
    - Multi-hop RAG with self-correction
    - Type-aware templates
    - Validator + revision
    - Memory per user
    """
    # Load memory context
    mem_ctx = get_memory_context(user_id)
    if USER_MEMORY.get(user_id, {}).get("past_petitions"):
        print(f"💾 Memory loaded — {len(USER_MEMORY[user_id]['past_petitions'])} past petition(s) found")

    state = {
        "user_story":          story,
        "user_id":             user_id,
        "jurisdiction":        "",
        "is_complete":         False,
        "missing_info":        [],
        "petition_type":       "",
        "target_court":        "",
        "next_step":           "",
        "primary_context":     "",
        "primary_citation":    "",
        "supporting_context":  "",
        "supporting_citation": "",
        "rag_attempts":        0,
        "petitioner":          "",
        "detenu":              "",
        "grounds_text":        "",
        "prayer":              "",
        "is_valid":            False,
        "validation_notes":    "",
        "revision_count":      0,
        "final_petition":      "",
        "memory_context":      mem_ctx
    }

    # ── Step 1: Info check (once only) ───────────────────────────────
    info_result = check_missing_info_node(state)
    state.update(info_result)

    if not state["is_complete"]:
        print(f"\n📋 Missing: {', '.join(state['missing_info'])}")
        reply = input("👤 Please provide these details: ")
        state["user_story"] += f"\nAdditional details: {reply}"
        state["is_complete"] = True
        state["next_step"]   = "proceed"

    # ── Step 2: Optional jurisdiction override ────────────────────────
    print("\n🏛️  Jurisdiction (press Enter to auto-detect, or type: Sindh HC / Lahore HC / Islamabad HC / Sessions Court Karachi / Sessions Court Lahore):")
    jur_input = input("   Jurisdiction: ").strip()
    if jur_input in COURT_FORMATS:
        state["jurisdiction"] = jur_input
        print(f"   → Using: {jur_input}")
    else:
        print("   → Auto-detecting from story...")

    # ── Step 3: Run pipeline (skip info_check node) ───────────────────
    from langgraph.graph import StateGraph, START, END as LEND

    builder2 = StateGraph(LegalGenState)
    builder2.add_node("orchestrator",  orchestrator_node)
    builder2.add_node("rag_primary",   rag_primary_node)
    builder2.add_node("rag_secondary", rag_secondary_node)
    builder2.add_node("extractor",     extract_fields_node)
    builder2.add_node("drafter",       drafter_node)
    builder2.add_node("validator",     validator_node)
    builder2.add_node("revision",      revision_node)

    builder2.add_edge(START,           "orchestrator")
    builder2.add_edge("orchestrator",  "rag_primary")
    builder2.add_conditional_edges(
        "rag_primary", route_after_rag,
        {"rag_retry": "rag_primary", "rag_secondary": "rag_secondary"}
    )
    builder2.add_edge("rag_secondary", "extractor")
    builder2.add_edge("extractor",     "drafter")
    builder2.add_edge("drafter",       "validator")
    builder2.add_conditional_edges(
        "validator", route_after_validator,
        {"revise": "revision", "done": LEND}
    )
    builder2.add_edge("revision", LEND)

    pipeline = builder2.compile()
    output   = pipeline.invoke(state)
    state.update(output)

    # ── Step 4: Save to memory ────────────────────────────────────────
    save_to_memory(user_id, state)

    # ── Step 5: Print result ──────────────────────────────────────────
    if state.get("final_petition"):
        print("\n" + "="*65)
        print(f"⚖️  {state.get('petition_type', 'LEGAL').upper()}")
        print(f"🏛️  {state.get('jurisdiction', '')}")
        print(f"📌 Primary Citation:    {state.get('primary_citation', 'N/A')}")
        print(f"📌 Supporting Citation: {state.get('supporting_citation', 'N/A')}")
        print(f"✅ Validated: {state.get('is_valid')} | {state.get('validation_notes', '')}")
        print("="*65 + "\n")
        print(state["final_petition"])

        # Save to file
        fname = f"/kaggle/working/{user_id}_{state.get('petition_type','petition').replace(' ','_')}.txt"
        with open(fname, "w") as f:
            f.write(state["final_petition"])
        print(f"\n💾 Saved to: {fname}")
    else:
        print("⚠️ No petition generated.")

In [51]:
test_cases = [
    ("rafsha_01", "My brother Ali Khan was arrested without warrant from our home in Gulshan-e-Iqbal by plain clothes officers on March 24 2026. My name is Rafsha Rahim. He is at Gulshan Police Station."),
    ("ahmed_01",  "An FIR has been registered against me at Clifton Police Station under Section 324 PPC. I have not been arrested yet. My name is Ahmed Raza. FIR date March 28 2026."),
    ("sara_01",   "My client Bilal Sheikh was arrested on March 20 2026 and has been in custody for 12 days at Defence Police Station under Section 409 PPC. I am his lawyer Sara Khan."),
    ("tariq_01",  "My shop in Saddar Karachi was sealed by KBCA on March 25 2026 without any prior notice. I am Tariq Mehmood and have valid lease documents.")
]

for user_id, story in test_cases:
    print(f"\n{'='*65}")
    print(f"🧪 {user_id.upper()}")
    print(f"{'='*65}")
    run_legal_assistant(story, user_id=user_id)
    print("\n")


🧪 RAFSHA_01
📋 [InfoCheck] Scanning for missing details...
   → Complete: True | Missing: []

🏛️  Jurisdiction (press Enter to auto-detect, or type: Sindh HC / Lahore HC / Islamabad HC / Sessions Court Karachi / Sessions Court Lahore):


   Jurisdiction:  


   → Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Habeas Corpus | Jurisdiction: Sindh HC | Reason: STEP 4: Person detained/arrested with NO FIR mentioned?
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: Crl.P.L.A.660_2024 (attempt 1)
   ⚠️ Not relevant (The case summary is unrelated to the user's brother's arrest in Karachi, as it pertains to a different case in Balochistan.) — retrying...
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: Crl.P.L.A.1038_2024 (attempt 2)
   ⚠️ Not relevant (The case summary is unrelated to the arrest of Rafsha's brother Ali Khan in Gulshan-e-Iqbal.) — retrying...
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: C.P.L.A.3637_2019 (attempt 3)
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: C.P.L.A.1809_2020
📝 [Extractor] Pulling structured fields...
✍️ [Drafter] Writing Habeas Corpus — Sindh HC...
✅ [Validator] Checking petition quality...
   → Valid: True | Sc

   Jurisdiction:  


   → Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Pre-Arrest Bail | Jurisdiction: Sessions Court Karachi | Reason: Step 3: Is there an FIR but person NOT yet arrested? -> YES = 'Pre-Arrest Bail' because an FIR has been registered against Ahmed Raza under Section 324 PPC, but he has not been arrested yet.
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: Crl.P.L.A.1117_2024 (attempt 1)
   ⚠️ Not relevant (Different FIR number, date, and police station, also different sections of PPC) — retrying...
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: Crl.P.L.A.645-L_2025 (attempt 2)
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: Crl.P.L.A.1075-L_2020
📝 [Extractor] Pulling structured fields...
✍️ [Drafter] Writing Pre-Arrest Bail — Sessions Court Karachi...
✅ [Validator] Checking petition quality...
   → Valid: True | Score: 8/10
   → Issues: ['Grounds section is not clearly defined, making it hard to distinguish

   Jurisdiction:  


   → Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Post-Arrest Bail | Jurisdiction: Sessions Court Karachi | Reason: STEP 2: Is there an FIR AND the person is already in custody/arrested? -> YES = 'Post-Arrest Bail' because there is a mention of arrest and custody, and a specific section (409 PPC) is mentioned which implies an FIR is involved.
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: Crl.P.L.A.1117_2024 (attempt 1)
   ⚠️ Not relevant (The case summary is from 2024, but the arrest occurred in 2026, making it irrelevant to the current situation.) — retrying...
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: Crl.P.L.A.974-L_2025 (attempt 2)
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: Crl.P.L.A.298_2023
📝 [Extractor] Pulling structured fields...
✍️ [Drafter] Writing Post-Arrest Bail — Sessions Court Karachi...
✅ [Validator] Checking petition quality...
   → Valid: True | Score: 8/10
   → Issues: ['

   Jurisdiction:  


   → Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Property/Encroachment | Jurisdiction: Sindh HC | Reason: STEP 1: Mention of property (shop), sealing by KBCA, and valid lease documents.
🔎 [RAG Primary] Searching 9226 judgments...
   → Primary found: C.P.L.A.1331-L_2017 (attempt 1)
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: C.P.L.A.767_2022
📝 [Extractor] Pulling structured fields...
✍️ [Drafter] Writing Property/Encroachment — Sindh HC...
✅ [Validator] Checking petition quality...
   → Valid: True | Score: 8/10
   → Issues: ['C.P.L.A.1331-L_2017 is not mentioned in the GROUNDS section', 'The petition does not clearly state the specific financial losses and hardship caused by the sealing of the shop']
💾 Memory saved for user: tariq_01

⚖️  PROPERTY/ENCROACHMENT
🏛️  Sindh HC
📌 Primary Citation:    C.P.L.A.1331-L_2017
📌 Supporting Citation: C.P.L.A.767_2022
✅ Validated: True | Overall, the petition is well-structured and effectively 

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/tariq_01_Property/Encroachment.txt'

In [23]:
# =========================
# ✅ COMPLETE UPGRADED LEGAL AGENT
# =========================

import os, json, re, shutil
from typing import TypedDict, List
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# =========================
# 🔐 CONFIG
# =========================
GROQ_API_KEY = "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"

SOURCE_DB  = "/kaggle/input/datasets/rafsharahim/chromadb1"
WORKING_DB = "/kaggle/working/legal_rag_db"

if os.path.exists(WORKING_DB):
    shutil.rmtree(WORKING_DB)
shutil.copytree(SOURCE_DB, WORKING_DB)

client = chromadb.PersistentClient(path=WORKING_DB)

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-large-en-v1.5"
)

collection = client.get_collection(
    name="legal_judgments_collection",
    embedding_function=embed_fn
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

groq_client = Groq(api_key=GROQ_API_KEY)

# =========================
# 🧠 MEMORY STORE
# =========================
user_memory = {}  # {user_id: {"style": "..."}}

# =========================
# 📦 STATE
# =========================
class LegalGenState(TypedDict):
    user_id: str
    user_story: str
    is_complete: bool
    missing_info: List[str]
    petition_type: str
    target_court: str
    jurisdiction: str
    research_context: str
    supporting_context: str
    case_id: str
    is_relevant: bool
    final_petition: str
    validated: bool

# =========================
# 🧩 TEMPLATE SYSTEM
# =========================
TEMPLATES = {
    "Habeas Corpus": "Illegal detention, Article 199, liberty violation",
    "Pre-Arrest Bail": "Apprehension of arrest, mala fide intent",
    "Post-Arrest Bail": "Further inquiry, no recovery",
    "Quashment": "FIR lacks ingredients of offence",
    "Property": "Illegal encroachment, possession dispute"
}

COURT_FORMAT = {
    "Sindh High Court": "IN THE HIGH COURT OF SINDH AT KARACHI",
    "Lahore High Court": "IN THE LAHORE HIGH COURT, LAHORE",
    "Islamabad High Court": "IN THE ISLAMABAD HIGH COURT"
}

# =========================
# 🔍 SELF-CORRECTING RAG
# =========================
def smart_rag_query(query, retries=2):
    for i in range(retries):
        results = collection.query(query_texts=[query], n_results=2)

        context = results['documents'][0][0]
        case_id = results['metadatas'][0][0].get('case_id', 'Unknown')

        # relevance check
        check_prompt = f"""
You are a Pakistani legal expert.

QUERY TYPE: {query}

CASE: {context}

TASK:
1. Identify case type (Habeas Corpus / Bail / Civil / Criminal etc.)
2. Decide if it is STRONGLY relevant.

Return JSON:
{{
"type": "...",
"relevant": true/false
}}
"""
        resp = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": check_prompt}],
            response_format={"type": "json_object"}
        )

        relevant = json.loads(resp.choices[0].message.content)["relevant"]

        if relevant:
            return context, case_id

        # refine query
        query = query + " constitutional rights Pakistan illegal detention"

    return context, case_id  # fallback

# =========================
# 🧠 MULTI-HOP RAG NODE
# =========================
def rag_node(state: LegalGenState):
    print("🔎 Multi-hop RAG...")

    primary_context, case_id = smart_rag_query(state["user_story"])

    supporting_query = "supporting precedent for " + state["petition_type"]
    supporting_context, _ = smart_rag_query(supporting_query)

    return {
        "research_context": primary_context,
        "supporting_context": supporting_context,
        "case_id": case_id,
        "is_relevant": True
    }

# =========================
# 🧠 ORCHESTRATOR
# =========================
def orchestrator(state: LegalGenState):
    prompt = f"""
Classify case:
{state['user_story']}

Return JSON:
{{"petition_type": "...", "jurisdiction": "Sindh High Court"}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    data = json.loads(resp.choices[0].message.content)

    return {
        "petition_type": data.get("petition_type", "Habeas Corpus"),
        "jurisdiction": data.get("jurisdiction", "Sindh High Court"),
        "target_court": data.get("jurisdiction", "Sindh High Court")
    }

# =========================
# ✍️ TEMPLATE-AWARE DRAFTER
# =========================
def drafter(state: LegalGenState):
    print("✍️ Drafting...")

    court_heading = COURT_FORMAT.get(state["jurisdiction"], "HIGH COURT")

    style = user_memory.get(state["user_id"], {}).get("style", "formal")

    template_hint = TEMPLATES.get(state["petition_type"], "")

    prompt = ChatPromptTemplate.from_template("""
Draft a Pakistani legal petition.

COURT: {court}
TYPE: {ptype}
STYLE: {style}

PRIMARY CASE:
{context}

SUPPORTING CASE:
{support}

HINT:
{template_hint}

FACTS:
{story}

Requirements:
- Proper legal format
- Strong grounds (That...)
- Use BOTH precedents
""")

    chain = prompt | llm | StrOutputParser()

    draft = chain.invoke({
        "court": court_heading,
        "ptype": state["petition_type"],
        "style": style,
        "context": state["research_context"],
        "support": state["supporting_context"],
        "template_hint": template_hint,
        "story": state["user_story"]
    })

    return {"final_petition": draft}

# =========================
# ✅ VALIDATOR NODE
# =========================
def validator(state: LegalGenState):
    print("🛡️ Validating petition...")

    prompt = f"""
Check this petition:

{state['final_petition']}

Return JSON:
{{
"valid": true/false,
"issues": ["..."],
"fix": "improved version if needed"
}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    result = json.loads(resp.choices[0].message.content)

    if not result["valid"]:
        return {"final_petition": result["fix"], "validated": True}

    return {"validated": True}

# =========================
# 🧠 MEMORY UPDATE
# =========================
def memory_node(state: LegalGenState):
    user_memory[state["user_id"]] = {
        "style": "formal structured detailed"
    }
    return {}

# =========================
# 🔗 GRAPH
# =========================
builder = StateGraph(LegalGenState)

builder.add_node("orchestrator", orchestrator)
builder.add_node("rag", rag_node)
builder.add_node("drafter", drafter)
builder.add_node("validator", validator)
builder.add_node("memory", memory_node)

builder.add_edge(START, "orchestrator")
builder.add_edge("orchestrator", "rag")
builder.add_edge("rag", "drafter")
builder.add_edge("drafter", "validator")
builder.add_edge("validator", "memory")
builder.add_edge("memory", END)

app = builder.compile()

# =========================
# ▶️ RUN FUNCTION
# =========================
def run_legal_assistant(user_id, story):
    state = {
        "user_id": user_id,
        "user_story": story,
        "is_complete": True,
        "missing_info": [],
        "petition_type": "",
        "target_court": "",
        "jurisdiction": "",
        "research_context": "",
        "supporting_context": "",
        "case_id": "",
        "is_relevant": True,
        "final_petition": "",
        "validated": False
    }

    result = app.invoke(state)

    print("\n" + "="*60)
    print(f"⚖️ {result['petition_type']}")
    print(f"🏛️ {result['target_court']}")
    print("="*60)
    print(result["final_petition"])

# VERSION 3

In [75]:
run_legal_assistant(
    user_id="user_1",
    story="I, Ahmed Ali, state that my brother Bilal Ali was picked up by police from our house in Gulshan-e-Iqbal, Karachi on 30 March 2026 without any warrant and his whereabouts are unknown."
)

🔎 Multi-hop RAG...
✍️ Drafting...
🛡️ Validating petition...

⚖️ Habeas Corpus
🏛️ Sindh High Court
IN THE HIGH COURT OF SINDH AT KARACHI

HABEAS CORPUS PETITION NO. _______ OF 2026

Ahmed Ali
Petitioner
Versus
The State
Respondent

TO,
The Honourable Chief Justice and Judges of the High Court of Sindh at Karachi.

The humble petition of the petitioner above named, respectfully shows:

1. That the petitioner is a citizen of Pakistan and a resident of Karachi.
2. That the petitioner's brother, Bilal Ali, was picked up by the police from their house in Gulshan-e-Iqbal, Karachi on 30 March 2026 without any warrant or legal justification.
3. That despite efforts to locate him, the whereabouts of Bilal Ali are still unknown, and the petitioner is deeply concerned about his safety and well-being.

THAT:
The detention of Bilal Ali is illegal and in violation of his fundamental right to liberty as enshrined in Article 9 of the Constitution of the Islamic Republic of Pakistan, 1973.
The petitione

In [76]:
# =========================
# ⚖️ UPGRADED LEGAL ASSISTANT
# =========================

import os, json, shutil
from typing import TypedDict, List
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# =========================
# 🔐 CONFIGURATION
# =========================
GROQ_API_KEY = "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"

SOURCE_DB  = "/kaggle/input/datasets/rafsharahim/chromadb1"
WORKING_DB = "/kaggle/working/legal_rag_db"

if os.path.exists(WORKING_DB):
    shutil.rmtree(WORKING_DB)
shutil.copytree(SOURCE_DB, WORKING_DB)

client = chromadb.PersistentClient(path=WORKING_DB)
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-large-en-v1.5")
collection = client.get_collection(name="legal_judgments_collection", embedding_function=embed_fn)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

# =========================
# 🧠 MEMORY STORE
# =========================
user_memory = {}  # user_id -> {"style": "formal / concise / detailed"}

# =========================
# 📦 STATE DEFINITION
# =========================
class LegalGenState(TypedDict):
    user_id: str
    user_story: str
    is_complete: bool
    missing_info: List[str]
    petition_type: str
    target_court: str
    jurisdiction: str
    research_context: str
    supporting_context: str
    case_id: str
    is_relevant: bool
    final_petition: str
    validated: bool

# =========================
# 🧩 TEMPLATES & COURT FORMATS
# =========================
TEMPLATES = {
    "Habeas Corpus": "Illegal detention, Article 199, liberty violation",
    "Pre-Arrest Bail": "Apprehension of arrest, mala fide intent",
    "Post-Arrest Bail": "Further inquiry, no recovery",
    "Quashment": "FIR lacks ingredients of offence",
    "Property": "Illegal encroachment, possession dispute"
}

COURT_FORMAT = {
    "Sindh High Court": "IN THE HIGH COURT OF SINDH AT KARACHI",
    "Lahore High Court": "IN THE LAHORE HIGH COURT, LAHORE",
    "Islamabad High Court": "IN THE ISLAMABAD HIGH COURT"
}

# =========================
# 🔍 SELF-CORRECTING RAG QUERY
# =========================
def smart_rag_query(query, retries=2):
    for i in range(retries):
        results = collection.query(query_texts=[query], n_results=2)
        context = results['documents'][0][0]
        case_id = results['metadatas'][0][0].get('case_id', 'Unknown')

        # Relevance check via LLM
        check_prompt = f"""
You are a Pakistani legal expert. 

Query: {query}
Case Context: {context}

Task:
1. Identify case type (Habeas Corpus / Bail / Civil / Criminal etc.)
2. Decide if it is strongly relevant

Return JSON:
{{"type": "...", "relevant": true/false}}
"""
        resp = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": check_prompt}],
            response_format={"type": "json_object"}
        )
        relevant = json.loads(resp.choices[0].message.content)["relevant"]

        if relevant:
            return context, case_id
        # Refine query if not relevant
        query += " constitutional rights Pakistan illegal detention"

    return context, case_id  # fallback

# =========================
# 🧠 MULTI-HOP RAG NODE
# =========================
def rag_node(state: LegalGenState):
    print("🔎 Multi-hop RAG...")
    primary_context, case_id = smart_rag_query(state["user_story"])
    supporting_query = f"supporting precedent for {state['petition_type']}"
    supporting_context, _ = smart_rag_query(supporting_query)
    return {
        "research_context": primary_context,
        "supporting_context": supporting_context,
        "case_id": case_id,
        "is_relevant": True
    }

# =========================
# 🧠 ORCHESTRATOR
# =========================
def orchestrator(state: LegalGenState):
    prompt = f"""
Classify the legal matter:
{state['user_story']}

Return JSON:
{{"petition_type": "...", "jurisdiction": "Sindh High Court"}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    data = json.loads(resp.choices[0].message.content)
    return {
        "petition_type": data.get("petition_type", "Habeas Corpus"),
        "jurisdiction": data.get("jurisdiction", "Sindh High Court"),
        "target_court": data.get("jurisdiction", "Sindh High Court")
    }

# =========================
# ✍️ TEMPLATE-AWARE DRAFTER
# =========================
def drafter(state: LegalGenState):
    print("✍️ Drafting...")

    court_heading = COURT_FORMAT.get(state["jurisdiction"], "HIGH COURT")
    style = user_memory.get(state["user_id"], {}).get("style", "formal structured detailed")
    template_hint = TEMPLATES.get(state["petition_type"], "")

    prompt = ChatPromptTemplate.from_template("""
Draft a Pakistani legal petition.

COURT: {court}
TYPE: {ptype}
STYLE: {style}

PRIMARY CASE:
{context}

SUPPORTING CASE:
{support}

HINT:
{template_hint}

FACTS:
{story}

Requirements:
- Proper legal format
- Numbered grounds (That...)
- Use both precedents clearly
- Concise, formal language
""")
    chain = prompt | llm | StrOutputParser()
    draft = chain.invoke({
        "court": court_heading,
        "ptype": state["petition_type"],
        "style": style,
        "context": state["research_context"],
        "support": state["supporting_context"],
        "template_hint": template_hint,
        "story": state["user_story"]
    })
    return {"final_petition": draft}

# =========================
# ✅ VALIDATOR NODE
# =========================
def validator(state: LegalGenState):
    print("🛡️ Validating petition...")
    prompt = f"""
Check this petition for:
- Proper numbering of grounds
- Precedent usage
- Citation of constitutional rights (Articles 9/10)
- Missing petitioner info

Petition Text:
{state['final_petition']}

Return JSON:
{{"valid": true/false, "issues": ["..."], "fix": "improved version if needed"}}
"""
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    result = json.loads(resp.choices[0].message.content)
    if not result["valid"]:
        return {"final_petition": result["fix"], "validated": True}
    return {"validated": True}

# =========================
# 🧠 MEMORY UPDATE
# =========================
def memory_node(state: LegalGenState):
    user_memory[state["user_id"]] = {"style": "formal structured detailed"}
    return {}

# =========================
# 🔗 STATE GRAPH
# =========================
builder = StateGraph(LegalGenState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("rag", rag_node)
builder.add_node("drafter", drafter)
builder.add_node("validator", validator)
builder.add_node("memory", memory_node)
builder.add_edge(START, "orchestrator")
builder.add_edge("orchestrator", "rag")
builder.add_edge("rag", "drafter")
builder.add_edge("drafter", "validator")
builder.add_edge("validator", "memory")
builder.add_edge("memory", END)
app = builder.compile()

# =========================
# ▶️ RUN FUNCTION
# =========================
def run_legal_assistant(user_id: str, story: str):
    state = {
        "user_id": user_id,
        "user_story": story,
        "is_complete": True,
        "missing_info": [],
        "petition_type": "",
        "target_court": "",
        "jurisdiction": "",
        "research_context": "",
        "supporting_context": "",
        "case_id": "",
        "is_relevant": True,
        "final_petition": "",
        "validated": False
    }
    result = app.invoke(state)
    print("\n" + "="*60)
    print(f"⚖️ {result['petition_type']}")
    print(f"🏛️ {result['target_court']}")
    print("="*60)
    print(result["final_petition"])

In [77]:
run_legal_assistant(
    user_id="user_1",
    story="I, Ahmed Ali, state that my brother Bilal Ali was picked up by police from our house in Gulshan-e-Iqbal, Karachi on 30 March 2026 without any warrant and his whereabouts are unknown."
)

🔎 Multi-hop RAG...
✍️ Drafting...
🛡️ Validating petition...

⚖️ Habeas Corpus
🏛️ Sindh High Court
IN THE HIGH COURT OF SINDH AT KARACHI

Habeas Corpus Petition No. _______ of 2026

Ahmed Ali
Petitioner
Versus
The State
Respondent

To,
The Honourable Chief Justice and Judges of the High Court of Sindh at Karachi.

The humble petition of the petitioner above named, most respectfully shows:

1. That the petitioner is the brother of Bilal Ali, who was picked up by the police from their house in Gulshan-e-Iqbal, Karachi on 30 March 2026 without any warrant.
2. That the petitioner has been unable to ascertain the whereabouts of his brother, Bilal Ali, and is apprehensive that he is being detained illegally by the respondents.

That:
1. The petitioner has reasonable grounds to believe that his brother, Bilal Ali, is being detained in violation of his fundamental right to liberty, as enshrined in Article 199 of the Constitution of the Islamic Republic of Pakistan.
2. The petitioner submits tha

In [82]:
# =========================
# PDF-READY LEGAL PETITION GENERATOR
# =========================
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch

def build_petition_text(petition_dict, court="Sindh High Court"):
    """Converts the RAG/dict draft into a fully formatted petition string"""
    court_heading = f"IN THE HIGH COURT OF {court.upper()}\n\n"
    petition_no = "HABEAS CORPUS PETITION NO. _______ OF 2026\n\n"
    petitioner_name = petition_dict.get("petitioner_info", {}).get("name", "Petitioner")
    petitioner_line = f"{petitioner_name}\nPetitioner\n"
    respondent_line = "Versus\nThe State\nRespondent\n\n"

    body = [court_heading, petition_no, petitioner_line, respondent_line]

    # Intro
    body.append("To,\nThe Honourable Chief Justice and Judges of the High Court of Sindh at Karachi.\n\n")
    body.append("The humble petition of the petitioner above named, most respectfully shows:\n\n")

    # Grounds / numbered facts
    for i, g in enumerate(petition_dict.get("grounds", []), start=1):
        body.append(f"{i}. {g}\n")

    # Supporting precedents
    if petition_dict.get("precedent_usage"):
        body.append("\nSupporting Precedents:\n")
        for k, v in petition_dict["precedent_usage"].items():
            body.append(f"- {v}\n")

    # Constitutional rights
    if petition_dict.get("const_rights"):
        body.append("\nConstitutional Rights Violated:\n")
        for k, v in petition_dict["const_rights"].items():
            body.append(f"- {v}\n")

    # Prayer / relief
    body.append("\nWHEREFORE, the petitioner prays that this Honourable Court may be pleased to:\n")
    body.append("* Issue a writ of Habeas Corpus, directing the respondents to produce the petitioner;\n")
    body.append("* Declare the detention illegal and unconstitutional;\n")
    body.append("* Direct the respondents to release the petitioner forthwith;\n")
    body.append("* Pass any other order deemed fit.\n")

    # Verification
    body.append("\nVerified at Karachi on this ___ day of 2026.\n")
    body.append(f"Signed,\n{petitioner_name}\nPetitioner\n")

    return "\n".join(body)


def save_petition_as_pdf(petition_text, filename="petition.pdf"):
    """Generates a well-formatted PDF using ReportLab"""
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4
    textobject = c.beginText()
    textobject.setTextOrigin(inch, height - inch)
    textobject.setFont("Times-Roman", 12)
    line_spacing = 14

    for line in petition_text.split("\n"):
        if line.strip() == "":
            textobject.moveCursor(0, -line_spacing / 2)  # smaller spacing for blank lines
        else:
            if line.endswith(":"):
                # bold-ish simulation for headings
                textobject.setFont("Times-Bold", 12)
            else:
                textobject.setFont("Times-Roman", 12)
            textobject.textLine(line)
        # Page break
        if textobject.getY() < inch:
            c.drawText(textobject)
            c.showPage()
            textobject = c.beginText()
            textobject.setTextOrigin(inch, height - inch)

    c.drawText(textobject)
    c.save()
    print(f"✅ Petition saved as PDF: {filename}")


def run_legal_assistant(user_id: str, story: str, pdf_filename="Habeas_Corpus_Petition.pdf"):
    state = {
        "user_id": user_id,
        "user_story": story,
        "is_complete": True,
        "missing_info": [],
        "petition_type": "",
        "target_court": "",
        "jurisdiction": "",
        "research_context": "",
        "supporting_context": "",
        "case_id": "",
        "is_relevant": True,
        "final_petition": "",
        "validated": False
    }

    result = app.invoke(state)

    # Convert dict draft to full petition body if needed
    if isinstance(result["final_petition"], dict):
        petition_text = build_petition_text(result["final_petition"])
    else:
        petition_text = result["final_petition"]

    print("\n" + "="*60)
    print(f"⚖️ {result['petition_type']}")
    print(f"🏛️ {result['target_court']}")
    print("="*60)
    print(petition_text)

    # Save to PDF
    pdf_path = f"/kaggle/working/{pdf_filename}"
    save_petition_as_pdf(petition_text, pdf_path)
    return pdf_path


# =========================
# Example Usage
# =========================
user_story = """
My brother Bilal Ali was picked up by Karachi police from our house on 30 March 2026 
without any warrant or legal justification. We cannot locate him and fear illegal detention.
"""
pdf_file = run_legal_assistant("user123", user_story)
print(f"PDF available at: {pdf_file}")

🔎 Multi-hop RAG...
✍️ Drafting...
🛡️ Validating petition...

⚖️ Habeas Corpus
🏛️ Sindh High Court
IN THE HIGH COURT OF SINDH AT KARACHI

HABEAS CORPUS PETITION NO. _______ OF 2026

BILAL ALI
PETITIONER

VERSUS

THE STATE
RESPONDENT

TO,
THE HON'BLE CHIEF JUSTICE AND HIS LORDSHIP'S COMPANION JUSTICES OF THE HON'BLE HIGH COURT OF SINDH AT KARACHI.

THE HUMBLE PETITION OF THE PETITIONER ABOVE NAMED:

MOST RESPECTFULLY SHOWETH:

1. That the petitioner is the brother of Bilal Ali, who was picked up by the Karachi police from their house on 30 March 2026 without any warrant or legal justification.
2. That the petitioner and his family are unable to locate Bilal Ali and fear that he is being illegally detained by the police.
3. That the petitioner has approached this Hon'ble Court seeking relief under Article 199 of the Constitution of the Islamic Republic of Pakistan, which guarantees the fundamental right to liberty and protection against arbitrary detention.

THAT:
1. The detention of Bila

In [79]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.6 MB/s eta 0:00:00a 0:00:01


# VERSION 4

In [72]:
# ── CELL 1: Imports & LLM Setup ──────────────────────────────────────────────
import os, re, json, time
from groq import Groq
from typing import TypedDict, List, Optional
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_KEY_HERE")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ Imports & LLM ready")

✅ Imports & LLM ready


In [73]:
# ── CELL 2: In-Session Memory ────────────────────────────────────────────────
USER_MEMORY = {}

def save_to_memory(user_id: str, state: dict):
    if user_id not in USER_MEMORY:
        USER_MEMORY[user_id] = {
            "past_petitions": [],
            "preferred_court": None,
            "citations_used":  [],
            "petition_types":  []
        }
    mem = USER_MEMORY[user_id]
    mem["past_petitions"].append({
        "story":       state.get("user_story", "")[:200],
        "type":        state.get("petition_type", ""),
        "court":       state.get("jurisdiction", ""),
        "citation":    state.get("primary_citation", ""),
        "draft_style": state.get("final_petition", "")[:300]
    })
    if state.get("jurisdiction"):
        mem["preferred_court"] = state["jurisdiction"]
    if state.get("primary_citation"):
        mem["citations_used"].append(state["primary_citation"])
    if state.get("petition_type"):
        mem["petition_types"].append(state["petition_type"])
    print(f"💾 Memory saved for user: {user_id}")

def get_memory_context(user_id: str) -> str:
    if user_id not in USER_MEMORY or not USER_MEMORY[user_id]["past_petitions"]:
        return "No previous petitions on record."
    mem = USER_MEMORY[user_id]
    return (
        f"User has filed {len(mem['past_petitions'])} previous petition(s).\n"
        f"Preferred court: {mem['preferred_court'] or 'Not set'}\n"
        f"Most used type: {max(set(mem['petition_types']), key=mem['petition_types'].count) if mem['petition_types'] else 'None'}\n"
        f"Recent citations: {', '.join(mem['citations_used'][-3:]) or 'None'}\n"
        f"Last draft sample: {mem['past_petitions'][-1]['draft_style'] if mem['past_petitions'] else 'None'}"
    )

print("✅ Memory system ready")

✅ Memory system ready


In [74]:
# ── CELL 3: LangGraph State ───────────────────────────────────────────────────
class LegalGenState(TypedDict):
    user_story:          str
    user_id:             str
    jurisdiction:        str

    is_complete:         bool
    missing_info:        List[str]

    petition_type:       str
    target_court:        str
    next_step:           str

    primary_context:     str
    primary_citation:    str
    supporting_context:  str
    supporting_citation: str
    rag_attempts:        int

    petitioner:          str
    detenu:              str
    facts_text:          str      # ← NEW: separate FACTS section
    grounds_text:        str
    prayer:              str

    is_valid:            bool
    validation_notes:    str
    validation_score:    int      # ← NEW: numeric score from validator
    revision_count:      int

    final_petition:      str
    memory_context:      str

    # ── Evaluation metrics (NEW) ────────────────────────────────────────
    eval_structure_score:   int   # 0-10: FACTS / GROUNDS / PRAYER present
    eval_citation_score:    int   # 0-10: citations tied to propositions
    eval_grounds_count:     int   # number of lettered grounds
    eval_redundancy_score:  int   # 0-10 (10 = no redundancy)
    eval_tone_score:        int   # 0-10 (10 = sharp habeas/bail tone)
    eval_overall_score:     float # weighted average
    eval_issues:            List[str]
    eval_timestamp:         str

print("✅ State defined (with facts_text + eval fields)")

✅ State defined (with facts_text + eval fields)


In [75]:
# ── CELL 4: Info Check Node ──────────────────────────────────────────────────
def check_missing_info_node(state: LegalGenState) -> dict:
    print("📋 [InfoCheck] Scanning for missing details...")
    text = state["user_story"].lower()
    orig = state["user_story"]
    missing = []

    has_filer = bool(
        re.search(r'my name is', text) or
        re.search(r'\bi am\b|\bi\'m\b', text) or
        re.search(r'\b(lawyer|advocate|counsel|sister|brother|wife|husband|father|mother|son|daughter|client)\b', text)
    )
    if not has_filer:
        missing.append("your name or relationship to the affected person")

    has_person = bool(
        re.search(r'\b(brother|sister|son|daughter|husband|wife|father|mother|client|accused|detenu|applicant)\b', text) or
        re.search(r'\b(mr|mrs|ms|dr)\.?\s+\w+', text) or
        len(re.findall(r'[A-Z][a-z]+\s+[A-Z][a-z]+', orig)) >= 1
    )
    if not has_person:
        missing.append("name of detained or accused person")

    has_location = bool(
        re.search(r'police station', text) or
        re.search(r'\b(karachi|lahore|islamabad|peshawar|quetta|multan|faisalabad|rawalpindi|hyderabad|sukkur)\b', text) or
        re.search(r'\b(gulshan|clifton|defence|dha|saddar|korangi|malir|orangi|landhi|johar|nazimabad|pechs)\b', text) or
        re.search(r'\b(kbca|kmc|nha|wasa|court|authority|agency|tribunal)\b', text) or
        re.search(r'\b(sector|block|town|district|area|road|colony|phase)\b', text)
    )
    if not has_location:
        missing.append("location or police station name")

    is_complete = len(missing) == 0
    print(f"   → Complete: {is_complete} | Missing: {missing}")
    return {
        "missing_info": missing,
        "is_complete":  is_complete,
        "next_step":    "proceed" if is_complete else "ask_user"
    }

print("✅ Info check ready")

✅ Info check ready


In [76]:
# ── CELL 5: Jurisdiction & Court Formats ─────────────────────────────────────
COURT_FORMATS = {
    "Sindh HC": {
        "full_name": "IN THE HIGH COURT OF SINDH AT KARACHI",
        "city":      "Karachi",
        "ag":        "Advocate General, Sindh",
        "home_sec":  "Home Secretary, Government of Sindh",
        "short":     "Sindh High Court"
    },
    "Lahore HC": {
        "full_name": "IN THE HIGH COURT OF LAHORE AT LAHORE",
        "city":      "Lahore",
        "ag":        "Advocate General, Punjab",
        "home_sec":  "Home Secretary, Government of Punjab",
        "short":     "Lahore High Court"
    },
    "Islamabad HC": {
        "full_name": "IN THE HIGH COURT OF ISLAMABAD",
        "city":      "Islamabad",
        "ag":        "Advocate General, Islamabad",
        "home_sec":  "Home Secretary, ICT Administration",
        "short":     "Islamabad High Court"
    },
    "Sessions Court Karachi": {
        "full_name": "IN THE COURT OF SESSIONS JUDGE, KARACHI",
        "city":      "Karachi",
        "ag":        "N/A",
        "home_sec":  "N/A",
        "short":     "Sessions Court Karachi"
    },
    "Sessions Court Lahore": {
        "full_name": "IN THE COURT OF SESSIONS JUDGE, LAHORE",
        "city":      "Lahore",
        "ag":        "N/A",
        "home_sec":  "N/A",
        "short":     "Sessions Court Lahore"
    }
}

def select_jurisdiction(petition_type: str, user_story: str, user_id: str) -> str:
    text = user_story.lower()
    mem  = USER_MEMORY.get(user_id, {})

    if any(w in text for w in ["karachi", "sindh", "gulshan", "clifton", "defence", "dha karachi", "saddar"]):
        return "Sindh HC" if petition_type in ["Habeas Corpus", "Quashment", "Property/Encroachment"] else "Sessions Court Karachi"
    if any(w in text for w in ["lahore", "punjab", "dha lahore", "gulberg", "faisalabad", "multan"]):
        return "Lahore HC" if petition_type in ["Habeas Corpus", "Quashment", "Property/Encroachment"] else "Sessions Court Lahore"
    if any(w in text for w in ["islamabad", "ict", "rawalpindi", "pindi"]):
        return "Islamabad HC"
    if mem.get("preferred_court"):
        return mem["preferred_court"]
    return "Sessions Court Karachi" if petition_type in ["Pre-Arrest Bail", "Post-Arrest Bail"] else "Sindh HC"

print("✅ Jurisdiction selector ready")

✅ Jurisdiction selector ready


In [77]:
# ── CELL 6: Orchestrator Node ─────────────────────────────────────────────────
def orchestrator_node(state: LegalGenState) -> dict:
    print("🧠 [Orchestrator] Classifying case...")
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
You are a Pakistani legal classifier. Follow this STRICT decision tree:

SITUATION: "{state['user_story']}"

STEP 1: Property, shop, land, sealing, demolition, KBCA, KMC, encroachment mentioned?
  → YES = "Property/Encroachment"

STEP 2: FIR exists AND person is already in custody/arrested?
  → YES = "Post-Arrest Bail"

STEP 3: FIR exists but person NOT yet arrested?
  → YES = "Pre-Arrest Bail"

STEP 4: Person detained/arrested with NO FIR mentioned?
  → YES = "Habeas Corpus"

STEP 5: Want to cancel/quash an existing FIR itself?
  → YES = "Quashment"

Return ONLY JSON:
{{"petition_type": "<one of 5 above>", "reasoning": "<which step matched>"}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(resp.choices[0].message.content)
    p_type = result.get("petition_type", "Habeas Corpus")

    # Respect manual jurisdiction override
    jurisdiction = state.get("jurisdiction") or select_jurisdiction(p_type, state["user_story"], state.get("user_id", "default"))

    print(f"   → Type: {p_type} | Jurisdiction: {jurisdiction}")
    return {
        "petition_type": p_type,
        "jurisdiction":  jurisdiction,
        "target_court":  COURT_FORMATS[jurisdiction]["full_name"],
        "next_step":     "rag"
    }

print("✅ Orchestrator ready")

✅ Orchestrator ready


In [78]:
# ── CELL 7: RAG Primary + Secondary ──────────────────────────────────────────
def rag_primary_node(state: LegalGenState) -> dict:
    print(f"🔎 [RAG Primary] Attempt {state.get('rag_attempts', 0) + 1}...")
    attempts = state.get("rag_attempts", 0)
    p_type   = state.get("petition_type", "")

    if attempts == 0:
        query = state["user_story"]
    elif attempts == 1:
        query = f"{p_type} illegal arrest without warrant Pakistan Supreme Court"
    else:
        query = "fundamental rights Article 9 10A Constitution Pakistan detention"

    results = collection.query(query_texts=[query], n_results=3)

    if not results['documents'] or not results['documents'][0]:
        return {
            "primary_context":  "No precedent found.",
            "primary_citation": "Articles 9 & 10A Constitution of Pakistan 1973",
            "rag_attempts":     attempts + 1,
            "next_step":        "rag_secondary"
        }

    best_doc  = results['documents'][0][0]
    best_meta = results['metadatas'][0][0]
    citation  = best_meta.get('case_id', 'Unknown')

    # Relevance check
    rel_resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
Is this case relevant to the user's situation?
USER: {state['user_story'][:300]}
CASE SUMMARY: {best_doc[:400]}
PETITION TYPE: {p_type}
Return ONLY JSON: {{"is_relevant": true/false, "reason": "one line"}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    rel = json.loads(rel_resp.choices[0].message.content)

    if not rel.get("is_relevant") and attempts < 2:
        print(f"   ⚠️ Not relevant ({rel.get('reason')}) — retrying...")
        return {
            "primary_context":  "",
            "primary_citation": "",
            "rag_attempts":     attempts + 1,
            "next_step":        "rag_retry"
        }

    print(f"   → Primary: {citation}")
    return {
        "primary_context":  best_doc,
        "primary_citation": citation,
        "rag_attempts":     attempts + 1,
        "next_step":        "rag_secondary"
    }


def rag_secondary_node(state: LegalGenState) -> dict:
    print("🔎 [RAG Secondary] Finding supporting authority...")
    p_type = state.get("petition_type", "")
    secondary_queries = {
        "Habeas Corpus":          "Article 9 liberty illegal detention writ habeas corpus warrant",
        "Post-Arrest Bail":       "Section 497 CrPC bail grant factors surety investigation incomplete",
        "Pre-Arrest Bail":        "Section 498 CrPC pre-arrest bail anticipatory mala fide FIR",
        "Quashment":              "Article 199 quashment FIR mala fide abuse process court",
        "Property/Encroachment":  "Article 10A 23 natural justice notice hearing sealing property rights"
    }
    query   = secondary_queries.get(p_type, "fundamental rights Constitution Pakistan")
    results = collection.query(query_texts=[query], n_results=5)

    if not results['documents'] or not results['documents'][0]:
        return {
            "supporting_context":  "General constitutional principles apply.",
            "supporting_citation": "Constitution of Pakistan 1973"
        }

    primary_cit = state.get("primary_citation", "")
    for i in range(len(results['documents'][0])):
        cit = results['metadatas'][0][i].get('case_id', '')
        if cit != primary_cit:
            print(f"   → Supporting: {cit}")
            return {
                "supporting_context":  results['documents'][0][i],
                "supporting_citation": cit
            }

    # Fallback to first
    cit = results['metadatas'][0][0].get('case_id', 'Constitution of Pakistan')
    return {
        "supporting_context":  results['documents'][0][0],
        "supporting_citation": cit
    }

print("✅ RAG nodes ready")

✅ RAG nodes ready


In [79]:
# ── CELL 8: Extractor Node (FACTS + GROUNDS split) ───────────────────────────
def extract_fields_node(state: LegalGenState) -> dict:
    print("📝 [Extractor] Pulling structured fields...")
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
You are a Pakistani legal drafter. Extract and generate structured petition fields.

STORY: {state['user_story']}
PETITION TYPE: {state.get('petition_type', '')}
PRIMARY CITATION: {state.get('primary_citation', '')}
SUPPORTING CITATION: {state.get('supporting_citation', '')}

RULES:
- Extract ONLY names that appear in the STORY. Do not invent names.
- facts: 3-4 short factual paragraphs starting with "That". Pure facts only, no law.
- grounds: 6+ lettered grounds (A. Because...). Each must cite a law, article, or case.
  Reference PRIMARY CITATION explicitly in at least 2 grounds.
- prayer: specific numbered prayers matching the petition type.
- No repetition between facts and grounds.

Return ONLY valid JSON with these exact keys:
{{
  "petitioner_name": "...",
  "detenu_name": "...",
  "facts": ["That ...", "That ...", "That ..."],
  "grounds": ["A. Because ...", "B. Because ...", "C. Because ...", ...],
  "prayer": "..."
}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )

    result      = json.loads(resp.choices[0].message.content)
    facts_list  = result.get("facts", [])
    grounds_list = result.get("grounds", [])

    return {
        "petitioner":   result.get("petitioner_name", "The Petitioner"),
        "detenu":       result.get("detenu_name",     "The Affected Person"),
        "facts_text":   "\n\n".join(facts_list),
        "grounds_text": "\n\n".join(grounds_list),
        "prayer":       result.get("prayer", "Grant the relief prayed for.")
    }

print("✅ Extractor updated — FACTS and GROUNDS now separate")

✅ Extractor updated — FACTS and GROUNDS now separate


In [80]:
# ── CELL 9: Drafter Node — Court-Ready Structure ─────────────────────────────
def drafter_node(state: LegalGenState) -> dict:
    p_type       = state.get("petition_type", "Habeas Corpus")
    jurisdiction = state.get("jurisdiction", "Sindh HC")
    court_info   = COURT_FORMATS.get(jurisdiction, COURT_FORMATS["Sindh HC"])

    print(f"✍️ [Drafter] Writing {p_type} — {jurisdiction}...")

    petitioner     = state.get("petitioner")    or "The Petitioner"
    detenu         = state.get("detenu")         or "The Affected Person"
    facts_text     = state.get("facts_text")     or "That the detention is illegal and without lawful authority."
    grounds_text   = state.get("grounds_text")   or "A. Because the detention is in violation of Article 9 of the Constitution."
    prayer         = state.get("prayer")         or "Grant the relief prayed for."
    primary_cit    = state.get("primary_citation")    or "the Supreme Court"
    supporting_cit = state.get("supporting_citation") or "the Supreme Court"
    court_header   = court_info["full_name"]
    home_sec       = court_info["home_sec"]
    ag             = court_info["ag"]
    user_story     = state.get("user_story", "")

    # Detect location
    loc_match = re.search(r'([\w\-\s]+ police station)', user_story, re.IGNORECASE)
    location  = loc_match.group(0).strip() if loc_match else "the concerned Police Station"

    # Detect authority for property cases
    story_lower = user_story.lower()
    if "kbca" in story_lower:
        authority = "Karachi Building Control Authority (KBCA)"
    elif "kmc" in story_lower:
        authority = "Karachi Metropolitan Corporation (KMC)"
    else:
        authority = "The Concerned Authority"

    # ── Type-specific header block ────────────────────────────────────────────
    header_map = {
        "Habeas Corpus": f"""{court_header}

Writ Petition No. ___/2026 (Habeas Corpus)
Under Article 199(1)(b)(i) of the Constitution of Pakistan, 1973

{petitioner}                                       ...Petitioner
Versus
1. Station House Officer, {location}
2. {home_sec}
3. {ag}                                            ...Respondents

HABEAS CORPUS PETITION""",

        "Post-Arrest Bail": f"""{court_header}

Crl. Misc. Application No. ___/2026
Under Section 497, Code of Criminal Procedure, 1898

{detenu} (Applicant/Accused)                       ...Applicant
Versus
The State                                          ...Respondent

APPLICATION FOR POST-ARREST BAIL""",

        "Pre-Arrest Bail": f"""{court_header}

Crl. Misc. Application No. ___/2026
Under Section 498, Code of Criminal Procedure, 1898

{petitioner} (Applicant/Accused)                   ...Applicant
Versus
The State                                          ...Respondent

APPLICATION FOR PRE-ARREST BAIL""",

        "Quashment": f"""{court_header}

Writ Petition No. ___/2026
Under Article 199 of the Constitution of Pakistan, 1973

{petitioner}                                       ...Petitioner
Versus
1. Station House Officer, {location}
2. The State
3. The Complainant                                 ...Respondents

CONSTITUTIONAL PETITION (QUASHMENT OF FIR)""",

        "Property/Encroachment": f"""{court_header}

Writ Petition No. ___/2026
Under Articles 199, 23 & 10A of the Constitution of Pakistan, 1973

{petitioner}                                       ...Petitioner
Versus
1. {authority}
2. Province of Sindh / Federation of Pakistan      ...Respondents

CONSTITUTIONAL PETITION"""
    }

    header = header_map.get(p_type, header_map["Habeas Corpus"])

    # ── Extra constitutional grounds per type ─────────────────────────────────
    extra_map = {
        "Habeas Corpus": f"""G. Because Article 9 of the Constitution of Pakistan categorically provides that no person shall be deprived of life or liberty save in accordance with law. The detention in the present case is in patent violation thereof.

H. Because Article 10 of the Constitution and Section 61 Cr.P.C. require that every arrested person must be produced before a Magistrate within 24 hours. Neither requirement has been complied with in the present case.

I. Because this Honourable Court in the exercise of its constitutional jurisdiction under Article 199 is not only empowered but duty-bound to issue a writ of Habeas Corpus to protect the fundamental rights of the detenue. Reliance is placed on {supporting_cit}.""",

        "Post-Arrest Bail": f"""G. Because the Applicant has no previous criminal record and is not a flight risk, satisfying the twin-test criteria laid down by the Supreme Court for the grant of bail.

H. Because continued incarceration of the Applicant at this stage is punitive rather than preventive in nature and is violative of Article 10A of the Constitution which guarantees the right to a fair trial.

I. Because as held in {supporting_cit}, bail is the rule and jail the exception, particularly where investigation is incomplete and direct evidence against the Applicant is lacking.""",

        "Pre-Arrest Bail": f"""G. Because the FIR was registered with undue delay and apparent mala fide intent to coerce and harass the Applicant in a civil or personal dispute.

H. Because the arrest of the Applicant at this juncture would cause irreparable harm to his reputation, livelihood, and family with no corresponding benefit to the investigation.

I. Because as held by the Supreme Court in {supporting_cit}, pre-arrest bail ought to be granted where the applicant cooperates with the investigation and no direct incriminating evidence exists.""",

        "Quashment": f"""G. Because this Honourable Court exercises inherent and plenary jurisdiction under Article 199 of the Constitution to quash any FIR that discloses no cognizable offence or has been lodged as an abuse of the process of law.

H. Because permitting the impugned FIR to continue would cause irreparable harm, prejudice, and stigma to the Petitioner with no lawful justification.

I. Because in {supporting_cit} the Supreme Court has held that malafide FIRs must be quashed at the earliest stage to prevent abuse of the criminal justice system and to protect citizens from vexatious prosecution.""",

        "Property/Encroachment": f"""G. Because Article 23 of the Constitution of Pakistan, 1973, guarantees to every citizen the right to acquire, hold, and dispose of property in any part of Pakistan — a right being arbitrarily violated by the Respondents.

H. Because Article 10A of the Constitution guarantees the right to fair trial and due process, including the right to be heard before any adverse order is passed — a right wholly denied to the Petitioner.

I. Because in {supporting_cit}, the Supreme Court has held that regulatory authorities cannot act in excess of their statutory mandate, and any action taken in such excess is void ab initio and of no legal effect."""
    }

    extra = extra_map.get(p_type, "")

    # ── Verification block ────────────────────────────────────────────────────
    verification = f"""VERIFICATION

I, {petitioner}, do hereby solemnly affirm and declare that the contents of this petition are true and correct to the best of my knowledge and belief, that nothing material has been concealed, and that the petition is filed bona fide.

Sworn at {court_info['city']} on this _____ day of __________, 2026.

                                         _______________________
                                         {petitioner}
                                         Deponent

Through Counsel:
_______________________
Advocate, {court_info['short']}
Enrollment No.: _______
Cell: _________________"""

    # ── Master prompt ─────────────────────────────────────────────────────────
    prompt = f"""
You are a Senior Advocate of the {court_info['short']} with 20 years of experience.

Write a complete, court-ready petition using EXACTLY the structure below.
Do NOT add disclaimers. Do NOT write "sample" or "draft". Do NOT change names or citations.
Use sharp, formal Pakistani court language.

═══════════════════════════════════════════════════════
{header}

Most Respectfully Sheweth:

FACTS

{facts_text}

GROUNDS

{grounds_text}

{extra}

PRAYER

It is therefore most respectfully prayed that this Honourable Court may be pleased to:

{prayer}

{verification}
═══════════════════════════════════════════════════════

CRITICAL RULES:
1. Write the petition EXACTLY in the structure above — do not merge FACTS and GROUNDS.
2. Use ONLY the names: petitioner = "{petitioner}", detenu/accused = "{detenu}".
3. Reference "{primary_cit}" by name in at least 2 of the grounds.
4. Grounds must follow the lettered format: A. Because... B. Because... etc.
5. Keep facts factual and brief. Keep grounds legal and sharp.
6. The tone must be urgent and forceful — this is a writ court, not a session.
7. Do NOT repeat any argument. Each ground must make a distinct legal point.
8. Write the complete petition now:
"""

    chain = ChatPromptTemplate.from_template("{text}") | llm | StrOutputParser()
    draft = chain.invoke({"text": prompt})
    return {"final_petition": draft}

print("✅ Drafter rewritten — FACTS / GROUNDS / PRAYER structure enforced")

✅ Drafter rewritten — FACTS / GROUNDS / PRAYER structure enforced


In [81]:
# ── CELL 10: Validator Node ───────────────────────────────────────────────────
def validator_node(state: LegalGenState) -> dict:
    print("✅ [Validator] Checking petition quality...")

    p_type   = state.get("petition_type", "")
    citation = state.get("primary_citation", "")
    petition = state.get("final_petition", "")[:3000]
    story    = state.get("user_story", "")
    names_in_story = re.findall(r'[A-Z][a-z]+ [A-Z][a-z]+', story)

    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
You are a senior judge's clerk reviewing a Pakistani court petition.

PETITION TYPE: {p_type}
PRIMARY CITATION: {citation}
NAMES FROM STORY: {names_in_story}

PETITION TEXT:
{petition}

Score each dimension 0-10:
1. structure_score: Are FACTS, GROUNDS, and PRAYER clearly separated as distinct sections?
2. citation_score: Is "{citation}" named in GROUNDS and tied to a specific legal proposition (not just mentioned)?
3. grounds_count: How many lettered grounds exist (A., B., C., ...)? Return the COUNT as integer.
4. redundancy_score: Is each ground making a DISTINCT point with NO repetition? (10 = fully distinct)
5. tone_score: Is the language sharp, urgent, formal Pakistani court language? (10 = excellent)
6. no_hallucinated_names: Are all person names in petition found in NAMES FROM STORY or standard titles (SHO, Home Secretary, Advocate General)? true/false

is_valid: true if structure_score >= 7 AND citation_score >= 6 AND grounds_count >= 5
overall: weighted average (structure 30% + citation 25% + redundancy 25% + tone 20%)
issues: list ONLY real problems, empty array if none

Return ONLY valid JSON:
{{
  "structure_score": 0-10,
  "citation_score": 0-10,
  "grounds_count": integer,
  "redundancy_score": 0-10,
  "tone_score": 0-10,
  "no_hallucinated_names": true/false,
  "is_valid": true/false,
  "overall": 0.0-10.0,
  "issues": [],
  "notes": "one line"
}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )

    result         = json.loads(resp.choices[0].message.content)
    is_valid       = result.get("is_valid", False)
    revision_count = state.get("revision_count", 0)
    issues         = [i for i in result.get("issues", []) if i]

    print(f"   → Valid: {is_valid} | Overall: {result.get('overall', 0):.1f}/10")
    if issues:
        print(f"   → Issues: {issues}")

    if not is_valid and revision_count < 1:
        print("   ⚠️ Sending for revision...")
        return {
            "is_valid":            False,
            "validation_notes":    "\n".join(issues),
            "validation_score":    int(result.get("overall", 0) * 10),
            "eval_structure_score":  result.get("structure_score", 0),
            "eval_citation_score":   result.get("citation_score", 0),
            "eval_grounds_count":    result.get("grounds_count", 0),
            "eval_redundancy_score": result.get("redundancy_score", 0),
            "eval_tone_score":       result.get("tone_score", 0),
            "eval_overall_score":    result.get("overall", 0.0),
            "eval_issues":           issues,
            "eval_timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
            "revision_count":        revision_count + 1,
            "next_step":             "revise"
        }

    return {
        "is_valid":            True,
        "validation_notes":    result.get("notes", "Approved."),
        "validation_score":    int(result.get("overall", 0) * 10),
        "eval_structure_score":  result.get("structure_score", 0),
        "eval_citation_score":   result.get("citation_score", 0),
        "eval_grounds_count":    result.get("grounds_count", 0),
        "eval_redundancy_score": result.get("redundancy_score", 0),
        "eval_tone_score":       result.get("tone_score", 0),
        "eval_overall_score":    result.get("overall", 0.0),
        "eval_issues":           issues,
        "eval_timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
        "revision_count":        revision_count,
        "next_step":             "done"
    }

print("✅ Validator updated — numeric scores + eval fields")

✅ Validator updated — numeric scores + eval fields


In [82]:
# ── CELL 11: Revision Node ────────────────────────────────────────────────────
def revision_node(state: LegalGenState) -> dict:
    print("🔧 [Revision] Fixing validator issues...")
    issues = state.get("validation_notes", "No specific issues noted.")

    prompt = f"""
You are a Senior Pakistani Advocate. Revise this petition to fix the issues noted below.

ISSUES TO FIX:
{issues}

STRUCTURE REQUIREMENT — the petition MUST have these three clearly labelled sections:
FACTS      → factual paragraphs starting with "That", no legal argument
GROUNDS    → lettered grounds A. Because... B. Because... citing articles and cases
PRAYER     → numbered reliefs sought

ORIGINAL PETITION:
{state.get('final_petition', '')}

RULES:
- Fix ONLY the listed issues
- Keep all citations and article references
- Ensure FACTS / GROUNDS / PRAYER are clearly separated
- Do NOT add disclaimers
- Return the complete revised petition only
"""
    chain   = ChatPromptTemplate.from_template("{text}") | llm | StrOutputParser()
    revised = chain.invoke({"text": prompt})
    return {"final_petition": revised}

print("✅ Revision node ready")

✅ Revision node ready


In [83]:
# ── CELL 12: Graph Assembly ───────────────────────────────────────────────────
def route_after_info(state):      return state.get("next_step", "ask_user")
def route_after_rag(state):       return state.get("next_step", "rag_secondary")
def route_after_validator(state): return state.get("next_step", "done")

builder = StateGraph(LegalGenState)

builder.add_node("info_check",    check_missing_info_node)
builder.add_node("orchestrator",  orchestrator_node)
builder.add_node("rag_primary",   rag_primary_node)
builder.add_node("rag_secondary", rag_secondary_node)
builder.add_node("extractor",     extract_fields_node)
builder.add_node("drafter",       drafter_node)
builder.add_node("validator",     validator_node)
builder.add_node("revision",      revision_node)

builder.add_edge(START, "info_check")
builder.add_conditional_edges("info_check",    route_after_info,      {"ask_user": END, "proceed": "orchestrator"})
builder.add_edge("orchestrator", "rag_primary")
builder.add_conditional_edges("rag_primary",   route_after_rag,       {"rag_retry": "rag_primary", "rag_secondary": "rag_secondary"})
builder.add_edge("rag_secondary", "extractor")
builder.add_edge("extractor",     "drafter")
builder.add_edge("drafter",       "validator")
builder.add_conditional_edges("validator",      route_after_validator, {"revise": "revision", "done": END})
builder.add_edge("revision", END)

legal_gen_app = builder.compile()
print("✅ Full graph compiled")
print("   Flow: InfoCheck → Orchestrator → RAG(x2) → Extractor → Drafter → Validator → [Revision] → END")

✅ Full graph compiled
   Flow: InfoCheck → Orchestrator → RAG(x2) → Extractor → Drafter → Validator → [Revision] → END


In [84]:
# ── CELL 13: Runner ───────────────────────────────────────────────────────────
def run_legal_assistant(story: str, user_id: str = "default"):
    mem_ctx = get_memory_context(user_id)
    if USER_MEMORY.get(user_id, {}).get("past_petitions"):
        print(f"💾 Memory loaded — {len(USER_MEMORY[user_id]['past_petitions'])} past petition(s)")

    state = {
        "user_story":            story,
        "user_id":               user_id,
        "jurisdiction":          "",
        "is_complete":           False,
        "missing_info":          [],
        "petition_type":         "",
        "target_court":          "",
        "next_step":             "",
        "primary_context":       "",
        "primary_citation":      "",
        "supporting_context":    "",
        "supporting_citation":   "",
        "rag_attempts":          0,
        "petitioner":            "",
        "detenu":                "",
        "facts_text":            "",
        "grounds_text":          "",
        "prayer":                "",
        "is_valid":              False,
        "validation_notes":      "",
        "validation_score":      0,
        "revision_count":        0,
        "final_petition":        "",
        "memory_context":        mem_ctx,
        "eval_structure_score":  0,
        "eval_citation_score":   0,
        "eval_grounds_count":    0,
        "eval_redundancy_score": 0,
        "eval_tone_score":       0,
        "eval_overall_score":    0.0,
        "eval_issues":           [],
        "eval_timestamp":        ""
    }

    # Step 1: Info check
    info_result = check_missing_info_node(state)
    state.update(info_result)

    if not state["is_complete"]:
        print(f"\n📋 Missing: {', '.join(state['missing_info'])}")
        reply = input("👤 Please provide these details: ")
        state["user_story"] += f"\nAdditional details: {reply}"
        state["is_complete"] = True
        state["next_step"]   = "proceed"

    # Step 2: Optional jurisdiction override
    print("\n🏛️  Jurisdiction (Enter to auto-detect, or type one):")
    print("   Options: Sindh HC | Lahore HC | Islamabad HC | Sessions Court Karachi | Sessions Court Lahore")
    jur_input = input("   → ").strip()
    if jur_input in COURT_FORMATS:
        state["jurisdiction"] = jur_input
        print(f"   Using: {jur_input}")
    else:
        print("   Auto-detecting from story...")

    # Step 3: Run pipeline (skip info_check)
    builder2 = StateGraph(LegalGenState)
    for name, fn in [
        ("orchestrator",  orchestrator_node),
        ("rag_primary",   rag_primary_node),
        ("rag_secondary", rag_secondary_node),
        ("extractor",     extract_fields_node),
        ("drafter",       drafter_node),
        ("validator",     validator_node),
        ("revision",      revision_node),
    ]:
        builder2.add_node(name, fn)

    builder2.add_edge(START, "orchestrator")
    builder2.add_edge("orchestrator", "rag_primary")
    builder2.add_conditional_edges("rag_primary",  route_after_rag,       {"rag_retry": "rag_primary", "rag_secondary": "rag_secondary"})
    builder2.add_edge("rag_secondary", "extractor")
    builder2.add_edge("extractor",     "drafter")
    builder2.add_edge("drafter",       "validator")
    builder2.add_conditional_edges("validator",     route_after_validator, {"revise": "revision", "done": END})
    builder2.add_edge("revision", END)

    pipeline = builder2.compile()
    output   = pipeline.invoke(state)
    state.update(output)

    # Step 4: Save memory
    save_to_memory(user_id, state)

    # Step 5: Print result
    if state.get("final_petition"):
        sep = "=" * 65
        print(f"\n{sep}")
        print(f"⚖️  {state.get('petition_type','').upper()}")
        print(f"🏛️  {state.get('jurisdiction','')}")
        print(f"📌 Primary Citation:    {state.get('primary_citation','N/A')}")
        print(f"📌 Supporting Citation: {state.get('supporting_citation','N/A')}")
        print(f"✅ Valid: {state.get('is_valid')} | Score: {state.get('eval_overall_score',0):.1f}/10")
        print(f"📊 Structure:{state['eval_structure_score']} | Citation:{state['eval_citation_score']} | Redundancy:{state['eval_redundancy_score']} | Tone:{state['eval_tone_score']} | Grounds:{state['eval_grounds_count']}")
        print(sep)
        print(state["final_petition"])

        fname = f"/kaggle/working/{user_id}_{state.get('petition_type','petition').replace(' ','_')}.txt"
        with open(fname, "w") as f:
            f.write(state["final_petition"])
        print(f"\n💾 Saved to: {fname}")

    else:
        print("⚠️ No petition generated.")

    return state   # ← return for metrics collection

print("✅ Runner ready")

✅ Runner ready


In [85]:
# ── CELL 14: Evaluation Metrics Dashboard ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_evaluation_dashboard(results: list[dict], title: str = "Legal Petition Quality Dashboard"):
    """
    results: list of state dicts returned by run_legal_assistant()
    """
    if not results:
        print("No results to plot.")
        return

    cases      = [r.get("user_id", f"Case {i+1}") for i, r in enumerate(results)]
    structure  = [r.get("eval_structure_score", 0)  for r in results]
    citation   = [r.get("eval_citation_score", 0)   for r in results]
    redundancy = [r.get("eval_redundancy_score", 0) for r in results]
    tone       = [r.get("eval_tone_score", 0)       for r in results]
    overall    = [r.get("eval_overall_score", 0.0)  for r in results]
    grounds    = [r.get("eval_grounds_count", 0)    for r in results]
    p_types    = [r.get("petition_type", "Unknown") for r in results]

    n     = len(results)
    x     = np.arange(n)
    width = 0.18

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.98)
    fig.patch.set_facecolor("#0f1117")
    for ax in axes.flat:
        ax.set_facecolor("#1a1d27")
        ax.spines[["top","right","left","bottom"]].set_color("#2d3148")
        ax.tick_params(colors="#c8cce8")
        ax.title.set_color("#e8eaf6")
        ax.xaxis.label.set_color("#a0a4c8")
        ax.yaxis.label.set_color("#a0a4c8")

    colors = {"Structure": "#7c83f0", "Citation": "#4dd0e1",
              "Redundancy": "#81c784", "Tone": "#ffb74d"}

    # ── Chart 1: Grouped bar — all 4 dimensions ───────────────────────────────
    ax1 = axes[0, 0]
    ax1.bar(x - 1.5*width, structure,  width, label="Structure",  color=colors["Structure"],  alpha=0.88)
    ax1.bar(x - 0.5*width, citation,   width, label="Citation",   color=colors["Citation"],   alpha=0.88)
    ax1.bar(x + 0.5*width, redundancy, width, label="Redundancy", color=colors["Redundancy"], alpha=0.88)
    ax1.bar(x + 1.5*width, tone,       width, label="Tone",       color=colors["Tone"],       alpha=0.88)
    ax1.set_xticks(x)
    ax1.set_xticklabels(cases, rotation=15, ha="right", fontsize=9)
    ax1.set_ylim(0, 11)
    ax1.set_ylabel("Score (0–10)")
    ax1.set_title("Quality Dimensions by Case")
    ax1.legend(fontsize=8, facecolor="#252836", labelcolor="#c8cce8",
               framealpha=0.7, loc="lower right")
    ax1.axhline(y=7, color="#ff6b6b", linewidth=1, linestyle="--", alpha=0.6, label="Min threshold (7)")
    for bars, vals in [(ax1.containers[0], structure), (ax1.containers[1], citation),
                       (ax1.containers[2], redundancy), (ax1.containers[3], tone)]:
        for bar, v in zip(bars, vals):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                     str(v), ha="center", va="bottom", fontsize=7, color="#c8cce8")

    # ── Chart 2: Overall score line ───────────────────────────────────────────
    ax2 = axes[0, 1]
    ax2.plot(cases, overall, marker="o", color="#7c83f0", linewidth=2.2,
             markersize=9, markerfacecolor="#fff", markeredgecolor="#7c83f0", markeredgewidth=2)
    ax2.fill_between(range(n), overall, alpha=0.15, color="#7c83f0")
    ax2.axhline(y=7, color="#ff6b6b", linewidth=1, linestyle="--", alpha=0.7)
    ax2.set_ylim(0, 10.5)
    ax2.set_xticks(range(n))
    ax2.set_xticklabels(cases, rotation=15, ha="right", fontsize=9)
    ax2.set_ylabel("Overall Score (0–10)")
    ax2.set_title("Overall Score per Case")
    for i, (c, v) in enumerate(zip(cases, overall)):
        ax2.annotate(f"{v:.1f}", (i, v), textcoords="offset points",
                     xytext=(0, 10), ha="center", fontsize=9, color="#e8eaf6")

    # ── Chart 3: Grounds count bar ────────────────────────────────────────────
    ax3 = axes[1, 0]
    bar_colors = ["#81c784" if g >= 5 else "#ffb74d" if g >= 3 else "#ef5350" for g in grounds]
    bars3 = ax3.bar(cases, grounds, color=bar_colors, alpha=0.85, width=0.55)
    ax3.axhline(y=5, color="#ff6b6b", linewidth=1.2, linestyle="--", alpha=0.7)
    ax3.set_ylabel("Number of Lettered Grounds")
    ax3.set_title("Grounds Count (min 5 required)")
    ax3.set_ylim(0, max(grounds + [6]) + 2)
    ax3.set_xticklabels(cases, rotation=15, ha="right", fontsize=9)
    for bar, v in zip(bars3, grounds):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(v), ha="center", va="bottom", fontsize=10, color="#e8eaf6", fontweight="bold")
    patches = [mpatches.Patch(color="#81c784", label="≥5 grounds (pass)"),
               mpatches.Patch(color="#ffb74d", label="3–4 grounds (warn)"),
               mpatches.Patch(color="#ef5350", label="<3 grounds (fail)")]
    ax3.legend(handles=patches, fontsize=8, facecolor="#252836", labelcolor="#c8cce8", framealpha=0.7)

    # ── Chart 4: Radar chart — avg scores across all cases ────────────────────
    ax4 = axes[1, 1]
    ax4.remove()
    ax4 = fig.add_subplot(2, 2, 4, polar=True)
    ax4.set_facecolor("#1a1d27")

    categories  = ["Structure", "Citation", "Redundancy", "Tone"]
    avg_scores  = [
        np.mean(structure),
        np.mean(citation),
        np.mean(redundancy),
        np.mean(tone)
    ]
    N      = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    vals   = avg_scores + avg_scores[:1]

    ax4.set_theta_offset(np.pi / 2)
    ax4.set_theta_direction(-1)
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories, size=10, color="#c8cce8")
    ax4.set_ylim(0, 10)
    ax4.set_yticks([2, 4, 6, 8, 10])
    ax4.set_yticklabels(["2","4","6","8","10"], size=7, color="#6b7090")
    ax4.plot(angles, vals, "o-", linewidth=2, color="#7c83f0")
    ax4.fill(angles, vals, alpha=0.25, color="#7c83f0")
    ax4.set_title("Average Quality Radar\n(all cases)", color="#e8eaf6", pad=18, fontsize=11)
    ax4.tick_params(colors="#555878")
    ax4.spines["polar"].set_color("#2d3148")
    ax4.grid(color="#2d3148", linewidth=0.8)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig("/kaggle/working/eval_dashboard.png", dpi=150, bbox_inches="tight",
                facecolor="#0f1117")
    plt.show()
    print("📊 Dashboard saved to /kaggle/working/eval_dashboard.png")


def print_eval_summary(results: list[dict]):
    """Print a clean text summary table of all evaluations."""
    print("\n" + "="*80)
    print("📊  EVALUATION SUMMARY")
    print("="*80)
    print(f"{'Case':<20} {'Type':<22} {'Struct':>6} {'Cite':>5} {'Redun':>6} {'Tone':>5} {'Grnds':>6} {'OVERALL':>8}")
    print("-"*80)
    for r in results:
        print(
            f"{r.get('user_id',''):<20} "
            f"{r.get('petition_type',''):<22} "
            f"{r.get('eval_structure_score',0):>6} "
            f"{r.get('eval_citation_score',0):>5} "
            f"{r.get('eval_redundancy_score',0):>6} "
            f"{r.get('eval_tone_score',0):>5} "
            f"{r.get('eval_grounds_count',0):>6} "
            f"{r.get('eval_overall_score',0.0):>7.1f}/10"
        )
    if results:
        avg_overall = np.mean([r.get("eval_overall_score", 0) for r in results])
        print("-"*80)
        print(f"{'AVERAGE':<49} {avg_overall:>40.1f}/10")
    print("="*80 + "\n")

print("✅ Evaluation metrics & dashboard ready")

✅ Evaluation metrics & dashboard ready


In [ ]:
# ── CELL 15: Run All Test Cases + Evaluation Dashboard ───────────────────────
test_cases = [
    ("rafsha_01", "My brother Ali Khan was arrested without warrant from our home in Gulshan-e-Iqbal by plain clothes officers on March 24 2026. My name is Rafsha Rahim. He is at Gulshan Police Station."),
    ("ahmed_01",  "An FIR has been registered against me at Clifton Police Station under Section 324 PPC. I have not been arrested yet. My name is Ahmed Raza. FIR date March 28 2026."),
    ("sara_01",   "My client Bilal Sheikh was arrested on March 20 2026 and has been in custody for 12 days at Defence Police Station under Section 409 PPC. I am his lawyer Sara Khan."),
]

all_results = []

for user_id, story in test_cases:
    print(f"\n{'='*65}")
    print(f"🧪 {user_id.upper()}")
    print(f"{'='*65}")
    result_state = run_legal_assistant(story, user_id=user_id)
    if result_state:
        all_results.append(result_state)
    print()

# Print text summary
print_eval_summary(all_results)

# Plot visual dashboard
plot_evaluation_dashboard(all_results, title="Legal Petition Quality Dashboard — All Test Cases")


🧪 RAFSHA_01
💾 Memory loaded — 1 past petition(s)
📋 [InfoCheck] Scanning for missing details...
   → Complete: True | Missing: []

🏛️  Jurisdiction (Enter to auto-detect, or type one):
   Options: Sindh HC | Lahore HC | Islamabad HC | Sessions Court Karachi | Sessions Court Lahore


   →  


   Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Habeas Corpus | Jurisdiction: Sindh HC
🔎 [RAG Primary] Attempt 1...
   ⚠️ Not relevant (The case summary is unrelated to the user's brother's arrest in Karachi, as it pertains to a habeas corpus petition in the High Court of Balochistan, Quetta.) — retrying...
🔎 [RAG Primary] Attempt 2...
   ⚠️ Not relevant (Different location, date, and circumstances) — retrying...
🔎 [RAG Primary] Attempt 3...
   → Primary: C.P.L.A.3637_2019
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: C.P.L.A.1809_2020
📝 [Extractor] Pulling structured fields...
✍️ [Drafter] Writing Habeas Corpus — Sindh HC...
✅ [Validator] Checking petition quality...
   → Valid: True | Overall: 8.4/10
💾 Memory saved for user: rafsha_01

⚖️  HABEAS CORPUS
🏛️  Sindh HC
📌 Primary Citation:    C.P.L.A.3637_2019
📌 Supporting Citation: C.P.L.A.1809_2020
✅ Valid: True | Score: 8.4/10
📊 Structure:9 | Citation:8 | Redundancy:9 | Tone:9 | Gro

   →  


   Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Pre-Arrest Bail | Jurisdiction: Sessions Court Karachi
🔎 [RAG Primary] Attempt 1...
   ⚠️ Not relevant (Different FIR number, date, and police station, also different sections of PPC) — retrying...
🔎 [RAG Primary] Attempt 2...
   → Primary: Crl.P.L.A.645-L_2025
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: Crl.P.L.A.1075-L_2020
📝 [Extractor] Pulling structured fields...
✍️ [Drafter] Writing Pre-Arrest Bail — Sessions Court Karachi...
✅ [Validator] Checking petition quality...
   → Valid: True | Overall: 8.4/10
💾 Memory saved for user: ahmed_01

⚖️  PRE-ARREST BAIL
🏛️  Sessions Court Karachi
📌 Primary Citation:    Crl.P.L.A.645-L_2025
📌 Supporting Citation: Crl.P.L.A.1075-L_2020
✅ Valid: True | Score: 8.4/10
📊 Structure:8 | Citation:8 | Redundancy:9 | Tone:9 | Grounds:8
IN THE COURT OF SESSIONS JUDGE, KARACHI

Crl. Misc. Application No. 1234/2026
Under Section 498, Code of Criminal Proce

# FINAL VERSION

In [5]:
# ── CELL 1: Imports & LLM Setup ──────────────────────────────────────────────
import os, re, json, time
from groq import Groq
from typing import TypedDict, List, Optional
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_KEY_HERE")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ Imports & LLM ready")

✅ Imports & LLM ready


In [6]:
# ── CELL 2: In-Session Memory ────────────────────────────────────────────────
USER_MEMORY = {}

def save_to_memory(user_id: str, state: dict):
    if user_id not in USER_MEMORY:
        USER_MEMORY[user_id] = {
            "past_petitions": [],
            "preferred_court": None,
            "citations_used":  [],
            "petition_types":  []
        }
    mem = USER_MEMORY[user_id]
    mem["past_petitions"].append({
        "story":       state.get("user_story", "")[:200],
        "type":        state.get("petition_type", ""),
        "court":       state.get("jurisdiction", ""),
        "citation":    state.get("primary_citation", ""),
        "draft_style": state.get("final_petition", "")[:300]
    })
    if state.get("jurisdiction"):
        mem["preferred_court"] = state["jurisdiction"]
    if state.get("primary_citation"):
        mem["citations_used"].append(state["primary_citation"])
    if state.get("petition_type"):
        mem["petition_types"].append(state["petition_type"])
    print(f"💾 Memory saved for user: {user_id}")

def get_memory_context(user_id: str) -> str:
    if user_id not in USER_MEMORY or not USER_MEMORY[user_id]["past_petitions"]:
        return "No previous petitions on record."
    mem = USER_MEMORY[user_id]
    return (
        f"User has filed {len(mem['past_petitions'])} previous petition(s).\n"
        f"Preferred court: {mem['preferred_court'] or 'Not set'}\n"
        f"Most used type: {max(set(mem['petition_types']), key=mem['petition_types'].count) if mem['petition_types'] else 'None'}\n"
        f"Recent citations: {', '.join(mem['citations_used'][-3:]) or 'None'}\n"
        f"Last draft sample: {mem['past_petitions'][-1]['draft_style'] if mem['past_petitions'] else 'None'}"
    )

print("✅ Memory system ready")

✅ Memory system ready


In [7]:
# ── CELL 3: LangGraph State ───────────────────────────────────────────────────
class LegalGenState(TypedDict):
    user_story:          str
    user_id:             str
    jurisdiction:        str
    is_complete:         bool
    missing_info:        List[str]
    petition_type:       str
    target_court:        str
    next_step:           str
    primary_context:     str
    primary_citation:    str
    supporting_context:  str
    supporting_citation: str
    rag_attempts:        int
    petitioner:          str
    detenu:              str
    facts_text:          str
    grounds_text:        str
    prayer:              str
    is_valid:            bool
    validation_notes:    str
    validation_score:    int
    revision_count:      int
    final_petition:      str
    memory_context:      str
    # ── NEW: pre-draft enrichment ────────────────────────────────────────────
    red_flags:           List[str]   # e.g. ["Section 324 is serious non-bailable"]
    legal_strategy:      str         # strategic memo fed to drafter
    citation_tier:       str         # "PLD" | "SCMR" | "YLR" | "CLC" | "CPLA"
    interim_relief:      str         # auto-injected urgent prayer
    # ── Evaluation metrics ───────────────────────────────────────────────────
    eval_structure_score:   int
    eval_citation_score:    int
    eval_grounds_count:     int
    eval_redundancy_score:  int
    eval_tone_score:        int
    eval_overall_score:     float
    eval_issues:            List[str]
    eval_timestamp:         str

print("✅ State defined")

✅ State defined


In [8]:
# ── CELL 4: Info Check Node ──────────────────────────────────────────────────
def check_missing_info_node(state: LegalGenState) -> dict:
    print("📋 [InfoCheck] Scanning for missing details...")
    text = state["user_story"].lower()
    orig = state["user_story"]
    missing = []

    has_filer = bool(
        re.search(r'my name is', text) or
        re.search(r'\bi am\b|\bi\'m\b', text) or
        re.search(r'\b(lawyer|advocate|counsel|sister|brother|wife|husband|father|mother|son|daughter|client)\b', text)
    )
    if not has_filer:
        missing.append("your name or relationship to the affected person")

    has_person = bool(
        re.search(r'\b(brother|sister|son|daughter|husband|wife|father|mother|client|accused|detenu|applicant)\b', text) or
        re.search(r'\b(mr|mrs|ms|dr)\.?\s+\w+', text) or
        len(re.findall(r'[A-Z][a-z]+\s+[A-Z][a-z]+', orig)) >= 1
    )
    if not has_person:
        missing.append("name of detained or accused person")

    has_location = bool(
        re.search(r'police station', text) or
        re.search(r'\b(karachi|lahore|islamabad|peshawar|quetta|multan|faisalabad|rawalpindi|hyderabad|sukkur)\b', text) or
        re.search(r'\b(gulshan|clifton|defence|dha|saddar|korangi|malir|orangi|landhi|johar|nazimabad|pechs)\b', text) or
        re.search(r'\b(kbca|kmc|nha|wasa|court|authority|agency|tribunal)\b', text) or
        re.search(r'\b(sector|block|town|district|area|road|colony|phase)\b', text)
    )
    if not has_location:
        missing.append("location or police station name")

    is_complete = len(missing) == 0
    print(f"   → Complete: {is_complete} | Missing: {missing}")
    return {
        "missing_info": missing,
        "is_complete":  is_complete,
        "next_step":    "proceed" if is_complete else "ask_user"
    }

print("✅ Info check ready")

✅ Info check ready


In [9]:
# ── CELL 5: Jurisdiction & Court Formats ─────────────────────────────────────
COURT_FORMATS = {
    "Sindh HC": {
        "full_name": "IN THE HIGH COURT OF SINDH AT KARACHI",
        "city":      "Karachi",
        "ag":        "Advocate General, Sindh",
        "home_sec":  "Home Secretary, Government of Sindh",
        "short":     "Sindh High Court"
    },
    "Lahore HC": {
        "full_name": "IN THE HIGH COURT OF LAHORE AT LAHORE",
        "city":      "Lahore",
        "ag":        "Advocate General, Punjab",
        "home_sec":  "Home Secretary, Government of Punjab",
        "short":     "Lahore High Court"
    },
    "Islamabad HC": {
        "full_name": "IN THE HIGH COURT OF ISLAMABAD",
        "city":      "Islamabad",
        "ag":        "Advocate General, Islamabad",
        "home_sec":  "Home Secretary, ICT Administration",
        "short":     "Islamabad High Court"
    },
    "Sessions Court Karachi": {
        "full_name": "IN THE COURT OF SESSIONS JUDGE, KARACHI",
        "city":      "Karachi",
        "ag":        "N/A",
        "home_sec":  "N/A",
        "short":     "Sessions Court Karachi"
    },
    "Sessions Court Lahore": {
        "full_name": "IN THE COURT OF SESSIONS JUDGE, LAHORE",
        "city":      "Lahore",
        "ag":        "N/A",
        "home_sec":  "N/A",
        "short":     "Sessions Court Lahore"
    }
}

def select_jurisdiction(petition_type: str, user_story: str, user_id: str) -> str:
    text = user_story.lower()
    mem  = USER_MEMORY.get(user_id, {})

    if any(w in text for w in ["karachi", "sindh", "gulshan", "clifton", "defence", "dha karachi", "saddar"]):
        return "Sindh HC" if petition_type in ["Habeas Corpus", "Quashment", "Property/Encroachment"] else "Sessions Court Karachi"
    if any(w in text for w in ["lahore", "punjab", "dha lahore", "gulberg", "faisalabad", "multan"]):
        return "Lahore HC" if petition_type in ["Habeas Corpus", "Quashment", "Property/Encroachment"] else "Sessions Court Lahore"
    if any(w in text for w in ["islamabad", "ict", "rawalpindi", "pindi"]):
        return "Islamabad HC"
    if mem.get("preferred_court"):
        return mem["preferred_court"]
    return "Sessions Court Karachi" if petition_type in ["Pre-Arrest Bail", "Post-Arrest Bail"] else "Sindh HC"

print("✅ Jurisdiction selector ready")

✅ Jurisdiction selector ready


In [10]:
# ── CELL 6: Orchestrator Node ─────────────────────────────────────────────────
def orchestrator_node(state: LegalGenState) -> dict:
    print("🧠 [Orchestrator] Classifying case...")
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
You are a Pakistani legal classifier. Follow this STRICT decision tree:

SITUATION: "{state['user_story']}"

STEP 1: Property, shop, land, sealing, demolition, KBCA, KMC, encroachment mentioned?
  → YES = "Property/Encroachment"

STEP 2: FIR exists AND person is already in custody/arrested?
  → YES = "Post-Arrest Bail"

STEP 3: FIR exists but person NOT yet arrested?
  → YES = "Pre-Arrest Bail"

STEP 4: Person detained/arrested with NO FIR mentioned?
  → YES = "Habeas Corpus"

STEP 5: Want to cancel/quash an existing FIR itself?
  → YES = "Quashment"

Return ONLY JSON:
{{"petition_type": "<one of 5 above>", "reasoning": "<which step matched>"}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(resp.choices[0].message.content)
    p_type = result.get("petition_type", "Habeas Corpus")

    # Respect manual jurisdiction override
    jurisdiction = state.get("jurisdiction") or select_jurisdiction(p_type, state["user_story"], state.get("user_id", "default"))

    print(f"   → Type: {p_type} | Jurisdiction: {jurisdiction}")
    return {
        "petition_type": p_type,
        "jurisdiction":  jurisdiction,
        "target_court":  COURT_FORMATS[jurisdiction]["full_name"],
        "next_step":     "rag"
    }

print("✅ Orchestrator ready")

✅ Orchestrator ready


In [11]:
# ── CELL 6B: Red Flag Detector ────────────────────────────────────────────────
# Catches dangerous legal errors BEFORE drafting (e.g. "324 is not serious")

SECTION_SEVERITY = {
    # PPC sections — seriousness tier
    "302": ("non-bailable", "murder", "HIGH"),
    "324": ("non-bailable", "attempt to murder", "HIGH"),
    "392": ("non-bailable", "robbery", "HIGH"),
    "395": ("non-bailable", "dacoity", "HIGH"),
    "409": ("non-bailable", "criminal breach of trust", "HIGH"),
    "411": ("non-bailable", "receiving stolen property", "MEDIUM"),
    "420": ("bailable", "cheating/fraud", "MEDIUM"),
    "489-F": ("non-bailable", "dishonoured cheque", "MEDIUM"),
    "506": ("bailable", "criminal intimidation", "LOW"),
    "447": ("bailable", "criminal trespass", "LOW"),
    "337": ("bailable", "hurt", "LOW"),
    "379": ("non-bailable", "theft", "MEDIUM"),
    "376": ("non-bailable", "rape", "HIGH"),
    "354": ("non-bailable", "assault on woman", "HIGH"),
    "7-ATA": ("non-bailable", "Anti-Terrorism Act offence", "HIGH"),
    "9-CNS": ("non-bailable", "Control of Narcotic Substances Act", "HIGH"),
}

BAIL_STRATEGY_MAP = {
    "HIGH": {
        "approach": "exceptional circumstances + further inquiry",
        "key_points": [
            "no direct incriminating evidence",
            "further inquiry required under Section 497(2) CrPC",
            "applicant poses no flight risk + has deep community ties",
            "complainant's version uncorroborated",
            "section is not a bar to bail — bail is the rule (AIR 1977 SC)",
        ],
        "avoid": "never argue the offence is 'not serious' — concede seriousness, argue circumstances",
    },
    "MEDIUM": {
        "approach": "mala fide + twin test",
        "key_points": [
            "no previous criminal record",
            "not a flight risk",
            "FIR registered with delay — suggests mala fide",
            "dispute is civil/commercial in nature",
        ],
        "avoid": "do not overclaim — focus on twin test criteria",
    },
    "LOW": {
        "approach": "straightforward bail — bailable offence",
        "key_points": [
            "offence is bailable as of right",
            "bail cannot be withheld for bailable offence",
            "Section 496 CrPC: bail is mandatory",
        ],
        "avoid": "none",
    },
}

def detect_red_flags(petition_type: str, user_story: str) -> tuple[list, str, str]:
    """
    Returns: (red_flags_list, severity, bail_strategy_memo)
    """
    flags = []
    severity = "LOW"
    strategy_notes = ""
    text = user_story.upper()

    detected_sections = []
    for sec, (bail_type, description, sev) in SECTION_SEVERITY.items():
        pattern = rf'\b{re.escape(sec)}\b'
        if re.search(pattern, text):
            detected_sections.append((sec, bail_type, description, sev))
            flags.append(
                f"Section {sec} PPC ({description}) is {bail_type.upper()} "
                f"— severity: {sev}. Do NOT argue it is minor."
            )
            if sev == "HIGH":
                severity = "HIGH"
            elif sev == "MEDIUM" and severity != "HIGH":
                severity = "MEDIUM"

    # Pre-arrest bail with HIGH section → mandatory flag
    if petition_type == "Pre-Arrest Bail" and severity == "HIGH":
        flags.append(
            "PRE-ARREST BAIL with HIGH-severity section: "
            "must argue extraordinary circumstances + mala fide intent of complainant. "
            "Mention 'concession not right' doctrine."
        )

    # Post-arrest bail non-bailable → further inquiry
    if petition_type == "Post-Arrest Bail" and severity == "HIGH":
        flags.append(
            "POST-ARREST BAIL (non-bailable): invoke Section 497(2) CrPC "
            "'further inquiry' — not mere Section 497(1) twin test."
        )

    # No FIR number but FIR mentioned
    if "FIR" in text and not re.search(r'FIR\s*(NO\.?|NUMBER)?\s*\d+', text):
        flags.append("FIR mentioned but no FIR number extracted — drafter will use placeholder.")

    strategy_notes = BAIL_STRATEGY_MAP.get(severity, BAIL_STRATEGY_MAP["LOW"])

    return flags, severity, strategy_notes


def red_flag_node(state: LegalGenState) -> dict:
    print("🚨 [RedFlag] Scanning for legal landmines...")
    p_type = state.get("petition_type", "")
    story  = state.get("user_story", "")

    flags, severity, strategy = detect_red_flags(p_type, story)

    if flags:
        print(f"   ⚠️  {len(flags)} flag(s) detected — severity: {severity}")
        for f in flags:
            print(f"      → {f}")
    else:
        print("   ✅ No red flags detected")

    return {
        "red_flags":    flags,
        "legal_strategy": json.dumps(strategy, ensure_ascii=False),
    }

print("✅ Red flag detector ready")

✅ Red flag detector ready


In [12]:
# ── CELL 6C: Strategy Engine ──────────────────────────────────────────────────
# Generates a legal strategy memo BEFORE drafting

STRATEGY_TEMPLATES = {
    "Habeas Corpus": """
STRATEGY: Habeas Corpus

CORE THEORY:
- Detention is illegal, mala fide, and without lawful authority
- No FIR / no warrant / no Magistrate production within 24 hours
- Article 9 (liberty) + Article 10 (arrest procedure) violated

MUST INCLUDE IN GROUNDS:
- Failure to produce before Magistrate within 24h (Article 10 + Section 61 CrPC)
- Plain-clothes/unidentified officers → unlawful arrest
- No detention order under any statute produced
- Call for production through SSP/IG chain of command

PRAYER MUST INCLUDE:
- "produce the detenu forthwith before this Honourable Court"
- "respondents submit para-wise comments"
- "intimate rank and identity of arresting officers"
- "interim order: produce detenu by [date]"

TONE:
- Urgent and alarming — "mala fide", "without lawful authority", "patent violation"
- Frame as constitutional emergency, not a routine application
""",

    "Post-Arrest Bail": """
STRATEGY: Post-Arrest Bail (Section 497 CrPC)

DETERMINE SEVERITY FIRST (injected via red_flags):
- HIGH severity: lead with Section 497(2) "further inquiry" route
- MEDIUM/LOW: lead with twin test (not flight risk, no previous record)

CORE THEORY:
- Bail is the rule, jail is the exception (Supreme Court consistent position)
- Continued detention is punitive, not preventive — violates Article 10A
- Investigation is incomplete — bail will not hamper it

MUST INCLUDE IN GROUNDS:
- "further inquiry within the meaning of Section 497(2) CrPC is required"
  (for HIGH-severity sections — this is the PRIMARY route)
- No direct incriminating evidence on record
- Applicant poses no flight risk — deep community ties, family, employment
- No previous criminal record
- Article 10A fair trial right — applicant needs access to counsel, documents
- Inordinate delay in investigation / arrest

PRAYER MUST INCLUDE:
- "admit the applicant to bail in the sum of Rs. ___ with surety of like amount"
- "interim bail during pendency of this application"
- "respondents submit challan / investigation report"

AVOID:
- Never say the offence is "not serious" if it is HIGH severity
- Do not conflate Section 497(1) and 497(2) tests
""",

    "Pre-Arrest Bail": """
STRATEGY: Pre-Arrest Bail (Section 498 CrPC)

KEY DOCTRINE:
- Pre-arrest bail is an EXTRAORDINARY remedy — concession not right
- Must show MALA FIDE / ULTERIOR MOTIVE behind FIR
- Must show arrest would cause irreparable harm

CORE THEORY:
- FIR is result of personal/civil dispute dressed as criminal complaint
- Registered with undue delay (showing afterthought and mala fide)
- Applicant is willing to cooperate with investigation — no flight risk

MUST INCLUDE IN GROUNDS:
- Delay in FIR registration + reason for delay (mala fide inference)
- Nature of dispute is civil/commercial — misuse of criminal process
- Applicant's willingness to cooperate (undertaking in prayer)
- Arrest would cause irreparable reputational + financial harm
- "extraordinary relief" + "no direct evidence" language

PRAYER MUST INCLUDE:
- "grant pre-arrest bail in the sum of Rs. ___ with surety"
- "applicant shall join investigation whenever called"
- "applicant shall not leave the country without court permission"
- "interim pre-arrest bail during pendency"
""",

    "Quashment": """
STRATEGY: Quashment of FIR (Article 199)

CORE THEORY:
- FIR discloses no cognizable offence on its face
- AND/OR FIR is mala fide — filed to harass, coerce, or extort
- This Court's inherent jurisdiction to prevent abuse of process

MUST INCLUDE IN GROUNDS:
- "FIR does not disclose ingredients of the alleged offence"
  (analyse each element of the section vs facts alleged)
- Complainant's motive — prior civil dispute, property matter, family feud
- Inordinate delay in reporting (belated FIR = mala fide)
- "permitting FIR to continue = abuse of process"
- Cite Supreme Court: mala fide FIRs must be quashed at earliest stage

PRAYER MUST INCLUDE:
- "quash/set aside FIR No. ___ dated ___ at ___ Police Station"
- "all consequential proceedings arising from impugned FIR"
- "interim: stay arrest of petitioner during pendency"
""",

    "Property/Encroachment": """
STRATEGY: Constitutional Petition (Property / SBCA / Encroachment)

UPDATED AUTHORITY NAME:
- Use "Sindh Building Control Authority (SBCA)" — KBCA was renamed
- Cite: Sindh Building Control Ordinance, 1979 (as amended)

CORE THEORY:
- Sealing/demolition without notice violates natural justice (audi alteram partem)
- Article 10A: right to be heard before adverse order
- Article 23: right to acquire and hold property

MUST INCLUDE IN GROUNDS:
- No show-cause notice issued before sealing/demolition order
- Petitioner was not heard — violation of audi alteram partem
- Authority acted ultra vires its statutory mandate
- Cite SBCO 1979 — specific sections on notice and hearing requirements
- Economic loss + fundamental rights violation

PRAYER MUST INCLUDE (CRITICAL — INTERIM RELIEF):
- "suspend operation of impugned sealing/demolition order"
- "de-seal the premises forthwith"
- "respondents not to interfere with petitioner's possession during pendency"
- "show cause why petition should not be allowed"
""",
}

def strategy_node(state: LegalGenState) -> dict:
    print("🧠 [Strategy] Generating legal strategy memo...")
    p_type    = state.get("petition_type", "Habeas Corpus")
    red_flags = state.get("red_flags", [])

    base_strategy = STRATEGY_TEMPLATES.get(p_type, STRATEGY_TEMPLATES["Habeas Corpus"])

    if red_flags:
        flag_block = "\n\nRED FLAGS TO INCORPORATE:\n" + "\n".join(f"- {f}" for f in red_flags)
        full_strategy = base_strategy + flag_block
    else:
        full_strategy = base_strategy

    print(f"   → Strategy loaded for: {p_type}")
    return {"legal_strategy": full_strategy}

print("✅ Strategy engine ready")

✅ Strategy engine ready


In [13]:
# ── CELL 6D: Citation Ranker ──────────────────────────────────────────────────
# Forces PLD > SCMR > YLR > CLC > CPLA hierarchy

CITATION_TIER_MAP = {
    "PLD":   1,   # Pakistan Legal Decisions — highest authority
    "SCMR":  2,   # Supreme Court Monthly Review
    "YLR":   3,   # Yearly Law Reporter
    "CLC":   4,   # Civil Law Cases
    "CPLA":  5,   # Civil Petition for Leave to Appeal — WEAKEST (not a judgment)
    "MLD":   4,
}

def get_citation_tier(citation_str: str) -> tuple[str, int]:
    """Detect the citation tier from a citation string."""
    c = citation_str.upper()
    for tier, rank in CITATION_TIER_MAP.items():
        if tier in c:
            return tier, rank
    return "UNKNOWN", 99


def citation_ranker_node(state: LegalGenState) -> dict:
    print("📚 [CitationRanker] Evaluating citation authority...")
    primary   = state.get("primary_citation", "")
    secondary = state.get("supporting_citation", "")

    p_tier, p_rank = get_citation_tier(primary)
    s_tier, s_rank = get_citation_tier(secondary)

    issues = []

    if p_rank >= 5:
        issues.append(
            f"Primary citation '{primary}' is {p_tier} (rank {p_rank}/5) — "
            f"CPLA is a petition for leave, NOT a judgment. "
            f"Replace with PLD or SCMR citation."
        )

    if s_rank >= 5:
        issues.append(
            f"Supporting citation '{secondary}' is {p_tier} — weak. "
            f"Prefer PLD/SCMR."
        )

    # Warn if both are the same tier
    if p_tier == s_tier and p_tier != "UNKNOWN":
        issues.append(
            f"Both citations are {p_tier} — diversify: use one SCMR + one PLD for stronger authority."
        )

    if issues:
        print(f"   ⚠️  Citation issues detected:")
        for i in issues:
            print(f"      → {i}")
    else:
        print(f"   ✅ Primary: {p_tier} (rank {p_rank}) | Supporting: {s_tier} (rank {s_rank})")

    # Build citation warning to inject into drafter prompt
    citation_warning = "\n".join(issues) if issues else ""

    return {
        "citation_tier":  p_tier,
        "validation_notes": state.get("validation_notes", "") + ("\n" + citation_warning if citation_warning else ""),
    }

print("✅ Citation ranker ready")

✅ Citation ranker ready


In [21]:
# ── CELL 7: RAG Primary + Secondary ──────────────────────────────────────────
def rag_primary_node(state: LegalGenState) -> dict:
    print(f"🔎 [RAG Primary] Attempt {state.get('rag_attempts', 0) + 1}...")
    attempts = state.get("rag_attempts", 0)
    p_type   = state.get("petition_type", "")

    if attempts == 0:
        query = state["user_story"]
    elif attempts == 1:
        query = f"{p_type} illegal arrest without warrant Pakistan Supreme Court"
    else:
        query = "fundamental rights Article 9 10A Constitution Pakistan detention"

    results = collection.query(query_texts=[query], n_results=3)

    if not results['documents'] or not results['documents'][0]:
        return {
            "primary_context":  "No precedent found.",
            "primary_citation": "Articles 9 & 10A Constitution of Pakistan 1973",
            "rag_attempts":     attempts + 1,
            "next_step":        "rag_secondary"
        }

    best_doc  = results['documents'][0][0]
    best_meta = results['metadatas'][0][0]
    citation  = best_meta.get('case_id', 'Unknown')

    # Relevance check
    rel_resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
Is this case relevant to the user's situation?
USER: {state['user_story'][:300]}
CASE SUMMARY: {best_doc[:400]}
PETITION TYPE: {p_type}
Return ONLY JSON: {{"is_relevant": true/false, "reason": "one line"}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )
    rel = json.loads(rel_resp.choices[0].message.content)

    if not rel.get("is_relevant") and attempts < 2:
        print(f"   ⚠️ Not relevant ({rel.get('reason')}) — retrying...")
        return {
            "primary_context":  "",
            "primary_citation": "",
            "rag_attempts":     attempts + 1,
            "next_step":        "rag_retry"
        }

    print(f"   → Primary: {citation}")
    return {
        "primary_context":  best_doc,
        "primary_citation": citation,
        "rag_attempts":     attempts + 1,
        "next_step":        "rag_secondary"
    }


def rag_secondary_node(state: LegalGenState) -> dict:
    print("🔎 [RAG Secondary] Finding supporting authority...")
    p_type = state.get("petition_type", "")
    secondary_queries = {
        "Habeas Corpus":          "Article 9 liberty illegal detention writ habeas corpus warrant",
        "Post-Arrest Bail":       "Section 497 CrPC bail grant factors surety investigation incomplete",
        "Pre-Arrest Bail":        "Section 498 CrPC pre-arrest bail anticipatory mala fide FIR",
        "Quashment":              "Article 199 quashment FIR mala fide abuse process court",
        "Property/Encroachment":  "Article 10A 23 natural justice notice hearing sealing property rights"
    }
    query   = secondary_queries.get(p_type, "fundamental rights Constitution Pakistan")
    results = collection.query(query_texts=[query], n_results=5)

    if not results['documents'] or not results['documents'][0]:
        return {
            "supporting_context":  "General constitutional principles apply.",
            "supporting_citation": "Constitution of Pakistan 1973"
        }

    primary_cit = state.get("primary_citation", "")
    for i in range(len(results['documents'][0])):
        cit = results['metadatas'][0][i].get('case_id', '')
        if cit != primary_cit:
            print(f"   → Supporting: {cit}")
            return {
                "supporting_context":  results['documents'][0][i],
                "supporting_citation": cit
            }

    # Fallback to first
    cit = results['metadatas'][0][0].get('case_id', 'Constitution of Pakistan')
    return {
        "supporting_context":  results['documents'][0][0],
        "supporting_citation": cit
    }

print("✅ RAG nodes ready")


✅ RAG nodes ready


In [14]:
# ── CELL 8: Extractor Node (FACTS + GROUNDS split) ───────────────────────────
def extract_fields_node(state: LegalGenState) -> dict:
    print("📝 [Extractor] Pulling structured fields...")
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
You are a Pakistani legal drafter. Extract and generate structured petition fields.

STORY: {state['user_story']}
PETITION TYPE: {state.get('petition_type', '')}
PRIMARY CITATION: {state.get('primary_citation', '')}
SUPPORTING CITATION: {state.get('supporting_citation', '')}

RULES:
- Extract ONLY names that appear in the STORY. Do not invent names.
- facts: 3-4 short factual paragraphs starting with "That". Pure facts only, no law.
- grounds: 6+ lettered grounds (A. Because...). Each must cite a law, article, or case.
  Reference PRIMARY CITATION explicitly in at least 2 grounds.
- prayer: specific numbered prayers matching the petition type.
- No repetition between facts and grounds.

Return ONLY valid JSON with these exact keys:
{{
  "petitioner_name": "...",
  "detenu_name": "...",
  "facts": ["That ...", "That ...", "That ..."],
  "grounds": ["A. Because ...", "B. Because ...", "C. Because ...", ...],
  "prayer": "..."
}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )

    result      = json.loads(resp.choices[0].message.content)
    facts_list  = result.get("facts", [])
    grounds_list = result.get("grounds", [])

    return {
        "petitioner":   result.get("petitioner_name", "The Petitioner"),
        "detenu":       result.get("detenu_name",     "The Affected Person"),
        "facts_text":   "\n\n".join(facts_list),
        "grounds_text": "\n\n".join(grounds_list),
        "prayer":       result.get("prayer", "Grant the relief prayed for.")
    }

print("✅ Extractor updated — FACTS and GROUNDS now separate")

✅ Extractor updated — FACTS and GROUNDS now separate


In [15]:
# ── CELL 8B: Interim Relief Injector ─────────────────────────────────────────
# Automatically adds the correct interim prayer for each petition type

INTERIM_RELIEF_MAP = {
    "Habeas Corpus": """i. This Honourable Court may be pleased to issue a Rule Nisi calling upon the Respondents to show cause why the detenu should not be released forthwith;
ii. During the pendency of the above rule, the Respondents be directed to produce the detenu before this Honourable Court on the next date of hearing;
iii. The Respondents be directed to intimate this Honourable Court of the place of detention and the identity and rank of the arresting/detaining officers forthwith;
iv. Any other interim order this Honourable Court may deem fit in the interest of justice.""",

    "Post-Arrest Bail": """i. The Applicant be admitted to interim bail during the pendency of this application, on such terms and conditions as this Honourable Court may deem fit;
ii. The Investigating Officer be directed to submit the investigation report / challan before the next date of hearing;
iii. Any other interim relief this Honourable Court deems appropriate.""",

    "Pre-Arrest Bail": """i. The Applicant be granted interim pre-arrest bail forthwith during the pendency of this application;
ii. The Respondents be restrained from arresting the Applicant during the pendency of this application;
iii. Subject to grant of bail, the Applicant undertakes to join investigation whenever called upon and shall not leave the country without prior permission of this Honourable Court;
iv. Any other interim order this Honourable Court may deem fit.""",

    "Quashment": """i. During the pendency of this petition, the Respondents be restrained from arresting the Petitioner in pursuance of the impugned FIR;
ii. A Rule Nisi be issued calling upon the Respondents to show cause why the impugned FIR should not be quashed;
iii. Any consequential proceedings arising from the impugned FIR be stayed during pendency;
iv. Any other interim relief this Honourable Court deems fit.""",

    "Property/Encroachment": """i. During the pendency of this petition, the operation of the impugned sealing/demolition order be suspended forthwith;
ii. The Respondents be directed to de-seal the premises of the Petitioner and restore possession immediately;
iii. The Respondents be restrained from causing any further damage, encroachment, or interference with the Petitioner's property during pendency;
iv. A Rule Nisi be issued calling upon the Respondents to show cause why the impugned order be not set aside;
v. Any other interim relief this Honourable Court deems fit in the interest of justice.""",
}

def interim_relief_node(state: LegalGenState) -> dict:
    print("⚡ [InterimRelief] Injecting urgency prayers...")
    p_type = state.get("petition_type", "Habeas Corpus")
    relief = INTERIM_RELIEF_MAP.get(p_type, INTERIM_RELIEF_MAP["Habeas Corpus"])
    print(f"   → Interim relief injected for: {p_type}")
    return {"interim_relief": relief}

print("✅ Interim relief injector ready")

✅ Interim relief injector ready


In [16]:
# ── CELL 9: Drafter Node — Court-Ready Structure ─────────────────────────────
def drafter_node(state: LegalGenState) -> dict:
    p_type       = state.get("petition_type", "Habeas Corpus")
    jurisdiction = state.get("jurisdiction", "Sindh HC")
    court_info   = COURT_FORMATS.get(jurisdiction, COURT_FORMATS["Sindh HC"])

    print(f"✍️ [Drafter] Writing {p_type} — {jurisdiction}...")

    petitioner     = state.get("petitioner")    or "The Petitioner"
    detenu         = state.get("detenu")         or "The Affected Person"
    facts_text     = state.get("facts_text")     or "That the detention is illegal and without lawful authority."
    grounds_text   = state.get("grounds_text")   or "A. Because the detention is in violation of Article 9 of the Constitution."
    prayer         = state.get("prayer")         or "Grant the relief prayed for."
    primary_cit    = state.get("primary_citation")    or "the Supreme Court"
    supporting_cit = state.get("supporting_citation") or "the Supreme Court"
    court_header   = court_info["full_name"]
    home_sec       = court_info["home_sec"]
    ag             = court_info["ag"]
    user_story     = state.get("user_story", "")
    legal_strategy = state.get("legal_strategy", "")
    red_flags      = state.get("red_flags", [])
    interim_relief = state.get("interim_relief", "")
    citation_tier  = state.get("citation_tier", "UNKNOWN")

    # Detect location
    loc_match = re.search(r'([\w\-\s]+ police station)', user_story, re.IGNORECASE)
    location  = loc_match.group(0).strip() if loc_match else "the concerned Police Station"

    # Detect authority — updated to SBCA
    story_lower = user_story.lower()
    if "kbca" in story_lower or "sbca" in story_lower:
        authority = "Sindh Building Control Authority (SBCA)"
    elif "kmc" in story_lower:
        authority = "Karachi Metropolitan Corporation (KMC)"
    else:
        authority = "The Concerned Authority"

    # ── Type-specific header block ─────────────────────────────────────────────
    header_map = {
        "Habeas Corpus": f"""{court_header}

Writ Petition No. ___/2026 (Habeas Corpus)
Under Article 199(1)(b)(i) of the Constitution of Pakistan, 1973

{petitioner}                                       ...Petitioner
Versus
1. Station House Officer, {location}
2. {home_sec}
3. {ag}                                            ...Respondents

HABEAS CORPUS PETITION""",

        "Post-Arrest Bail": f"""{court_header}

Crl. Misc. Application No. ___/2026
Under Section 497, Code of Criminal Procedure, 1898
[Section 497(2) CrPC — Further Inquiry]

{detenu} (Applicant/Accused)                       ...Applicant
Versus
The State                                          ...Respondent

APPLICATION FOR POST-ARREST BAIL""",

        "Pre-Arrest Bail": f"""{court_header}

Crl. Misc. Application No. ___/2026
Under Section 498, Code of Criminal Procedure, 1898

{petitioner} (Applicant/Accused)                   ...Applicant
Versus
The State                                          ...Respondent

APPLICATION FOR PRE-ARREST BAIL
[Extraordinary Relief — Concession Not Right]""",

        "Quashment": f"""{court_header}

Writ Petition No. ___/2026
Under Article 199 of the Constitution of Pakistan, 1973

{petitioner}                                       ...Petitioner
Versus
1. Station House Officer, {location}
2. The State
3. The Complainant                                 ...Respondents

CONSTITUTIONAL PETITION (QUASHMENT OF FIR)""",

        "Property/Encroachment": f"""{court_header}

Writ Petition No. ___/2026
Under Articles 199, 23 & 10A of the Constitution of Pakistan, 1973
Read with Sindh Building Control Ordinance, 1979

{petitioner}                                       ...Petitioner
Versus
1. {authority}
2. Province of Sindh                               ...Respondents

CONSTITUTIONAL PETITION
[Impugned: Sealing/Demolition Order — Stay Urgently Sought]""",
    }

    header = header_map.get(p_type, header_map["Habeas Corpus"])

    # ── Extra constitutional grounds per type ─────────────────────────────────
    extra_map = {
        "Habeas Corpus": f"""G. Because Article 9 of the Constitution categorically provides that no person shall be deprived of life or liberty save in accordance with law. The detention herein is mala fide and without lawful authority — a patent violation.

H. Because Article 10 of the Constitution and Section 61 Cr.P.C. mandate that every arrested person be produced before a Magistrate within 24 hours. No such production has taken place, rendering the continued detention wholly illegal.

I. Because the arresting officers were plain-clothes personnel whose identity, rank, and legal authority to arrest have not been disclosed — an additional badge of illegality attaching to the impugned detention.

J. Because this Honourable Court, in exercise of its constitutional jurisdiction under Article 199, is not only empowered but duty-bound to protect fundamental rights and issue a writ of Habeas Corpus. Reliance is placed on {supporting_cit}. The Respondents must be directed to produce the detenu, submit para-wise comments, and intimate this Court of the place of detention forthwith.""",

        "Post-Arrest Bail": f"""G. Because the instant case requires "further inquiry" within the meaning of Section 497(2) CrPC — the evidence on record is contradictory, the role of the Applicant is not specifically established, and a deeper examination of the material is imperative before any adverse conclusion can be drawn.

H. Because as held consistently by the Supreme Court, including in {supporting_cit}, bail is the rule and jail is the exception, and punitive incarceration before conviction strikes at the root of Article 10A of the Constitution which guarantees the right to a fair trial.

I. Because the Applicant has no previous criminal antecedents, has deep family roots in the community, and is not a flight risk — satisfying both limbs of the conventional twin-test for bail under Section 497(1) CrPC.

J. Because no direct incriminating evidence has been placed on record. The Applicant's continued detention is causing irreparable harm to his family, livelihood, and is serving no investigative purpose at this stage.""",

        "Pre-Arrest Bail": f"""G. Because the FIR was registered with an inordinate and unexplained delay, which in itself is a strong inference of mala fide intent and an afterthought designed to harass, coerce, and humiliate the Applicant in a dispute that is civil and personal in nature.

H. Because pre-arrest bail is an extraordinary concession — not a right — and is warranted where, as here, the complainant has an ulterior motive, the FIR is an abuse of the criminal process, and the Applicant is willing to fully cooperate with investigation.

I. Because arrest at this stage would cause irreparable reputational harm, financial loss, and emotional trauma to the Applicant and his family, with no corresponding benefit to the investigation whatsoever.

J. Because as held in {supporting_cit}, this Honourable Court grants pre-arrest bail where the risk of abuse of process and misuse of criminal law in a personal or commercial dispute is apparent on the face of the record.""",

        "Quashment": f"""G. Because this Honourable Court exercises plenary jurisdiction under Article 199 of the Constitution to quash any FIR that, on its face, does not disclose the essential ingredients of the alleged offence, or has been lodged with mala fide intent as an abuse of the process of law.

H. Because the facts alleged in the impugned FIR, even if taken at their highest and accepted in entirety, do not disclose the commission of the offence set out therein. No cognizable offence is made out on the face of the FIR.

I. Because the complainant and the Petitioner are involved in an ongoing civil/personal dispute, and the FIR is a transparent attempt to convert a civil matter into a criminal complaint and to use police machinery as a tool of coercion.

J. Because permitting the impugned FIR to continue would cause irreparable prejudice and stigma, and as held in {supporting_cit}, the Supreme Court has consistently held that mala fide FIRs must be quashed at the earliest stage to prevent further abuse of the criminal justice system.""",

        "Property/Encroachment": f"""G. Because Article 23 of the Constitution of Pakistan, 1973, guarantees every citizen the right to acquire, hold, and dispose of property — a right being arbitrarily, illegally, and forcibly violated by the Respondents without any lawful authority.

H. Because Article 10A of the Constitution guarantees the right to a fair trial and due process, including the right to be heard and given notice before any adverse action is taken. The Respondents issued the impugned order without any prior notice or opportunity to be heard — a flagrant violation of the audi alteram partem principle.

I. Because the Sindh Building Control Ordinance, 1979 prescribes a mandatory statutory procedure — including show-cause notice, inquiry, and order — before any sealing or demolition can be effected. No such procedure was followed, rendering the impugned order void ab initio.

J. Because as held in {supporting_cit}, regulatory authorities cannot act in excess of their statutory mandate, and any order made in violation of mandatory procedural requirements and without hearing the affected party is void and of no legal effect whatsoever.""",
    }

    extra = extra_map.get(p_type, "")

    # ── Red flag warning block ────────────────────────────────────────────────
    red_flag_block = ""
    if red_flags:
        red_flag_block = f"""
CRITICAL RED FLAGS — STRICTLY OBSERVE:
{chr(10).join(f'  ⚠️ {f}' for f in red_flags)}

CITATION AUTHORITY WARNING:
Primary citation tier: {citation_tier}
{"⚠️ CPLA citations are petitions for leave — NOT judgments. Use PLD/SCMR citations for legal propositions." if citation_tier == "CPLA" else "✅ Citation tier is acceptable."}
"""

    # ── Verification block ─────────────────────────────────────────────────────
    verification = f"""VERIFICATION

I, {petitioner}, do hereby solemnly affirm and declare that the contents of this petition are true and correct to the best of my knowledge and belief, that nothing material has been concealed, and that the petition is filed bona fide.

Sworn at {court_info['city']} on this _____ day of __________, 2026.

                                         _______________________
                                         {petitioner}
                                         Deponent

Through Counsel:
_______________________
Advocate, {court_info['short']}
Enrollment No.: _______
Cell: _________________"""

    # ── Master prompt ──────────────────────────────────────────────────────────
    prompt = f"""
You are a Senior Advocate of the {court_info['short']} with 20 years of experience.

Write a complete, court-ready petition using EXACTLY the structure below.
Do NOT add disclaimers. Do NOT write "sample" or "draft". Do NOT change names or citations.
Use sharp, urgent, formal Pakistani court language. This is a live writ court.

═══════════════════════════════════════════════════════
{header}

Most Respectfully Sheweth:

FACTS

{facts_text}

GROUNDS

{grounds_text}

{extra}

PRAYER

It is therefore most respectfully prayed that this Honourable Court may be pleased to:

{prayer}

INTERIM/URGENT PRAYER

It is further prayed that in the interim, pending final adjudication:

{interim_relief}

{verification}
═══════════════════════════════════════════════════════

LEGAL STRATEGY MEMO (follow this — do not reproduce in petition):
{legal_strategy}

{red_flag_block}

CRITICAL DRAFTING RULES:
1. FACTS and GROUNDS must be clearly separated — never merge them.
2. Use ONLY names: petitioner = "{petitioner}", detenu/accused = "{detenu}".
3. Reference "{primary_cit}" by name in AT LEAST 2 grounds with a specific proposition.
4. "{primary_cit}" is {citation_tier} — {"NOTE: argue from principle, not just from the CPLA number. State the legal proposition the court held." if citation_tier == "CPLA" else "use it as authority for a specific legal proposition."}
5. Grounds follow lettered format: A. Because... B. Because... etc. (minimum 8 grounds).
6. Tone must be urgent and forceful — "mala fide", "without lawful authority", "patent violation".
7. No repetition — each ground makes a DISTINCT legal point.
8. Include the INTERIM/URGENT PRAYER section verbatim as drafted above.
9. The Property petition must cite Sindh Building Control Ordinance 1979 + SBCA (not KBCA).
10. Write the complete petition now:
"""

    chain = ChatPromptTemplate.from_template("{text}") | llm | StrOutputParser()
    draft = chain.invoke({"text": prompt})
    return {"final_petition": draft}

print("✅ Drafter upgraded — strategy + red flags + interim relief + SBCA fix")

✅ Drafter upgraded — strategy + red flags + interim relief + SBCA fix


In [17]:
# ── CELL 10: Validator Node ───────────────────────────────────────────────────
def validator_node(state: LegalGenState) -> dict:
    print("✅ [Validator] Checking petition quality...")

    p_type   = state.get("petition_type", "")
    citation = state.get("primary_citation", "")
    petition = state.get("final_petition", "")[:3000]
    story    = state.get("user_story", "")
    names_in_story = re.findall(r'[A-Z][a-z]+ [A-Z][a-z]+', story)

    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": f"""
You are a senior judge's clerk reviewing a Pakistani court petition.

PETITION TYPE: {p_type}
PRIMARY CITATION: {citation}
NAMES FROM STORY: {names_in_story}

PETITION TEXT:
{petition}

Score each dimension 0-10:
1. structure_score: Are FACTS, GROUNDS, and PRAYER clearly separated as distinct sections?
2. citation_score: Is "{citation}" named in GROUNDS and tied to a specific legal proposition (not just mentioned)?
3. grounds_count: How many lettered grounds exist (A., B., C., ...)? Return the COUNT as integer.
4. redundancy_score: Is each ground making a DISTINCT point with NO repetition? (10 = fully distinct)
5. tone_score: Is the language sharp, urgent, formal Pakistani court language? (10 = excellent)
6. no_hallucinated_names: Are all person names in petition found in NAMES FROM STORY or standard titles (SHO, Home Secretary, Advocate General)? true/false

is_valid: true if structure_score >= 7 AND citation_score >= 6 AND grounds_count >= 5
overall: weighted average (structure 30% + citation 25% + redundancy 25% + tone 20%)
issues: list ONLY real problems, empty array if none

Return ONLY valid JSON:
{{
  "structure_score": 0-10,
  "citation_score": 0-10,
  "grounds_count": integer,
  "redundancy_score": 0-10,
  "tone_score": 0-10,
  "no_hallucinated_names": true/false,
  "is_valid": true/false,
  "overall": 0.0-10.0,
  "issues": [],
  "notes": "one line"
}}
"""}],
        response_format={"type": "json_object"},
        temperature=0
    )

    result         = json.loads(resp.choices[0].message.content)
    is_valid       = result.get("is_valid", False)
    revision_count = state.get("revision_count", 0)
    issues         = [i for i in result.get("issues", []) if i]

    print(f"   → Valid: {is_valid} | Overall: {result.get('overall', 0):.1f}/10")
    if issues:
        print(f"   → Issues: {issues}")

    if not is_valid and revision_count < 1:
        print("   ⚠️ Sending for revision...")
        return {
            "is_valid":            False,
            "validation_notes":    "\n".join(issues),
            "validation_score":    int(result.get("overall", 0) * 10),
            "eval_structure_score":  result.get("structure_score", 0),
            "eval_citation_score":   result.get("citation_score", 0),
            "eval_grounds_count":    result.get("grounds_count", 0),
            "eval_redundancy_score": result.get("redundancy_score", 0),
            "eval_tone_score":       result.get("tone_score", 0),
            "eval_overall_score":    result.get("overall", 0.0),
            "eval_issues":           issues,
            "eval_timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
            "revision_count":        revision_count + 1,
            "next_step":             "revise"
        }

    return {
        "is_valid":            True,
        "validation_notes":    result.get("notes", "Approved."),
        "validation_score":    int(result.get("overall", 0) * 10),
        "eval_structure_score":  result.get("structure_score", 0),
        "eval_citation_score":   result.get("citation_score", 0),
        "eval_grounds_count":    result.get("grounds_count", 0),
        "eval_redundancy_score": result.get("redundancy_score", 0),
        "eval_tone_score":       result.get("tone_score", 0),
        "eval_overall_score":    result.get("overall", 0.0),
        "eval_issues":           issues,
        "eval_timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
        "revision_count":        revision_count,
        "next_step":             "done"
    }

print("✅ Validator updated — numeric scores + eval fields")

✅ Validator updated — numeric scores + eval fields


In [18]:
# ── CELL 11: Revision Node ────────────────────────────────────────────────────
def revision_node(state: LegalGenState) -> dict:
    print("🔧 [Revision] Fixing validator issues...")
    issues = state.get("validation_notes", "No specific issues noted.")

    prompt = f"""
You are a Senior Pakistani Advocate. Revise this petition to fix the issues noted below.

ISSUES TO FIX:
{issues}

STRUCTURE REQUIREMENT — the petition MUST have these three clearly labelled sections:
FACTS      → factual paragraphs starting with "That", no legal argument
GROUNDS    → lettered grounds A. Because... B. Because... citing articles and cases
PRAYER     → numbered reliefs sought

ORIGINAL PETITION:
{state.get('final_petition', '')}

RULES:
- Fix ONLY the listed issues
- Keep all citations and article references
- Ensure FACTS / GROUNDS / PRAYER are clearly separated
- Do NOT add disclaimers
- Return the complete revised petition only
"""
    chain   = ChatPromptTemplate.from_template("{text}") | llm | StrOutputParser()
    revised = chain.invoke({"text": prompt})
    return {"final_petition": revised}

print("✅ Revision node ready")

✅ Revision node ready


In [22]:
# ── CELL 12: Graph Assembly ───────────────────────────────────────────────────
def route_after_info(state):      return state.get("next_step", "ask_user")
def route_after_rag(state):       return state.get("next_step", "rag_secondary")
def route_after_validator(state): return state.get("next_step", "done")

builder = StateGraph(LegalGenState)

# Register all nodes
builder.add_node("info_check",      check_missing_info_node)
builder.add_node("orchestrator",    orchestrator_node)
builder.add_node("red_flag",        red_flag_node)          # NEW
builder.add_node("strategy",        strategy_node)          # NEW
builder.add_node("citation_ranker", citation_ranker_node)   # NEW
builder.add_node("rag_primary",     rag_primary_node)
builder.add_node("rag_secondary",   rag_secondary_node)
builder.add_node("extractor",       extract_fields_node)
builder.add_node("interim_relief",  interim_relief_node)    # NEW
builder.add_node("drafter",         drafter_node)
builder.add_node("validator",       validator_node)
builder.add_node("revision",        revision_node)

# Wire the graph
builder.add_edge(START, "info_check")
builder.add_conditional_edges("info_check", route_after_info,
    {"ask_user": END, "proceed": "orchestrator"})
builder.add_edge("orchestrator",    "red_flag")
builder.add_edge("red_flag",        "strategy")
builder.add_edge("strategy",        "rag_primary")
builder.add_conditional_edges("rag_primary", route_after_rag,
    {"rag_retry": "rag_primary", "rag_secondary": "rag_secondary"})
builder.add_edge("rag_secondary",   "citation_ranker")
builder.add_edge("citation_ranker", "extractor")
builder.add_edge("extractor",       "interim_relief")
builder.add_edge("interim_relief",  "drafter")
builder.add_edge("drafter",         "validator")
builder.add_conditional_edges("validator", route_after_validator,
    {"revise": "revision", "done": END})
builder.add_edge("revision", END)

legal_gen_app = builder.compile()
print("✅ Full graph compiled")
print("   Flow: InfoCheck → Orchestrator → RedFlag → Strategy → RAG(x2)")
print("         → CitationRanker → Extractor → InterimRelief → Drafter → Validator → [Revision] → END")

✅ Full graph compiled
   Flow: InfoCheck → Orchestrator → RedFlag → Strategy → RAG(x2)
         → CitationRanker → Extractor → InterimRelief → Drafter → Validator → [Revision] → END


In [23]:
# ── CELL 13: Runner ───────────────────────────────────────────────────────────
def run_legal_assistant(story: str, user_id: str = "default"):
    mem_ctx = get_memory_context(user_id)
    if USER_MEMORY.get(user_id, {}).get("past_petitions"):
        print(f"💾 Memory loaded — {len(USER_MEMORY[user_id]['past_petitions'])} past petition(s)")

    state = {
        "user_story": story, "user_id": user_id, "jurisdiction": "",
        "is_complete": False, "missing_info": [], "petition_type": "",
        "target_court": "", "next_step": "", "primary_context": "",
        "primary_citation": "", "supporting_context": "", "supporting_citation": "",
        "rag_attempts": 0, "petitioner": "", "detenu": "", "facts_text": "",
        "grounds_text": "", "prayer": "", "is_valid": False,
        "validation_notes": "", "validation_score": 0, "revision_count": 0,
        "final_petition": "", "memory_context": mem_ctx,
        # new fields
        "red_flags": [], "legal_strategy": "", "citation_tier": "UNKNOWN",
        "interim_relief": "",
        # eval
        "eval_structure_score": 0, "eval_citation_score": 0,
        "eval_grounds_count": 0, "eval_redundancy_score": 0,
        "eval_tone_score": 0, "eval_overall_score": 0.0,
        "eval_issues": [], "eval_timestamp": "",
    }

    # Step 1: Info check
    info_result = check_missing_info_node(state)
    state.update(info_result)
    if not state["is_complete"]:
        print(f"\n📋 Missing: {', '.join(state['missing_info'])}")
        reply = input("👤 Please provide these details: ")
        state["user_story"] += f"\nAdditional details: {reply}"
        state["is_complete"] = True
        state["next_step"]   = "proceed"

    # Step 2: Optional jurisdiction override
    print("\n🏛️  Jurisdiction (Enter to auto-detect, or type one):")
    print("   Options: Sindh HC | Lahore HC | Islamabad HC | Sessions Court Karachi | Sessions Court Lahore")
    jur_input = input("   → ").strip()
    if jur_input in COURT_FORMATS:
        state["jurisdiction"] = jur_input
        print(f"   Using: {jur_input}")
    else:
        print("   Auto-detecting from story...")

    # Step 3: Run pipeline (all nodes)
    builder2 = StateGraph(LegalGenState)
    for name, fn in [
        ("orchestrator",    orchestrator_node),
        ("red_flag",        red_flag_node),
        ("strategy",        strategy_node),
        ("rag_primary",     rag_primary_node),
        ("rag_secondary",   rag_secondary_node),
        ("citation_ranker", citation_ranker_node),
        ("extractor",       extract_fields_node),
        ("interim_relief",  interim_relief_node),
        ("drafter",         drafter_node),
        ("validator",       validator_node),
        ("revision",        revision_node),
    ]:
        builder2.add_node(name, fn)

    builder2.add_edge(START, "orchestrator")
    builder2.add_edge("orchestrator",    "red_flag")
    builder2.add_edge("red_flag",        "strategy")
    builder2.add_edge("strategy",        "rag_primary")
    builder2.add_conditional_edges("rag_primary", route_after_rag,
        {"rag_retry": "rag_primary", "rag_secondary": "rag_secondary"})
    builder2.add_edge("rag_secondary",   "citation_ranker")
    builder2.add_edge("citation_ranker", "extractor")
    builder2.add_edge("extractor",       "interim_relief")
    builder2.add_edge("interim_relief",  "drafter")
    builder2.add_edge("drafter",         "validator")
    builder2.add_conditional_edges("validator", route_after_validator,
        {"revise": "revision", "done": END})
    builder2.add_edge("revision", END)

    pipeline = builder2.compile()
    output   = pipeline.invoke(state)
    state.update(output)

    # Step 4: Save memory
    save_to_memory(user_id, state)

    # Step 5: Print result
    if state.get("final_petition"):
        sep = "=" * 65
        print(f"\n{sep}")
        print(f"⚖️  {state.get('petition_type','').upper()}")
        print(f"🏛️  {state.get('jurisdiction','')}")
        print(f"📌 Primary:   {state.get('primary_citation','N/A')} [{state.get('citation_tier','?')}]")
        print(f"📌 Supporting:{state.get('supporting_citation','N/A')}")
        if state.get("red_flags"):
            print(f"🚨 Red flags: {len(state['red_flags'])}")
            for f in state["red_flags"]:
                print(f"   → {f}")
        print(f"✅ Valid: {state.get('is_valid')} | Score: {state.get('eval_overall_score',0):.1f}/10")
        print(f"📊 Struct:{state['eval_structure_score']} Cite:{state['eval_citation_score']} "
              f"Redun:{state['eval_redundancy_score']} Tone:{state['eval_tone_score']} "
              f"Grounds:{state['eval_grounds_count']}")
        print(sep)
        print(state["final_petition"])

        fname = f"/kaggle/working/{user_id}_{state.get('petition_type','petition').replace(' ','_')}.txt"
        with open(fname, "w") as f:
            f.write(state["final_petition"])
        print(f"\n💾 Saved to: {fname}")
    else:
        print("⚠️ No petition generated.")

    return state

print("✅ Runner ready")

✅ Runner ready


In [24]:
# ── CELL 14: Evaluation Metrics Dashboard ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_evaluation_dashboard(results: list[dict], title: str = "Legal Petition Quality Dashboard"):
    """
    results: list of state dicts returned by run_legal_assistant()
    """
    if not results:
        print("No results to plot.")
        return

    cases      = [r.get("user_id", f"Case {i+1}") for i, r in enumerate(results)]
    structure  = [r.get("eval_structure_score", 0)  for r in results]
    citation   = [r.get("eval_citation_score", 0)   for r in results]
    redundancy = [r.get("eval_redundancy_score", 0) for r in results]
    tone       = [r.get("eval_tone_score", 0)       for r in results]
    overall    = [r.get("eval_overall_score", 0.0)  for r in results]
    grounds    = [r.get("eval_grounds_count", 0)    for r in results]
    p_types    = [r.get("petition_type", "Unknown") for r in results]

    n     = len(results)
    x     = np.arange(n)
    width = 0.18

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.98)
    fig.patch.set_facecolor("#0f1117")
    for ax in axes.flat:
        ax.set_facecolor("#1a1d27")
        ax.spines[["top","right","left","bottom"]].set_color("#2d3148")
        ax.tick_params(colors="#c8cce8")
        ax.title.set_color("#e8eaf6")
        ax.xaxis.label.set_color("#a0a4c8")
        ax.yaxis.label.set_color("#a0a4c8")

    colors = {"Structure": "#7c83f0", "Citation": "#4dd0e1",
              "Redundancy": "#81c784", "Tone": "#ffb74d"}

    # ── Chart 1: Grouped bar — all 4 dimensions ───────────────────────────────
    ax1 = axes[0, 0]
    ax1.bar(x - 1.5*width, structure,  width, label="Structure",  color=colors["Structure"],  alpha=0.88)
    ax1.bar(x - 0.5*width, citation,   width, label="Citation",   color=colors["Citation"],   alpha=0.88)
    ax1.bar(x + 0.5*width, redundancy, width, label="Redundancy", color=colors["Redundancy"], alpha=0.88)
    ax1.bar(x + 1.5*width, tone,       width, label="Tone",       color=colors["Tone"],       alpha=0.88)
    ax1.set_xticks(x)
    ax1.set_xticklabels(cases, rotation=15, ha="right", fontsize=9)
    ax1.set_ylim(0, 11)
    ax1.set_ylabel("Score (0–10)")
    ax1.set_title("Quality Dimensions by Case")
    ax1.legend(fontsize=8, facecolor="#252836", labelcolor="#c8cce8",
               framealpha=0.7, loc="lower right")
    ax1.axhline(y=7, color="#ff6b6b", linewidth=1, linestyle="--", alpha=0.6, label="Min threshold (7)")
    for bars, vals in [(ax1.containers[0], structure), (ax1.containers[1], citation),
                       (ax1.containers[2], redundancy), (ax1.containers[3], tone)]:
        for bar, v in zip(bars, vals):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                     str(v), ha="center", va="bottom", fontsize=7, color="#c8cce8")

    # ── Chart 2: Overall score line ───────────────────────────────────────────
    ax2 = axes[0, 1]
    ax2.plot(cases, overall, marker="o", color="#7c83f0", linewidth=2.2,
             markersize=9, markerfacecolor="#fff", markeredgecolor="#7c83f0", markeredgewidth=2)
    ax2.fill_between(range(n), overall, alpha=0.15, color="#7c83f0")
    ax2.axhline(y=7, color="#ff6b6b", linewidth=1, linestyle="--", alpha=0.7)
    ax2.set_ylim(0, 10.5)
    ax2.set_xticks(range(n))
    ax2.set_xticklabels(cases, rotation=15, ha="right", fontsize=9)
    ax2.set_ylabel("Overall Score (0–10)")
    ax2.set_title("Overall Score per Case")
    for i, (c, v) in enumerate(zip(cases, overall)):
        ax2.annotate(f"{v:.1f}", (i, v), textcoords="offset points",
                     xytext=(0, 10), ha="center", fontsize=9, color="#e8eaf6")

    # ── Chart 3: Grounds count bar ────────────────────────────────────────────
    ax3 = axes[1, 0]
    bar_colors = ["#81c784" if g >= 5 else "#ffb74d" if g >= 3 else "#ef5350" for g in grounds]
    bars3 = ax3.bar(cases, grounds, color=bar_colors, alpha=0.85, width=0.55)
    ax3.axhline(y=5, color="#ff6b6b", linewidth=1.2, linestyle="--", alpha=0.7)
    ax3.set_ylabel("Number of Lettered Grounds")
    ax3.set_title("Grounds Count (min 5 required)")
    ax3.set_ylim(0, max(grounds + [6]) + 2)
    ax3.set_xticklabels(cases, rotation=15, ha="right", fontsize=9)
    for bar, v in zip(bars3, grounds):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(v), ha="center", va="bottom", fontsize=10, color="#e8eaf6", fontweight="bold")
    patches = [mpatches.Patch(color="#81c784", label="≥5 grounds (pass)"),
               mpatches.Patch(color="#ffb74d", label="3–4 grounds (warn)"),
               mpatches.Patch(color="#ef5350", label="<3 grounds (fail)")]
    ax3.legend(handles=patches, fontsize=8, facecolor="#252836", labelcolor="#c8cce8", framealpha=0.7)

    # ── Chart 4: Radar chart — avg scores across all cases ────────────────────
    ax4 = axes[1, 1]
    ax4.remove()
    ax4 = fig.add_subplot(2, 2, 4, polar=True)
    ax4.set_facecolor("#1a1d27")

    categories  = ["Structure", "Citation", "Redundancy", "Tone"]
    avg_scores  = [
        np.mean(structure),
        np.mean(citation),
        np.mean(redundancy),
        np.mean(tone)
    ]
    N      = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    vals   = avg_scores + avg_scores[:1]

    ax4.set_theta_offset(np.pi / 2)
    ax4.set_theta_direction(-1)
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(categories, size=10, color="#c8cce8")
    ax4.set_ylim(0, 10)
    ax4.set_yticks([2, 4, 6, 8, 10])
    ax4.set_yticklabels(["2","4","6","8","10"], size=7, color="#6b7090")
    ax4.plot(angles, vals, "o-", linewidth=2, color="#7c83f0")
    ax4.fill(angles, vals, alpha=0.25, color="#7c83f0")
    ax4.set_title("Average Quality Radar\n(all cases)", color="#e8eaf6", pad=18, fontsize=11)
    ax4.tick_params(colors="#555878")
    ax4.spines["polar"].set_color("#2d3148")
    ax4.grid(color="#2d3148", linewidth=0.8)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig("/kaggle/working/eval_dashboard.png", dpi=150, bbox_inches="tight",
                facecolor="#0f1117")
    plt.show()
    print("📊 Dashboard saved to /kaggle/working/eval_dashboard.png")


def print_eval_summary(results: list[dict]):
    """Print a clean text summary table of all evaluations."""
    print("\n" + "="*80)
    print("📊  EVALUATION SUMMARY")
    print("="*80)
    print(f"{'Case':<20} {'Type':<22} {'Struct':>6} {'Cite':>5} {'Redun':>6} {'Tone':>5} {'Grnds':>6} {'OVERALL':>8}")
    print("-"*80)
    for r in results:
        print(
            f"{r.get('user_id',''):<20} "
            f"{r.get('petition_type',''):<22} "
            f"{r.get('eval_structure_score',0):>6} "
            f"{r.get('eval_citation_score',0):>5} "
            f"{r.get('eval_redundancy_score',0):>6} "
            f"{r.get('eval_tone_score',0):>5} "
            f"{r.get('eval_grounds_count',0):>6} "
            f"{r.get('eval_overall_score',0.0):>7.1f}/10"
        )
    if results:
        avg_overall = np.mean([r.get("eval_overall_score", 0) for r in results])
        print("-"*80)
        print(f"{'AVERAGE':<49} {avg_overall:>40.1f}/10")
    print("="*80 + "\n")

print("✅ Evaluation metrics & dashboard ready")

✅ Evaluation metrics & dashboard ready


In [26]:
# ── CELL 15: Run All Test Cases + Evaluation Dashboard ───────────────────────
test_cases = [
    # Rafsha — Habeas Corpus (plain-clothes, no FIR)
    ("rafsha_01",
     "My brother Ali Khan was arrested without warrant from our home in Gulshan-e-Iqbal "
     "by plain clothes officers on March 24 2026. My name is Rafsha Rahim. "
     "He is being held at Gulshan Police Station. No FIR has been registered."),

    # Ahmed — Pre-Arrest Bail with HIGH-severity section 324
    ("ahmed_01",
     "An FIR No. 45/2026 has been registered against me at Clifton Police Station "
     "under Section 324 PPC (attempt to murder) on March 28 2026. "
     "I have not been arrested yet. My name is Ahmed Raza. "
     "The complainant is my business partner with whom I have a financial dispute."),

    # Sara — Post-Arrest Bail, Section 409 (non-bailable, serious)
    ("sara_01",
     "My client Bilal Sheikh was arrested on March 20 2026 under Section 409 PPC "
     "and has been in custody for 12 days at Defence Police Station. "
     "I am his lawyer Sara Khan. No direct evidence against him has been produced."),

    # Zara — Quashment (mala fide FIR, civil dispute)
    ("zara_01",
     "An FIR No. 88/2026 was registered against me at Saddar Police Station "
     "under Section 420 and 506 PPC by my former business partner. "
     "The dispute is entirely commercial — he owes me money. "
     "My name is Zara Mirza. The FIR was registered 3 months after the dispute started."),
]



In [27]:
# ── EXTRA TEST CASES: Advanced Legal Edge Scenarios ───────────────────────

extra_test_cases = [

    # 1. Habeas Corpus — Agency pickup (sensitive / unknown location)
    ("hassan_01",
     "My brother Hassan was picked up by unknown persons suspected to be law enforcement agencies "
     "from DHA Karachi on April 2 2026. My name is Ayesha Hassan. "
     "We have visited multiple police stations but no one is acknowledging his custody. "
     "His whereabouts are unknown."),

    # 2. Habeas Corpus — FIR exists but illegal detention beyond 24 hours
    ("usman_01",
     "My cousin Usman was arrested in FIR No. 101/2026 at Korangi Police Station "
     "on March 29 2026. My name is Farhan Ali. "
     "He has not been produced before any Magistrate even after 3 days of arrest."),

    # 3. Pre-Arrest Bail — Political victimization
    ("bilal_01",
     "An FIR has been lodged against me under Sections 147, 148, and 149 PPC "
     "at Model Colony Police Station after a political protest. "
     "My name is Bilal Ahmed. I fear arrest as I belong to an opposition party "
     "and this is a clear case of victimization."),

    # 4. Pre-Arrest Bail — Family dispute (false allegation)
    ("hina_01",
     "My in-laws have registered an FIR against me under Section 406 PPC at Gulberg Police Station. "
     "My name is Hina Noor. The allegations are false and arise out of matrimonial disputes. "
     "I have not been arrested yet."),

    # 5. Post-Arrest Bail — Recovery already made
    ("imran_01",
     "I am Imran Khan. I was arrested under Section 379 PPC on March 25 2026 "
     "and the alleged stolen items have already been recovered by police. "
     "I have been in custody for 10 days at Liaquatabad Police Station."),

    # 6. Post-Arrest Bail — Co-accused granted bail
    ("saad_01",
     "My brother Saad was arrested under Section 302 PPC in FIR No. 77/2026 "
     "at Malir Police Station. I am his sister Sana Saad. "
     "The co-accused in the same case has already been granted bail by the court."),

    # 7. Property — Demolition threat (urgent stay needed)
    ("naveed_01",
     "SBCA officials have issued a demolition notice to my house in North Nazimabad "
     "claiming illegal construction. My name is Naveed Iqbal. "
     "I was not given proper opportunity to be heard and demolition is expected tomorrow."),

    # 8. Property — Encroachment by private party (not authority)
    ("asad_01",
     "My neighbor has illegally occupied a portion of my land in PECHS Karachi. "
     "My name is Asad Malik. Despite complaints, police have not taken any action."),

    # 9. Quashment — FIR after long delay (strong mala fide)
    ("fatima_01",
     "An FIR under Section 506 PPC was registered against me after 6 months of a minor dispute. "
     "My name is Fatima Zahra. The complainant is misusing the law to harass me."),

    # 10. Cyber Crime — FIA involvement
    ("ali_01",
     "FIA Cyber Crime Wing has issued me a notice under PECA Act for alleged online fraud. "
     "My name is Ali Raza. I believe the complaint is false and based on misunderstanding."),

    # 11. Missing child — Habeas Corpus variant
    ("zainab_01",
     "My 14-year-old daughter Zainab has been taken by her uncle without my consent. "
     "My name is Shazia Begum. He is refusing to return her and her location is unknown."),

    # 12. Harassment case — Pre-arrest bail (sensitive)
    ("omar_01",
     "An FIR under Section 509 PPC has been registered against me at Saddar Police Station. "
     "My name is Omar Farooq. The allegations are fabricated after a workplace dispute."),

    # 13. Business fraud — Civil vs criminal overlap
    ("kashif_01",
     "My business partner has registered an FIR under Section 420 PPC against me "
     "after a failed investment deal. My name is Kashif Ali. "
     "This is purely a civil matter being given criminal color."),

    # 14. Illegal sealing — multiple shops affected (public law angle)
    ("rehman_01",
     "SBCA sealed multiple shops in our market in Saddar without notice. "
     "My name is Abdul Rehman. All shopkeepers have valid documents and licenses."),

    # 15. Preventive detention misuse
    ("junaid_01",
     "My brother Junaid has been detained by police under preventive laws "
     "without any clear reason. My name is Hamza Junaid. "
     "We believe this is misuse of authority."),

    # 16. Bail after long custody (delay in trial)
    ("nadeem_01",
     "I am Nadeem Akhtar. I have been in jail for 8 months under Section 337 PPC "
     "and the trial has not progressed at all."),

    # 17. FIR cross-version (counter-blast case)
    ("salman_01",
     "After I filed an FIR against someone, they registered a counter FIR against me "
     "under Section 506 PPC. My name is Salman Khan. "
     "This is clearly a counterblast to my complaint."),

    # 18. Employment dispute turned criminal
    ("rabia_01",
     "My employer has filed an FIR against me for breach of trust under Section 406 PPC. "
     "My name is Rabia Ahmed. This is actually a salary dispute."),

    # 19. Illegal arrest at workplace
    ("danish_01",
     "Police arrested me from my office without warrant under unclear allegations. "
     "My name is Danish Siddiqui. I was not informed of any FIR."),

    # 20. Land grabbing by authority
    ("yousuf_01",
     "Local authorities have taken possession of my land in Scheme 33 Karachi "
     "without compensation or notice. My name is Yousuf Khan."),

]

In [28]:
# ── CELL 15: Run All Test Cases (Base + Advanced) + Evaluation Dashboard ───────────────────────

# Combine base + advanced test cases
all_test_cases = test_cases + extra_test_cases

all_results = []

print(f"\n🚀 Running {len(all_test_cases)} Total Test Cases...")
print("="*70)

for user_id, story in all_test_cases:
    print(f"\n{'='*65}")
    print(f"🧪 {user_id.upper()}")
    print(f"{'='*65}")
    
    result_state = run_legal_assistant(story, user_id=user_id)
    
    if result_state:
        all_results.append(result_state)
    
    print()

# ── Summary Report ────────────────────────────────────────────────────────
print("\n" + "="*70)
print("📊 FINAL EVALUATION SUMMARY")
print("="*70)

print_eval_summary(all_results)

# ── Visualization Dashboard ───────────────────────────────────────────────
plot_evaluation_dashboard(
    all_results,
    title="Legal Petition Quality Dashboard — v3 (Full Benchmark Suite)"
)

# ── Optional: Stress Testing (Random Subset) ──────────────────────────────
import random

stress_sample = random.sample(all_test_cases, min(10, len(all_test_cases)))

print("\n" + "="*70)
print("🔥 STRESS TEST RUN (Random 10 Cases)")
print("="*70)

stress_results = []

for user_id, story in stress_sample:
    print(f"\n{'='*65}")
    print(f"⚡ {user_id.upper()} (Stress Test)")
    print(f"{'='*65}")
    
    result_state = run_legal_assistant(story, user_id=f"{user_id}_stress")
    
    if result_state:
        stress_results.append(result_state)
    
    print()

print("\n📊 Stress Test Summary:")
print_eval_summary(stress_results)


🚀 Running 24 Total Test Cases...

🧪 RAFSHA_01
📋 [InfoCheck] Scanning for missing details...
   → Complete: True | Missing: []

🏛️  Jurisdiction (Enter to auto-detect, or type one):
   Options: Sindh HC | Lahore HC | Islamabad HC | Sessions Court Karachi | Sessions Court Lahore


   →  


   Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Habeas Corpus | Jurisdiction: Sindh HC
🚨 [RedFlag] Scanning for legal landmines...
   ⚠️  1 flag(s) detected — severity: LOW
      → FIR mentioned but no FIR number extracted — drafter will use placeholder.
🧠 [Strategy] Generating legal strategy memo...
   → Strategy loaded for: Habeas Corpus
🔎 [RAG Primary] Attempt 1...
   → Primary: Crl.P.L.A.271_2024
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: C.P.L.A.1809_2020
📚 [CitationRanker] Evaluating citation authority...
   ⚠️  Citation issues detected:
      → Primary citation 'Crl.P.L.A.271_2024' is UNKNOWN (rank 99/5) — CPLA is a petition for leave, NOT a judgment. Replace with PLD or SCMR citation.
      → Supporting citation 'C.P.L.A.1809_2020' is UNKNOWN — weak. Prefer PLD/SCMR.
📝 [Extractor] Pulling structured fields...
⚡ [InterimRelief] Injecting urgency prayers...
   → Interim relief injected for: Habeas Corpus
✍️ [Drafter] Writing

   →  


   Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Pre-Arrest Bail | Jurisdiction: Sessions Court Karachi
🚨 [RedFlag] Scanning for legal landmines...
   ⚠️  2 flag(s) detected — severity: HIGH
      → Section 324 PPC (attempt to murder) is NON-BAILABLE — severity: HIGH. Do NOT argue it is minor.
      → PRE-ARREST BAIL with HIGH-severity section: must argue extraordinary circumstances + mala fide intent of complainant. Mention 'concession not right' doctrine.
🧠 [Strategy] Generating legal strategy memo...
   → Strategy loaded for: Pre-Arrest Bail
🔎 [RAG Primary] Attempt 1...
   → Primary: Crl.P.L.A.165-K_2022
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: Crl.P.L.A.1075-L_2020
📚 [CitationRanker] Evaluating citation authority...
   ⚠️  Citation issues detected:
      → Primary citation 'Crl.P.L.A.165-K_2022' is UNKNOWN (rank 99/5) — CPLA is a petition for leave, NOT a judgment. Replace with PLD or SCMR citation.
      → Supporting citatio

...

   →  


   Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Post-Arrest Bail | Jurisdiction: Sessions Court Karachi
🚨 [RedFlag] Scanning for legal landmines...
   ✅ No red flags detected
🧠 [Strategy] Generating legal strategy memo...
   → Strategy loaded for: Post-Arrest Bail
🔎 [RAG Primary] Attempt 1...
   → Primary: Crl.P.L.A.725_2023
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: Crl.P.L.A.298_2023
📚 [CitationRanker] Evaluating citation authority...
   ⚠️  Citation issues detected:
      → Primary citation 'Crl.P.L.A.725_2023' is UNKNOWN (rank 99/5) — CPLA is a petition for leave, NOT a judgment. Replace with PLD or SCMR citation.
      → Supporting citation 'Crl.P.L.A.298_2023' is UNKNOWN — weak. Prefer PLD/SCMR.
📝 [Extractor] Pulling structured fields...
⚡ [InterimRelief] Injecting urgency prayers...
   → Interim relief injected for: Post-Arrest Bail
✍️ [Drafter] Writing Post-Arrest Bail — Sessions Court Karachi...
✅ [Validator] Checking pet

   →  


   Auto-detecting from story...
🧠 [Orchestrator] Classifying case...
   → Type: Pre-Arrest Bail | Jurisdiction: Islamabad HC
🚨 [RedFlag] Scanning for legal landmines...
   ⚠️  1 flag(s) detected — severity: LOW
      → FIR mentioned but no FIR number extracted — drafter will use placeholder.
🧠 [Strategy] Generating legal strategy memo...
   → Strategy loaded for: Pre-Arrest Bail
🔎 [RAG Primary] Attempt 1...
   → Primary: Crl.P.L.A.165-K_2022
🔎 [RAG Secondary] Finding supporting authority...
   → Supporting: Crl.P.L.A.1075-L_2020
📚 [CitationRanker] Evaluating citation authority...
   ⚠️  Citation issues detected:
      → Primary citation 'Crl.P.L.A.165-K_2022' is UNKNOWN (rank 99/5) — CPLA is a petition for leave, NOT a judgment. Replace with PLD or SCMR citation.
      → Supporting citation 'Crl.P.L.A.1075-L_2020' is UNKNOWN — weak. Prefer PLD/SCMR.
📝 [Extractor] Pulling structured fields...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmkha0xbehcs56s2te0snt8r` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99987, Requested 903. Please try again in 12m48.96s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [30]:
from pathlib import Path
import subprocess

def txt_to_latex_pdf(txt_path):
    txt_path = Path(txt_path)
    content = txt_path.read_text()

    latex_template = f"""
\\documentclass[12pt]{{article}}
\\usepackage[a4paper,margin=1in]{{geometry}}
\\usepackage{{setspace}}
\\usepackage{{parskip}}

\\title{{Legal Petition}}
\\date{{}}

\\begin{{document}}

\\begin{{center}}
\\textbf{{\\Large LEGAL PETITION}} \\\\[10pt]
\\end{{center}}

\\onehalfspacing

{content.replace('_', '\\_')}

\\end{{document}}
"""

    tex_path = txt_path.with_suffix(".tex")
    pdf_path = txt_path.with_suffix(".pdf")

    tex_path.write_text(latex_template)

    subprocess.run(["pdflatex", "-interaction=nonstopmode", str(tex_path)], check=True)

    return pdf_path

In [32]:
!apt-get install texlive

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  dvisvgm fonts-lmodern fonts-texgyre libkpathsea6 libptexenc1 libsynctex2
  libteckit0 libtexlua53 libtexluajit2 libwoff1 libzzip-0-13 lmodern t1utils
  tex-common tex-gyre texlive-base texlive-binaries texlive-fonts-recommended
  texlive-latex-base texlive-latex-recommended tipa xfonts-encodings
  xfonts-utils
Suggested packages:
  debhelper perl-tk xpdf | pdf-viewer xzdec texlive-fonts-recommended-doc
  texlive-latex-base-doc texlive-latex-recommended-doc texlive-luatex
  texlive-pstricks tipa-doc
The following NEW packages will be installed:
  dvisvgm fonts-lmodern fonts-texgyre libkpathsea6 libptexenc1 libsynctex2
  libteckit0 libtexlua53 libtexluajit2 libwoff1 libzzip-0-13 lmodern t1utils
  tex-common tex-gyre texlive texlive-base texlive-binaries
  texlive-fonts-recommended texlive-latex-base texlive-latex-recommended tipa
  xfonts

In [33]:
from pathlib import Path

working_dir = Path("/kaggle/working")

txt_files = list(working_dir.glob("*.txt"))

pdf_outputs = []

for file in txt_files:
    print(f"📄 Converting: {file.name}")
    pdf = txt_to_latex_pdf(file)
    pdf_outputs.append(pdf)

print("\n✅ All PDFs generated:")
for p in pdf_outputs:
    print(p)

📄 Converting: usman_01_Post-Arrest_Bail.txt
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(/kaggle/working/usman_01_Post-Arrest_Bail.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size12.clo))
(/usr/share/texlive/texmf-dist/tex/latex/geometry/geometry.sty
(/usr/share/texlive/texmf-dist/tex/latex/graphics/keyval.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifvtex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty)))
(/usr/share/texlive/texmf-dist/tex/latex/setspace/setspace.sty)
(/usr/share/texlive/texmf-dist/tex/latex/parskip/parskip.sty
(/usr/share/texlive/texmf-dist/tex/latex/kvoptions/kvoptions.sty
(/usr/share/texlive/texmf-dist/tex/generic/ltxcmds/ltxcmds.